# Phase 1 — Cross-Dataset Benchmark (S1–S6) + SDA/TDSS + DA Baseline
**Framework:** PyTorch · albumentations · Google Colab A100

Original cell plan of Phase 1 (cell names were kept so that they match the result files). Labels such as "proposed" reflect the plan at the time; the paper's conclusions are based on the controlled add-on experiments below.

---

| Cell | Content |
|------|---------|
| 0    | Setup & Mount Drive |
| 1    | Dataset Download & Verify (3-class: RingSpot excluded) |
| 2    | Dataset Characterization + MMD Domain Gap |
| 3    | Model Definitions (5 architectures) |
| 4    | Training Utilities (F1–F7 TropAug · replace_head · TTA) |
| 5    | S1: PV→PV (within-domain baseline) |
| 6    | S2: PV→PlantDoc · S3: PV→SLD |
| 7    | S4: PV→Thai (zero-shot) |
| 8    | S5: Multi→Thai (standard aug) |
| 9    | S6: Multi+F1–F7+TTA→Thai |
| 10   | Ablation Study (Config A–D) |
| 10b  | Systematic Degradation Analysis (SDA) + TDSS |
| 10c  | Domain Adaptation Baseline Comparison (CORAL · AdaBN) |
| 11   | Statistical Significance Test (McNemar · Bootstrap CI) |
| 12   | Split Sensitivity (3 splits · S4 vs S6) |
| 13   | Grad-CAM Visualization |
| 14   | Confusion Matrix (S4–S6) |
| 15   | Inference Latency Benchmark |
| 16   | CEA Publication-Ready Figures |
| 17   | Export Full Results Table |

> **Before running:** Runtime → Change runtime type → **A100 GPU**
> **3-class:** Healthy · RedRot · Rust (RingSpot excluded — absent from all source datasets)
> **Cell order:** 0→1→2→3→4→5→6→7→8→9→10→10b→10c→11→12→13→14→15→16→17

## Code for: What curated benchmarks miss: leakage auditing, controlled augmentation tests and blinded field evaluation of a deployed sugarcane leaf disease classifier

Run order: **Config → Pre-flight → Cell 0 → 1.x → 3.1 → 4.x → S1–S6 → Add-on (A0 → A10, B1–B4, S4-fair v2, C1, S2-fixed, A7c, Z)**.
Cells marked **Note:** are superseded and their results are not reported in the paper.

**Data.** PlantVillage, PlantDoc and the Sugarcane Leaf Disease dataset (Daphal and Koli, 2022) are public (downloaded in Cell 1.2 via the Kaggle API; set `KAGGLE_USERNAME` / `KAGGLE_KEY` in Colab Secrets).
The Thai sugarcane leaf dataset (Baipod et al., 2025) is **not public** — request it from its owners.
Field images collected through the LINE system are **not shared** (ethics approval: Mahasarakham University, no. 452-459/2569); cells C1.x need them and are skipped if `FIELD_DIR` does not exist.

In [ ]:
# ── Config: edit these paths ──────────────────────────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive'
BASE_DRIVE    = f'{DRIVE_ROOT}/sugarcane_research_3class'   # checkpoints, tables, figures
THAI_SRC_DIR  = f'{DRIVE_ROOT}/Thai_Local'                  # Thai dataset (not public): <split>/{train,test}/<class>/
FIELD_DIR     = f'{DRIVE_ROOT}/field_images'                # LINE field images (not public)
FIELD_LOG_CSV = f'{DRIVE_ROOT}/field_log.csv'               # deployed-system log (not public)
DATA_DIR      = '/content/datasets'                         # public datasets are downloaded here


## Pre-flight Check — run before any other cell

This cell:
1. **Checks old checkpoints** trained with 4 classes (not compatible with the 3-class setup); they are deleted only if `AUTO_DELETE_CKPT = True`
2. **Checks the environment** (GPU, Drive, datasets)
3. **Shows a checklist** before running

> Run this cell once before starting — re-running it is harmless

In [ ]:
# AUTO_DELETE_CKPT=False → report only, nothing is deleted (set True to delete invalid checkpoints)
AUTO_DELETE_CKPT = False
# ── Pre-flight Check & Auto Cleanup ──────────────────────────────────────────
# Run this cell before any other cell
# Check (and optionally delete) old checkpoints that are not compatible with the 3-class setup
# ─────────────────────────────────────────────────────────────────────────────
import os, torch
from pathlib import Path

# ── paths (must match Cell 0) ────────────────────────────────────────────────
# BASE_DRIVE: set in the Config cell
MODELS_DIR  = f'{BASE_DRIVE}/models'

# ── 3-class setup ─────────────────────────────────────────────────────────────
TARGET_CLASSES = ['Healthy', 'RedRot', 'Rust']  # 3-class (RingSpot excluded)
OLD_NUM_CLASSES = 4  # obsolete value to remove

print('='*60)
print('  Pre-flight Check — 3-class Setup')
print('='*60)

# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'[OK] GPU  : {gpu_name} ({vram_gb:.1f} GB)')
else:
    print('[ERROR] GPU  : no GPU found — switch the runtime to A100 first')

# ── 2. Check Google Drive ─────────────────────────────────────────────────────
drive_ok = Path(DRIVE_ROOT).exists()
print(f'{'[OK]' if drive_ok else '[ERROR]'} Drive : {'mounted' if drive_ok else 'not mounted — run Cell 0 first'}')

# ── 3. Check old, incompatible checkpoints ───────────────────────────────────
print()
print('Checking old checkpoints (4-class → to delete):')

STALE_PATTERNS = [
    'S1_*.pt',    # trained 38-class PV → replace_head → 4 → delete (now → 3)
    'S5_*.pt',    # trained union ~68 classes → delete (union changed)
    'S6_*.pt',    # trained union ~68 classes → delete (union changed)
    'abl_*.pt',   # ablation 38-class head → delete
]

# Note: S1 does not always need deleting — S1 is trained on 38-class PV, which is correct
# but replace_head now changes 38→3 instead of 38→4 in Cell 7
# → S1 checkpoints remain valid and are kept
SAFE_TO_KEEP = ['S1_']

deleted, kept, not_found = [], [], []

models_path = Path(MODELS_DIR)
if not models_path.exists():
    print(f'  Note: MODELS_DIR does not exist yet: {MODELS_DIR} (created when Cell 0 runs)')
else:
    for pattern in STALE_PATTERNS:
        prefix = pattern.replace('_*.pt', '_')
        for pt_file in models_path.glob(pattern):
            # check whether it is safe to keep
            if any(pt_file.name.startswith(s) for s in SAFE_TO_KEEP):
                kept.append(pt_file.name)
                continue
            # check head size in the checkpoint
            try:
                sd = torch.load(pt_file, map_location='cpu')
                key = next((k for k in sd
                            if any(x in k for x in
                                   ['fc.weight','head.fc.weight',
                                    'classifier.weight','head.weight'])), None)
                head_size = sd[key].shape[0] if key else -1
                # delete if head_size is not 3 or 38 (source training)
                # 38 = PlantVillage source training → still valid
                # 3  = 3-class target → correct
                # others (4, 68, 27, ...) → delete
                if head_size not in (3, 38):
                    os.remove(pt_file) if AUTO_DELETE_CKPT else None
                    deleted.append(f'{pt_file.name} (head={head_size})')
                else:
                    kept.append(f'{pt_file.name} (head={head_size} OK)')
            except Exception as e:
                # unreadable → delete
                os.remove(pt_file) if AUTO_DELETE_CKPT else None
                deleted.append(f'{pt_file.name} (unreadable: {e})')

if deleted:
    print(f'  Delete {len(deleted)} incompatible checkpoint(s):')
    for f in deleted:
        print(f'      → {f}')
else:
    print('  [OK] no old checkpoints to delete')

if kept:
    print(f'  Keep {len(kept)} checkpoint(s):')
    for f in kept:
        print(f'      → {f}')

# ── 4. Check dataset paths ────────────────────────────────────────────────────
print()
print('Checking dataset paths:')
THAI_BASE_DIR = THAI_SRC_DIR
THAI_TEST_CHECK = f'{THAI_BASE_DIR}/80/test'
dataset_checks = [
    ('Thai_Local (test/80)', THAI_TEST_CHECK),
    ('sugarcane_research/models', MODELS_DIR),
]
for name, path in dataset_checks:
    exists = Path(path).exists()
    # count class folders
    if exists:
        folders = [d.name for d in Path(path).iterdir()
                   if d.is_dir() and d.name not in ['RingSpot']]
        status = f'[OK] found — folders: {folders}'
    else:
        status = '[WARNING] not found (will be created or downloaded in Cell 1)'
    print(f'  {name:<30} {status}')

# ── 5. Confirm 3-class setup ─────────────────────────────────────────────────
print()
print('='*60)
print('  Confirm 3-class setup')
print('='*60)
print(f'  TARGET_CLASSES : {TARGET_CLASSES}')
print(f'  EXCLUDED       : ["RingSpot"] — absent from PV/PlantDoc/SLD')
print(f'  NUM_CLASSES    : 3')
print()
print('  Correct cell order:')
print('  0→1→2→3→4→5→6→7→8→9→10→10b→10c→11→12→13→14→15→16→17')
print()
print('[OK] Pre-flight check complete — ready to run Cell 0')


## Cell 0 — Setup & Mount Google Drive

In [ ]:
# ── 0.1 Mount Drive ──────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 0.2 Install dependencies ─────────────────────────────────────────────────
import subprocess, sys
pkgs = ['timm', 'grad-cam', 'scikit-learn', 'statsmodels', 'seaborn',
        'matplotlib', 'pandas', 'openpyxl', 'kaggle',
        'albumentations>=1.3.0', 'opencv-python-headless']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=True)

import albumentations as A
print(f'albumentations version: {A.__version__}')
# requires >= 1.3.0 for RandomRain, RandomFog, ImageCompression
assert tuple(int(x) for x in A.__version__.split('.')[:2]) >= (1,3),     'albumentations >= 1.3.0 required'

# ── 0.3 Imports ──────────────────────────────────────────────────────────────
import os, time, json, random, warnings
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torchvision import models
import timm

from albumentations.pytorch import ToTensorV2
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (confusion_matrix, f1_score,
                              precision_score, recall_score)
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings('ignore')

# ── 0.4 Reproducibility ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# ── 0.5 Device ───────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ── 0.6 Paths ────────────────────────────────────────────────────────────────
# BASE_DRIVE: set in the Config cell
DATA_DIR    = '/content/datasets'


MODELS_DIR  = f'{BASE_DRIVE}/models'
FIGURES_DIR = f'{BASE_DRIVE}/figures'
TABLES_DIR  = f'{BASE_DRIVE}/tables'
for d in [DATA_DIR, MODELS_DIR, FIGURES_DIR, TABLES_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ── 0.7 Global config ────────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE = 256  # 85GB RAM → batch 256
EPOCHS      = 30
PATIENCE    = 7
LR          = 1e-4
NUM_WORKERS = 8  # 85GB RAM + A100 → workers 8
# ── RAM & Performance config (85GB RAM) ──────────────────────────────────────
# A100 40GB VRAM + 85GB RAM → use the RAM cache fully
RAM_CACHE_GB     = 75.0   # maximum RAM cache (leaves ~10GB buffer)
BATCH_SIZE       = 256   # larger batch → better GPU utilization
NUM_WORKERS      = 8     # parallel data loading
PREFETCH_FACTOR  = 4     # batches prefetched ahead
PERSISTENT_WORKERS = True   # do not kill workers between epochs


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# CLASS_NAMES is overridden with the actual Thai folders in Cell 1
# RingSpot is excluded because it is absent from the source datasets (PV/PlantDoc/SLD)
# → reported as source-dataset coverage gap finding in paper
CLASS_NAMES = ['Healthy', 'RedRot', 'Rust']
NUM_CLASSES = 3
EXCLUDE_CLASSES = ['RingSpot']  # classes absent from all source datasets

ARCHITECTURES = ['resnet50','efficientnet_b4','densenet121','vit_b16','convnext_base']
ARCH_LABELS   = ['ResNet-50','EfficientNet-B4','DenseNet-121','ViT-B/16','ConvNeXt-B']

RESULTS = []   # rows from every scenario
print('[OK] Setup complete')

# CEA figure sizes (originally defined in Cell 16 but already used from Cell 2.3)
COL1 = 3.5   # inches — single column
COL2 = 7.24  # inches — double column


In [ ]:
# ── RAM Utilization Monitor (85GB RAM) ──────────────────────────────────────
import psutil, torch

ram    = psutil.virtual_memory()
ram_gb = ram.total / 1e9
used   = ram.used  / 1e9
avail  = ram.available / 1e9

print(f'RAM  : {ram_gb:.1f} GB total | {used:.1f} GB used | {avail:.1f} GB available')
print(f'Config: BATCH_SIZE=256 | NUM_WORKERS=8 | PREFETCH=4')
print(f'        RAM_CACHE_GB=75.0 (cache budget)')

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM : {vram:.1f} GB')

# check that there is enough RAM for the cache
if avail >= 75.0:
    print(f'[OK] enough RAM for the full dataset cache ({avail:.1f} GB available)')
else:
    print(f'[WARNING] low RAM ({avail:.1f} GB) — consider reducing RAM_CACHE_GB')

print('[OK] RAM check complete')


## Cell 1 — Dataset Download & Verify

In [ ]:
# ── 1.1 Kaggle API (read from Colab Secrets — no key stored in this file) ─────
from google.colab import userdata
import json as _json
creds = {"username": userdata.get('KAGGLE_USERNAME'),
         "key": userdata.get('KAGGLE_KEY')}
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    _json.dump(creds, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('[OK] Kaggle API ready')

In [ ]:
# ── 1.2 Download source datasets ────────────────────────────────────────────
import subprocess

def count_images(path):
    p = Path(path)
    return sum(len(list(p.rglob(e)))
               for e in ['*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG'])

SOURCES = {
    'PlantVillage': ('datasets','abdallahalidev/plantvillage-dataset',
                     DATA_DIR+'/plantvillage'),
    'PlantDoc':     ('datasets','abdulhasibuddin/plant-doc-dataset',
                     DATA_DIR+'/plantdoc'),
    'SLD':          ('datasets','nirmalsankalana/sugarcane-leaf-disease-dataset',
                     DATA_DIR+'/sld'),
}
for name,(kind,slug,dest) in SOURCES.items():
    if Path(dest).exists() and any(Path(dest).iterdir()):
        print(f'  skip {name} ({count_images(dest):,} imgs)')
        continue
    print(f'  downloading {name}...', end=' ', flush=True)
    r = subprocess.run(['kaggle',kind,'download',slug,
                        '--path',dest,'--unzip','--quiet'],
                       capture_output=True, text=True)
    print('OK' if r.returncode==0 else f'FAIL: {r.stderr[:100]}')
print('[OK] Source datasets ready')


In [ ]:

# ── 1.3 Path mapping ─────────────────────────────────────────────────────────
THAI_BASE_DIR = THAI_SRC_DIR

THAI_SPLITS = {
    '90': {'train': f'{THAI_BASE_DIR}/90/train', 'test': f'{THAI_BASE_DIR}/90/test'},
    '80': {'train': f'{THAI_BASE_DIR}/80/train', 'test': f'{THAI_BASE_DIR}/80/test'},
    '75': {'train': f'{THAI_BASE_DIR}/75/train', 'test': f'{THAI_BASE_DIR}/75/test'},
}
THAI_DEFAULT_SPLIT = '80'
THAI_TRAIN = THAI_SPLITS[THAI_DEFAULT_SPLIT]['train']
THAI_TEST  = THAI_SPLITS[THAI_DEFAULT_SPLIT]['test']

DATASET_PATHS = {
    'PlantVillage': f'{DATA_DIR}/plantvillage/plantvillage dataset/color',
    'PlantDoc':     f'{DATA_DIR}/plantdoc/PlantDoc-Dataset/train',
    'SLD':          f'{DATA_DIR}/sld',
}

# Override CLASS_NAMES with the actual Thai folders, excluding EXCLUDE_CLASSES
if Path(THAI_TEST).exists():
    all_classes = sorted([d.name for d in Path(THAI_TEST).iterdir() if d.is_dir()])
    CLASS_NAMES = [c for c in all_classes if c not in EXCLUDE_CLASSES]
    NUM_CLASSES = len(CLASS_NAMES)
    excluded = [c for c in all_classes if c in EXCLUDE_CLASSES]
    print(f'[OK] Thai CLASS_NAMES (3-class): {CLASS_NAMES}  (NUM_CLASSES={NUM_CLASSES})')
    print(f'   Excluded (absent from source): {excluded}')
else:
    print(f'[WARNING] THAI_TEST not found: {THAI_TEST}')

# check every path
for name, path in {**DATASET_PATHS,
                   'Thai_test': THAI_TEST,
                   'Thai_train': THAI_TRAIN}.items():
    p = Path(path)
    status = f'[OK]  {count_images(path):>6,} images' if p.exists() else '[ERROR]  NOT FOUND'
    print(f'  {name:<15} {status}  → {path}')


# ── make_test_loader: filter out EXCLUDE_CLASSES ─────────────────────────────
import torchvision.transforms as TVT

class FilteredImageFolder(torch.utils.data.Dataset):
    """ImageFolder that drops EXCLUDE_CLASSES and remaps labels to be contiguous.
    Compatible with tta_predict: has .samples and .alb_transform attributes
    """
    def __init__(self, root, class_names, transform=None):
        self.class_names   = class_names
        self.alb_transform = transform   # named alb_transform for compatibility with tta_predict
        self.samples       = []
        for label_idx, cls in enumerate(class_names):
            cls_dir = Path(root) / cls
            if not cls_dir.exists():
                print(f'  [WARNING] {cls} not found in {root}'); continue
            for ext in ('*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG'):
                for img_path in cls_dir.glob(ext):
                    self.samples.append((str(img_path), label_idx))
        print(f'  FilteredImageFolder: {len(self.samples)} images | classes: {class_names}')

    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        img = np.array(Image.open(path).convert('RGB'))
        if self.alb_transform:
            img = self.alb_transform(image=img)['image']
        return img, label

def make_test_loader(test_root, class_names=None, batch_size=None):
    if batch_size is None:
        batch_size = BATCH_SIZE
    if class_names is None:
        class_names = CLASS_NAMES  # 3-class (RingSpot excluded)
    ds = FilteredImageFolder(test_root, class_names, transform=VAL_TRANSFORM)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
        prefetch_factor=4, persistent_workers=True)
    return loader, class_names

print('[OK] make_test_loader (3-class filtered) ready')


In [ ]:
# ── 1.4 Copy Thai dataset from Drive to /content ─────────────────────────────
import shutil
if not Path(THAI_BASE_DIR).exists() and    Path(THAI_SRC_DIR).exists():
    shutil.copytree(
        THAI_SRC_DIR,
        THAI_BASE_DIR)
    print(f'[OK] Copied Thai_Local → {THAI_BASE_DIR}')
else:
    print(f'[OK] Thai_Local already at {THAI_BASE_DIR}')


In [ ]:
# ── 1.5 Filter SLD → 3 classes aligned with Thai ────────────────────────────
import shutil

SLD_RAW    = f'{DATA_DIR}/sld'
SLD_FILTER = f'{DATA_DIR}/sld_filtered'

# SLD_CLASS_MAP: src folder name → dst folder name (must match Thai CLASS_NAMES)
# Thai uses: Healthy, RedRot, RingSpot, Rust
# SLD has:  Healthy, RedRot, Rust  (no RingSpot — noted as limitation in paper)
SLD_CLASS_MAP = {
    'Healthy': 'Healthy',
    'RedRot':  'RedRot',
    'Rust':    'Rust',
    # Mosaic, Yellow excluded (not in the Thai dataset)
    # RingSpot is not in SLD → unseen class for S3/S5/S6 (reported as finding)
}

if Path(SLD_FILTER).exists():
    shutil.rmtree(SLD_FILTER)
Path(SLD_FILTER).mkdir(parents=True)

for src_name, dst_name in SLD_CLASS_MAP.items():
    src_dir = Path(SLD_RAW) / src_name
    dst_dir = Path(SLD_FILTER) / dst_name
    if not src_dir.exists():
        print(f'  [WARNING] {src_name} not found in SLD'); continue
    shutil.copytree(src_dir, dst_dir)
    print(f'  [OK] {src_name} → {dst_name}  ({count_images(dst_dir)} images)')

DATASET_PATHS['SLD'] = f'{DATA_DIR}/sld'
print(f'\nSLD filtered classes: {sorted([d.name for d in Path(SLD_FILTER).iterdir()])}')
print('Note: RingSpot absent from SLD → unseen class in multi-source scenarios')
print('      This is reported as a finding (source dataset limitation)')


## Cell 2 — Dataset Characterization + MMD Domain Gap

In [ ]:
# ── 2.1 Class distribution per dataset (incl. the 3 Thai splits) ────────────
from PIL import Image

def count_classes(root_path):
    p = Path(root_path)
    result = {}
    for cls_dir in sorted(p.iterdir()):
        if cls_dir.is_dir():
            n = count_images(cls_dir)   # handles .jpeg .JPEG
            if n > 0:
                result[cls_dir.name] = n
    return result

dist_records = []

# Source datasets
# Note: the SLD used here is sld_filtered (3 classes), after Cell 1.6 has run
for name in ['PlantVillage','PlantDoc','SLD']:
    path = DATASET_PATHS[name]
    if not Path(path).exists(): continue
    cls_dist = count_classes(path)
    total = sum(cls_dist.values())
    for cls, n in cls_dist.items():
        dist_records.append({'Dataset': name, 'Split': '-', 'Subset': 'all',
                              'Class': cls, 'Count': n, 'Pct': round(n/total*100,1)})

# Thai dataset — 3 splits × train/test
for split, paths in THAI_SPLITS.items():
    for subset, path in paths.items():
        if not Path(path).exists(): continue
        cls_dist = count_classes(path)
        total = sum(cls_dist.values())
        for cls, n in cls_dist.items():
            dist_records.append({'Dataset': 'Thai', 'Split': split,
                                  'Subset': subset, 'Class': cls,
                                  'Count': n, 'Pct': round(n/total*100,1)})

df_dist = pd.DataFrame(dist_records)

# Summary table
print('Class Distribution Summary:')
summary_rows = []
# Note: the SLD used here is sld_filtered (3 classes), after Cell 1.6 has run
for name in ['PlantVillage','PlantDoc','SLD']:
    sub = df_dist[df_dist['Dataset']==name]
    if sub.empty: continue
    summary_rows.append({'Dataset': name, 'Split': '-',
        'Total': sub['Count'].sum(),
        'Classes': sub['Class'].nunique(),
        'Min': sub['Count'].min(), 'Max': sub['Count'].max(),
        'Imbalance': round(sub['Count'].max()/max(sub['Count'].min(),1), 2)})

for split in ['90','80','75']:
    sub = df_dist[(df_dist['Dataset']=='Thai') & (df_dist['Split']==split)]
    if sub.empty: continue
    for ss in ['train','test']:
        s2 = sub[sub['Subset']==ss]
        if s2.empty: continue
        summary_rows.append({'Dataset': f'Thai ({split}/{ss})', 'Split': split,
            'Total': s2['Count'].sum(),
            'Classes': s2['Class'].nunique(),
            'Min': s2['Count'].min(), 'Max': s2['Count'].max(),
            'Imbalance': round(s2['Count'].max()/max(s2['Count'].min(),1), 2)})

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))


In [ ]:
# ── 2.2 Image quality statistics ─────────────────────────────────────────────
import cv2

def sample_image_stats(root_path, n_sample=200):
    p = Path(root_path)
    imgs = []
    for ext in ['*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG']:
        imgs.extend(list(p.rglob(ext)))
    imgs = random.sample(imgs, min(n_sample, len(imgs)))

    widths, heights, brightnesses, blurs = [], [], [], []
    for imgp in imgs:
        try:
            img = cv2.imread(str(imgp))
            if img is None: continue
            h, w = img.shape[:2]
            widths.append(w); heights.append(h)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            brightnesses.append(gray.mean())
            blurs.append(cv2.Laplacian(gray, cv2.CV_64F).var())
        except: pass

    return {
        'width_mean': np.mean(widths), 'height_mean': np.mean(heights),
        'brightness_mean': np.mean(brightnesses), 'brightness_std': np.std(brightnesses),
        'blur_score_mean': np.mean(blurs), 'n_sampled': len(widths)
    }

quality_rows = []
for name, path in DATASET_PATHS.items():
    if not Path(path).exists(): continue
    print(f'  Sampling {name}...')
    stats_d = sample_image_stats(path)
    stats_d['Dataset'] = name
    quality_rows.append(stats_d)
    print(f'  [OK] {name}: {stats_d["n_sampled"]} samples · '
          f'avg {stats_d["width_mean"]:.0f}×{stats_d["height_mean"]:.0f}px · '
          f'brightness {stats_d["brightness_mean"]:.1f}±{stats_d["brightness_std"]:.1f} · '
          f'blur {stats_d["blur_score_mean"]:.0f}')

df_quality = pd.DataFrame(quality_rows)


In [ ]:
# add Thai stats to df_quality
print('  Sampling Thai...')
thai_stats = sample_image_stats(THAI_BASE_DIR)
thai_stats['Dataset'] = 'Thai'
df_quality = pd.concat([df_quality, pd.DataFrame([thai_stats])],
                        ignore_index=True)
print(f'  [OK] Thai: {thai_stats["n_sampled"]} samples · '
      f'blur {thai_stats["blur_score_mean"]:.0f}')
print(df_quality[['Dataset','blur_score_mean']])

In [ ]:
# ── 2.3 Figure 1 — Dataset Characterization (CEA style + pastel) ─────────────
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(COL2, 3.5))
fig.patch.set_facecolor('white')

# ── (a) Dataset size + class count ───────────────────────────────────────────
ds_names  = ['PlantVillage', 'PlantDoc', 'SLD\n(5-class)',
             'Thai\n(80/Train)', 'Thai\n(80/Test)']
ds_counts = [
    count_images(DATASET_PATHS['PlantVillage']),
    count_images(DATASET_PATHS['PlantDoc']),
    count_images(DATASET_PATHS['SLD']),
    count_images(THAI_TRAIN),
    count_images(THAI_TEST),
]
ds_classes = [38, 27, 5, 4, 4]

colors_bar = ['#90CAF9','#FFCC80','#A5D6A7','#C5E1A5','#C5E1A5']

bars = axes[0].bar(ds_names, ds_counts, color=colors_bar,
                   edgecolor='white', linewidth=0.8, alpha=0.92, zorder=2)
ax2 = axes[0].twinx()
ax2.plot(ds_names, ds_classes, 'D--', color='#E53935',
         markersize=5, linewidth=1.2, zorder=3)
ax2.set_ylabel('Number of Classes', fontsize=8, color='#E53935')
ax2.tick_params(axis='y', labelcolor='#E53935', labelsize=7)
ax2.set_ylim(0, 55)

offsets = [1800, 80, 55, 110, 28]
for bar, n, off in zip(bars, ds_counts, offsets):
    axes[0].text(bar.get_x() + bar.get_width()/2.,
                 bar.get_height() + off,
                 f'{n:,}', ha='center', va='bottom',
                 fontsize=7, fontweight='bold', zorder=4)

line_h, = ax2.plot([], [], 'D--', color='#E53935', label='# Classes')
axes[0].legend(handles=[line_h], loc='upper right', fontsize=7)
axes[0].set_title('(a) Dataset Size and Number of Classes',
                   fontsize=9, fontweight='bold', pad=6)
axes[0].set_ylabel('Number of Images', fontsize=8)
axes[0].set_ylim(0, max(ds_counts) * 1.22)
axes[0].tick_params(axis='x', labelsize=7)
axes[0].tick_params(axis='y', labelsize=7)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].grid(axis='y', alpha=0.25, linestyle='--', linewidth=0.6, zorder=0)

# ── (b) Image quality metrics ─────────────────────────────────────────────────
qual_names = ['PlantVillage', 'PlantDoc', 'SLD', 'Thai']

brightness_mu = [
    df_quality.loc[df_quality['Dataset']=='PlantVillage','brightness_mean'].values[0],
    df_quality.loc[df_quality['Dataset']=='PlantDoc','brightness_mean'].values[0],
    df_quality.loc[df_quality['Dataset']=='SLD','brightness_mean'].values[0],
    df_quality.loc[df_quality['Dataset']=='Thai','brightness_mean'].values[0],
]
brightness_std = [
    df_quality.loc[df_quality['Dataset']=='PlantVillage','brightness_std'].values[0],
    df_quality.loc[df_quality['Dataset']=='PlantDoc','brightness_std'].values[0],
    df_quality.loc[df_quality['Dataset']=='SLD','brightness_std'].values[0],
    df_quality.loc[df_quality['Dataset']=='Thai','brightness_std'].values[0],
]
blur_raw = [
    df_quality.loc[df_quality['Dataset']=='PlantVillage','blur_score_mean'].values[0],
    df_quality.loc[df_quality['Dataset']=='PlantDoc','blur_score_mean'].values[0],
    df_quality.loc[df_quality['Dataset']=='SLD','blur_score_mean'].values[0],
    df_quality.loc[df_quality['Dataset']=='Thai','blur_score_mean'].values[0],
]

def norm(vals):
    mn, mx = min(vals), max(vals)
    return [(v-mn)/(mx-mn) if mx > mn else 0 for v in vals]

x = np.arange(len(qual_names))
w = 0.25

b1 = axes[1].bar(x - w, norm(brightness_mu),  w,
                  label='Brightness μ', color='#90CAF9', alpha=0.90)
b2 = axes[1].bar(x,     norm(brightness_std), w,
                  label='Brightness σ', color='#FFCC80', alpha=0.90)
b3 = axes[1].bar(x + w, norm(blur_raw),       w,
                  label='Blur score',   color='#A5D6A7', alpha=0.90)

for bar, val in zip(b3, blur_raw):
    axes[1].text(bar.get_x() + bar.get_width()/2.,
                 bar.get_height() + 0.02,
                 f'{val:.0f}', ha='center', va='bottom',
                 fontsize=7, color='#2E7D32', fontweight='bold')

sld_bar = b3[2]
sld_x   = sld_bar.get_x() + sld_bar.get_width()/2.
sld_y   = norm(blur_raw)[2]
axes[1].annotate('SLD: low blur\n→ domain gap vs Thai',
                  xy=(sld_x, sld_y + 0.04),
                  xytext=(sld_x - 0.85, sld_y + 0.38),
                  fontsize=7, color='#E53935',
                  arrowprops=dict(arrowstyle='->', color='#E53935', lw=1.0))

axes[1].set_title('(b) Image Quality Metrics by Dataset',
                   fontsize=9, fontweight='bold', pad=6)
axes[1].set_xticks(x)
axes[1].set_xticklabels(qual_names, fontsize=8)
axes[1].set_ylabel('Normalized Value (0–1)', fontsize=8)
axes[1].tick_params(labelsize=7)
axes[1].set_ylim(0, 1.30)
axes[1].legend(fontsize=7, loc='upper left')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].grid(axis='y', alpha=0.25, linestyle='--', linewidth=0.6)

plt.tight_layout(pad=0.5)
fig_path = f'{FIGURES_DIR}/fig1_dataset_characterization.pdf'
plt.savefig(fig_path, dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(fig_path.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'[OK] Figure 1 saved → {fig_path}')

## Cell 2.4 — Domain Gap Quantification (MMD)
> Run after Cell 4 (Training Utilities) because it needs VAL_TRANSFORM

In [ ]:
# ── 2.4 Domain Gap Quantification — MMD (transform fixed for ImageFolder) ──
import torchvision.transforms as TVT
import torchvision.models as tvm

# VAL_TRANSFORM for MMD uses plain torchvision
# (ImageFolder needs a torchvision-style transform, not albumentations)
MMD_TRANSFORM = TVT.Compose([
    TVT.Resize((IMG_SIZE, IMG_SIZE)),
    TVT.ToTensor(),
    TVT.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

@torch.no_grad()
def extract_features(data_path, n_max=500):
    """Extract ResNet-50 pool5 features (2048-dim)"""
    feat_model = tvm.resnet50(weights='IMAGENET1K_V2')
    feat_model.fc = torch.nn.Identity()
    feat_model = feat_model.to(DEVICE).eval()

    ds      = ImageFolder(data_path, transform=MMD_TRANSFORM)
    indices = torch.randperm(len(ds),
                             generator=torch.Generator().manual_seed(SEED))[:n_max]
    subset  = torch.utils.data.Subset(ds, indices.tolist())
    loader  = DataLoader(subset, batch_size=64, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
        prefetch_factor=4, persistent_workers=True)
    feats = []
    for imgs, _ in loader:
        feats.append(feat_model(imgs.to(DEVICE)).cpu())
    del feat_model; torch.cuda.empty_cache()
    return torch.cat(feats, dim=0)

# ── the rest (mmd_rbf, loop, figure) runs as before ─────────────────────────

## Cell 3 — Model Definitions (5 Architectures)

In [ ]:
# ── 3.1 Build model function ─────────────────────────────────────────────────
def build_model(arch: str, num_classes: int = 4, pretrained: bool = True):
    """Return model, param count, GFLOPs estimate."""
    if arch == 'resnet50':
        m = models.resnet50(weights='IMAGENET1K_V2' if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif arch == 'efficientnet_b4':
        m = timm.create_model('efficientnet_b4', pretrained=pretrained,
                               num_classes=num_classes)
    elif arch == 'densenet121':
        m = models.densenet121(weights='IMAGENET1K_V1' if pretrained else None)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif arch == 'vit_b16':
        m = timm.create_model('vit_base_patch16_224', pretrained=pretrained,
                               num_classes=num_classes)
    elif arch == 'convnext_base':
        m = timm.create_model('convnext_base', pretrained=pretrained,
                               num_classes=num_classes)
    else:
        raise ValueError(f'Unknown arch: {arch}')
    return m.to(DEVICE)


ARCHITECTURES = ['resnet50', 'efficientnet_b4', 'densenet121',
                 'vit_b16', 'convnext_base']
ARCH_LABELS   = ['ResNet-50', 'EfficientNet-B4', 'DenseNet-121',
                  'ViT-B/16', 'ConvNeXt-B']

# Quick sanity check
print('Architecture parameter counts:')
for arch, label in zip(ARCHITECTURES, ARCH_LABELS):
    m = build_model(arch)
    params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'  {label:<20} {params:.1f}M params')
    del m
torch.cuda.empty_cache()
print('[OK] All architectures OK')


## Cell 4 — Training Utilities

In [ ]:
# ── 4.1 Albumentations transforms — Standard Aug + F1–F7 TropAug ─────────────
#
# Standard augmentation (S1–S5 baseline):
#   HorizontalFlip p=0.5 · VerticalFlip p=0.3
#   ColorJitter (brightness/contrast/saturation=0.2, hue=0.1) p=0.5
#
# Tropical-Aware Augmentation F1–F7 (S6 proposed):
#   F1 Bright sunlight  RandomBrightnessContrast (+0.3,+0.5)  \
#   F2 Deep shadow      RandomBrightnessContrast (−0.4,−0.1)  / OneOf p=0.50
#   F3 Motion blur      MotionBlur kernel 7–15                  p=0.30
#   F4 Leaf occlusion   CoarseDropout 1–4 holes max 60px        p=0.40
#   F5 Rain droplets    RandomRain heavy                         p=0.30
#   F6 Fog/haze         RandomFog coef 0.15–0.30                 p=0.30
#   F7 JPEG compression ImageCompression quality 60–85           p=0.50
#
# Reference: Buslaev et al. (2020) Information 11(2), 125.

_LIGHTING_P  = 0.50
_BLUR_P      = 0.30
_OCCLUDE_P   = 0.40
_RAIN_P      = 0.30
_FOG_P       = 0.30
_COMPRESS_P  = 0.50


def get_train_transform(mode: str = 'standard') -> A.Compose:
    """mode: 'standard' | 'tropical'  — returns albumentations Compose"""
    base = [
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2,
                      saturation=0.2, hue=0.1, p=0.5),
    ]
    if mode == 'standard':
        aug = base
    elif mode == 'tropical':
        aug = base + [
            # F1/F2: lighting — OneOf prevents bright+dark simultaneously
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=(0.3, 0.5),
                                           contrast_limit=(0.0, 0.2), p=1.0),
                A.RandomBrightnessContrast(brightness_limit=(-0.4, -0.1),
                                           contrast_limit=(-0.2, 0.0), p=1.0),
            ], p=_LIGHTING_P),
            A.MotionBlur(blur_limit=(7, 15), p=_BLUR_P),             # F3
            A.CoarseDropout(num_holes_range=(1, 4),
                            hole_height_range=(20, 60),
                            hole_width_range=(20, 60),
                            fill_value=0, p=_OCCLUDE_P),             # F4
            A.RandomRain(rain_type='heavy',
                         drop_length=20, drop_width=1,
                         blur_value=3, p=_RAIN_P),                   # F5
            A.RandomFog(fog_coef_range=(0.15, 0.30),
                        alpha_coef=0.1, p=_FOG_P),                   # F6
            A.ImageCompression(quality_range=(60, 85),
                               compression_type='jpeg',
                               p=_COMPRESS_P),                       # F7
        ]
    else:
        raise ValueError(f'Unknown mode: {mode!r}')
    aug += [A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return A.Compose(aug)


VAL_TRANSFORM = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# Quick sanity check
_img = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
for _m in ['standard', 'tropical']:
    _out = get_train_transform(_m)(image=_img)['image']
    assert _out.shape == (3, IMG_SIZE, IMG_SIZE), f'Shape error in {_m}'
print('[OK] Transforms OK  standard & tropical (F1–F7)')
print(f'   F1/F2 lighting p={_LIGHTING_P} | F3 blur p={_BLUR_P} | '
      f'F4 occlude p={_OCCLUDE_P}')
print(f'   F5 rain p={_RAIN_P} | F6 fog p={_FOG_P} | F7 JPEG p={_COMPRESS_P}')


# ── 4.2 AlbumentationsDataset ────────────────────────────────────────────────
class AlbumentationsDataset(ImageFolder):
    """ImageFolder that uses an albumentations Compose instead of torchvision transforms"""
    def __init__(self, root, alb_transform=None, **kwargs):
        super().__init__(root, transform=None, **kwargs)
        self.alb_transform = alb_transform

    def __getitem__(self, index):
        path, label = self.samples[index]
        img = np.array(Image.open(path).convert('RGB'))
        if self.alb_transform is not None:
            img = self.alb_transform(image=img)['image']
        return img, label


# ── 4.3 replace_head — supports every architecture ───────────────────────────
def replace_head(model, arch: str, n_classes: int):
    """
    Replace classification head → n_classes.
    ConvNeXt (timm): head.fc (NormMlpClassifierHead)
    ViT (timm):      head (Linear)
    ResNet:          fc
    DenseNet:        classifier
    EfficientNet:    classifier (timm)
    """
    replaced = False
    if hasattr(model, 'head'):
        head = model.head
        if hasattr(head, 'fc') and isinstance(head.fc, nn.Linear):
            # ConvNeXt timm: NormMlpClassifierHead → .fc
            in_f = head.fc.in_features
            head.fc = nn.Linear(in_f, n_classes).to(DEVICE)
            replaced = True
        elif isinstance(head, nn.Linear):
            # ViT timm: head is Linear directly
            model.head = nn.Linear(head.in_features, n_classes).to(DEVICE)
            replaced = True
    if not replaced and hasattr(model, 'fc') and isinstance(model.fc, nn.Linear):
        model.fc = nn.Linear(model.fc.in_features, n_classes).to(DEVICE)
        replaced = True
    if not replaced and hasattr(model, 'classifier'):
        c = model.classifier
        if isinstance(c, nn.Linear):
            model.classifier = nn.Linear(c.in_features, n_classes).to(DEVICE)
        else:
            model.classifier[-1] = nn.Linear(
                c[-1].in_features, n_classes).to(DEVICE)
        replaced = True
    if not replaced:
        raise ValueError(
            f'replace_head: cannot find head for arch={arch}. '
            f'Head children: {[n for n,_ in model.head.named_children()]}')
    return model


# ── 4.4 DataLoader helpers ────────────────────────────────────────────────────
def make_loaders(data_path, aug_mode='standard', val_split=0.15,
                 batch_size=BATCH_SIZE):
    ds = AlbumentationsDataset(data_path,
                               alb_transform=get_train_transform(aug_mode))
    n_val   = int(len(ds) * val_split)
    n_train = len(ds) - n_val
    ds_tr, ds_vl = random_split(ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(SEED))
    ds_vl_eval = deepcopy(ds_vl)
    ds_vl_eval.dataset.alb_transform = VAL_TRANSFORM
    return (DataLoader(ds_tr, batch_size=batch_size, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=True,
        prefetch_factor=4, persistent_workers=True),
            DataLoader(ds_vl_eval, batch_size=batch_size, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True,
        prefetch_factor=4, persistent_workers=True),
            ds.classes)


def make_test_loader(test_root, class_names=None, batch_size=None):
    if batch_size is None:
        batch_size = BATCH_SIZE
    if class_names is None:
        class_names = CLASS_NAMES
    ds = FilteredImageFolder(test_root, class_names, transform=VAL_TRANSFORM)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        prefetch_factor=PREFETCH_FACTOR,
                        persistent_workers=True)
    return loader, class_names

# ── 4.5 TTA — 8 real views ───────────────────────────────────────────────────
# V1 original · V2 h-flip · V3 v-flip · V4 both flips
# V5 F7-JPEG  · V6 F7+hflip · V7 F1-mild bright · V8 F2-mild shadow

def _tv(*augs):
    return A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), *augs,
                      A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                      ToTensorV2()])

TTA_TRANSFORMS = [
    _tv(),
    _tv(A.HorizontalFlip(p=1.0)),
    _tv(A.VerticalFlip(p=1.0)),
    _tv(A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)),
    _tv(A.ImageCompression(quality_range=(60,85),
                           compression_type='jpeg', p=1.0)),
    _tv(A.HorizontalFlip(p=1.0),
        A.ImageCompression(quality_range=(60,85),
                           compression_type='jpeg', p=1.0)),
    _tv(A.RandomBrightnessContrast(brightness_limit=(0.1,0.2),
                                   contrast_limit=(0.0,0.1), p=1.0)),
    _tv(A.RandomBrightnessContrast(brightness_limit=(-0.2,-0.05),
                                   contrast_limit=(-0.1,0.0), p=1.0)),
]


@torch.no_grad()
def tta_predict(model, loader, n_aug: int = 8):
    """
    Real TTA: average softmax over n_aug distinct augmented views.
    Each view applies a different albumentations transform to raw PIL images.
    Returns (preds, labels, probs_avg).
    """
    assert n_aug <= len(TTA_TRANSFORMS)
    model.eval()
    all_labels = np.array([lbl for _, lbl in loader.dataset.samples])
    probs_sum  = None

    for tfm in TTA_TRANSFORMS[:n_aug]:
        orig = loader.dataset.alb_transform
        loader.dataset.alb_transform = tfm
        v_probs = []
        for imgs, _ in DataLoader(loader.dataset, batch_size=BATCH_SIZE,
                                   shuffle=False, num_workers=NUM_WORKERS,
                                   pin_memory=True,
        prefetch_factor=4, persistent_workers=True):
            v_probs.append(torch.softmax(
                model(imgs.to(DEVICE)), dim=1).cpu())
        loader.dataset.alb_transform = orig
        batch = torch.cat(v_probs, dim=0)
        probs_sum = batch if probs_sum is None else probs_sum + batch

    probs_avg = probs_sum / n_aug
    preds     = probs_avg.argmax(dim=1).numpy()
    return preds, all_labels, probs_avg.numpy()


# ── 4.6 SelectiveHead — for evaluating S3/S5/S6 on Thai data ────────────────
# model trained on the union (MAX_CLASSES) → select only the Thai class logits
def get_union_classes(sources=('PlantVillage','PlantDoc','SLD')):
    ds_list = [AlbumentationsDataset(DATASET_PATHS[s], alb_transform=VAL_TRANSFORM)
               for s in sources if Path(DATASET_PATHS[s]).exists()]
    return sorted(set(c for d in ds_list for c in d.classes)), ds_list

def get_thai_indices_in_union(union_classes, thai_classes):
    indices, missing = [], []
    for tc in thai_classes:
        if tc in union_classes:
            indices.append(union_classes.index(tc))
        else:
            indices.append(-1)
            missing.append(tc)
    if missing:
        print(f'  Note: Classes absent from source datasets: {missing}')
        print(f'      → Excluded from evaluation (3-class setup)')
    return indices


class SelectiveHead(nn.Module):
    """Select logits by index — used when model output > NUM_CLASSES"""
    def __init__(self, base_model, indices):
        super().__init__()
        self.base    = base_model
        self.indices = indices
    def forward(self, x):
        logits = self.base(x)
        out = torch.full((logits.size(0), len(self.indices)),
                         float('-inf'), device=logits.device)
        for i, idx in enumerate(self.indices):
            if idx >= 0:
                out[:, i] = logits[:, idx]
        return out


# ── 4.7 Train one epoch ───────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct, total = 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda'):
            out  = model(imgs); loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for imgs, lbs in loader:
        preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
        labels.extend(lbs.numpy())
    return np.array(preds), np.array(labels)


def compute_metrics(preds, labels, class_names):
    acc  = (preds == labels).mean() * 100
    f1   = f1_score(labels, preds, average='macro', zero_division=0) * 100
    prec = precision_score(labels, preds, average='macro', zero_division=0) * 100
    rec  = recall_score(labels, preds, average='macro', zero_division=0) * 100
    pcf1 = f1_score(labels, preds, average=None, zero_division=0) * 100
    return {'acc': round(acc,2), 'f1': round(f1,2),
            'precision': round(prec,2), 'recall': round(rec,2),
            'per_class_f1': {c: round(v,2)
                             for c,v in zip(class_names, pcf1)}}


def train_model(arch, train_loader, val_loader, class_names,
                epochs=EPOCHS, patience=PATIENCE, lr=LR,
                save_path=None, verbose=True):
    n_cls     = len(class_names)
    model     = build_model(arch, num_classes=n_cls)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.amp.GradScaler(device='cuda')

    best_acc, best_state, no_improve = 0., None, 0
    history = {'train_loss':[], 'train_acc':[], 'val_acc':[]}

    for epoch in range(1, epochs+1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                          optimizer, criterion, scaler)
        vp, vl = evaluate(model, val_loader)
        val_acc = (vp == vl).mean() * 100
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc*100)
        history['val_acc'].append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  Ep {epoch:3d}/{epochs} | loss {tr_loss:.4f} | '
                  f'train {tr_acc*100:.2f}% | val {val_acc:.2f}% | '
                  f'best {best_acc:.2f}% | {time.time()-t0:.1f}s')

        if no_improve >= patience:
            print(f'  EarlyStopping @ ep {epoch} (best {best_acc:.2f}%)')
            break

    model.load_state_dict(best_state)
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        torch.save(best_state, save_path)
    return model, best_acc, history

print('[OK] Training utilities ready')
print(f'   TTA: {len(TTA_TRANSFORMS)} real views (V1-V4 flips, V5-V6 F7-JPEG, V7-V8 lighting)')


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct, total = 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda'):
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total
print('[OK] gradient clipping ready')

In [ ]:
BATCH_SIZE = 256  # 85GB RAM → batch 256

# RAM optimization: cache the whole dataset in RAM (85GB available)
# less disk I/O → 3-5x faster on A100
class CachedMultiSourceDataset(torch.utils.data.Dataset):
    def __init__(self, dataset_paths, aug_mode='standard'):
        tmp = [AlbumentationsDataset(p, alb_transform=VAL_TRANSFORM)
               for p in dataset_paths if Path(p).exists()]
        self.union_classes = sorted(set(c for d in tmp for c in d.classes))
        self.class_to_idx  = {c: i for i, c in enumerate(self.union_classes)}
        print('  Caching images to RAM...', end=' ', flush=True)
        self.images, self.labels = [], []
        for p in dataset_paths:
            if not Path(p).exists(): continue
            ds = AlbumentationsDataset(p, alb_transform=VAL_TRANSFORM)
            for img_path, local_idx in ds.samples:
                img = np.array(
                    Image.open(img_path).convert('RGB')
                         .resize((IMG_SIZE, IMG_SIZE)))
                self.images.append(img)
                self.labels.append(
                    self.class_to_idx[ds.classes[local_idx]])
        self.images = np.stack(self.images)
        print(f'done — {len(self.images):,} imgs '
              f'({self.images.nbytes/1e9:.1f} GB)')
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return None, self.labels[i]

def make_cached_loaders(aug_mode='standard', val_split=0.15):
    source_paths = [DATASET_PATHS[s]
                    for s in ['PlantVillage','PlantDoc','SLD']
                    if Path(DATASET_PATHS[s]).exists()]
    full_ds = CachedMultiSourceDataset(source_paths, aug_mode=aug_mode)
    n_val   = int(len(full_ds) * val_split)
    n_train = len(full_ds) - n_val
    idx_tr, idx_vl = torch.utils.data.random_split(
        range(len(full_ds)), [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED))

    class _SubDS(torch.utils.data.Dataset):
        def __init__(self, base, indices, transform):
            self.base = base; self.indices = list(indices); self.tfm = transform
        def __len__(self): return len(self.indices)
        def __getitem__(self, i):
            img = self.tfm(image=self.base.images[self.indices[i]])['image']
            return img, self.base.labels[self.indices[i]]

    tr_loader = DataLoader(
        _SubDS(full_ds, idx_tr, get_train_transform(aug_mode)),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True,
        prefetch_factor=2, persistent_workers=True)
    vl_loader = DataLoader(
        _SubDS(full_ds, idx_vl, VAL_TRANSFORM),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True,
        prefetch_factor=2, persistent_workers=True)
    return tr_loader, vl_loader, full_ds.union_classes

print('[OK] CachedMultiSourceDataset + make_cached_loaders ready')
print(f'   BATCH_SIZE: {BATCH_SIZE}')

In [ ]:


def train_model_ablation(arch, train_loader, val_loader, class_names,
                         epochs=20, patience=5, lr=LR,
                         save_path=None, verbose=True, freeze_epochs=5):
    model     = build_model(arch, num_classes=len(class_names))
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = torch.amp.GradScaler(device='cuda')
    best_acc, best_state, no_improve = 0., None, 0
    history  = {'train_loss':[], 'train_acc':[], 'val_acc':[]}
    optimizer = None; scheduler = None

    for epoch in range(1, epochs+1):
        if epoch == 1:
            for name, param in model.named_parameters():
                if not any(x in name for x in ['head','fc','classifier']):
                    param.requires_grad = False
            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=lr*10, weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=freeze_epochs)
            if verbose: print(f'  Ep 1–{freeze_epochs}: backbone frozen')
        elif epoch == freeze_epochs + 1:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=lr, weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=epochs-freeze_epochs)
            if verbose: print(f'  Ep {freeze_epochs+1}+: full fine-tune')

        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                          optimizer, criterion, scaler)
        vp, vl  = evaluate(model, val_loader)
        val_acc = (vp == vl).mean() * 100
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc*100)
        history['val_acc'].append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = deepcopy(model.state_dict())
            no_improve = 0
            if save_path:
                Path(save_path).parent.mkdir(parents=True, exist_ok=True)
                torch.save(best_state, save_path)
        else:
            no_improve += 1

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  Ep {epoch:3d}/{epochs} | loss {tr_loss:.4f} | '
                  f'train {tr_acc*100:.2f}% | val {val_acc:.2f}% | '
                  f'best {best_acc:.2f}% | {time.time()-t0:.1f}s')
        if no_improve >= patience:
            if verbose: print(f'  EarlyStopping @ ep {epoch} (best {best_acc:.2f}%)')
            break

    model.load_state_dict(best_state)
    return model, best_acc, history

print('[OK] train_model_ablation ready (frozen backbone + gradient clipping)')

## Run order note: S2, S3 → S4 → S5 → S6, then load `df_results` (cell "load from results_all.csv") before checking the best architecture

## Cell 5 — S1: PV→PV (Within-domain Baseline)
| Scenario | Train | Test | Purpose |
|----------|-------|------|---------|
| S1 | PlantVillage | PlantVillage | Within-domain upper bound · ΔAcc=0 |

In [ ]:
def make_test_loader_raw(data_path, batch_size=None):
    if batch_size is None:
        batch_size = BATCH_SIZE
    ds = AlbumentationsDataset(data_path, alb_transform=VAL_TRANSFORM)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        prefetch_factor=PREFETCH_FACTOR,
                        persistent_workers=True)
    return loader, ds.classes

In [ ]:
# ── 5.1 S1: Within-domain baseline ──────────────────────────────────────────
print('Scenario S1: PV → PV (within-domain)')
RESULTS = []

for arch, label in zip(ARCHITECTURES, ARCH_LABELS):
    print(f'\n  [{arch}]')
    save_path = f'{MODELS_DIR}/S1_{arch}.pt'

    if Path(save_path).exists():
        print('  checkpoint found — loading')
        model = build_model(arch, num_classes=38)
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    else:
        tr_loader, vl_loader, tr_classes = make_loaders(
            DATASET_PATHS['PlantVillage'], aug_mode='standard')
        model, _, _ = train_model(arch, tr_loader, vl_loader,
                                  tr_classes, save_path=save_path)

    test_loader, test_classes = make_test_loader_raw(DATASET_PATHS['PlantVillage'])
    preds, labels_arr = evaluate(model, test_loader)
    metrics = compute_metrics(preds, labels_arr, test_classes)
    RESULTS.append({'Scenario':'S1','Label':'PV→PV (within-domain)',
                    'Architecture':label,'Arch_id':arch,
                    'Train':'PlantVillage','Test':'PlantVillage',**metrics})
    print(f'  → S1 Acc: {metrics["acc"]:.2f}% | F1: {metrics["f1"]:.2f}%')
    del model; torch.cuda.empty_cache()

pd.DataFrame(RESULTS).to_csv(f'{TABLES_DIR}/results_S1.csv', index=False)
print('\n[OK] S1 complete')


## Cell 6 — S2: PV→PlantDoc · S3: PV→SLD (Cross-dataset eval only)
| Scenario | Train | Test | Purpose |
|----------|-------|------|---------|
| S2 | PlantVillage | PlantDoc | Cross-dataset non-Thai |
| S3 | PlantVillage | SLD (3-class) | Cross-dataset non-Thai |

> **Note:** the original S2 compared mismatched class indices — the paper uses **S2-fixed** (end of the notebook) instead. *Superseded by S2-fixed; not reported.*

In [ ]:
# ── 6.1 S2: PV → PlantDoc (eval only — load S1, no replace head) ──────────────
print('Scenario S2: PV → PlantDoc (cross-dataset, eval only)')

for arch, label in zip(ARCHITECTURES, ARCH_LABELS):
    print(f'\n  [{arch}]')
    s1_path = f'{MODELS_DIR}/S1_{arch}.pt'
    if not Path(s1_path).exists():
        print(f'  [WARNING] S1 checkpoint not found — skip'); continue

    model = build_model(arch, num_classes=38)
    model.load_state_dict(torch.load(s1_path, map_location=DEVICE))
    model.to(DEVICE).eval()

    pd_loader, pd_classes = make_test_loader_raw(DATASET_PATHS['PlantDoc'])
    preds, labels_arr = evaluate(model, pd_loader)
    metrics = compute_metrics(preds, labels_arr, pd_classes)
    RESULTS.append({'Scenario':'S2','Label':'PV→PlantDoc (cross-dataset)',
                    'Architecture':label,'Arch_id':arch,
                    'Train':'PlantVillage','Test':'PlantDoc',**metrics})
    print(f'  → S2 Acc: {metrics["acc"]:.2f}% | F1: {metrics["f1"]:.2f}%')
    del model; torch.cuda.empty_cache()

# ── 6.2 S3: PV → SLD (eval only — load S1, no replace head) ──────────────────
print('\nScenario S3: PV → SLD (cross-dataset, eval only)')

for arch, label in zip(ARCHITECTURES, ARCH_LABELS):
    print(f'\n  [{arch}]')
    s1_path = f'{MODELS_DIR}/S1_{arch}.pt'
    if not Path(s1_path).exists():
        print(f'  [WARNING] S1 checkpoint not found — skip'); continue

    model = build_model(arch, num_classes=38)
    model.load_state_dict(torch.load(s1_path, map_location=DEVICE))
    model.to(DEVICE).eval()

    sld_loader, sld_classes = make_test_loader_raw(DATASET_PATHS['SLD'])
    preds, labels_arr = evaluate(model, sld_loader)
    metrics = compute_metrics(preds, labels_arr, sld_classes)
    RESULTS.append({'Scenario':'S3','Label':'PV→SLD (cross-dataset)',
                    'Architecture':label,'Arch_id':arch,
                    'Train':'PlantVillage','Test':'SLD',**metrics})
    print(f'  → S3 Acc: {metrics["acc"]:.2f}% | F1: {metrics["f1"]:.2f}%')
    del model; torch.cuda.empty_cache()

pd.DataFrame(RESULTS).to_csv(f'{TABLES_DIR}/results_S2_S3.csv', index=False)
print('\n[OK] S2–S3 complete')

In [ ]:
# check replace_head for S4
model_check = build_model('resnet50', num_classes=38)
model_check.load_state_dict(torch.load(f'{MODELS_DIR}/S1_resnet50.pt', map_location=DEVICE))
model_check = replace_head(model_check, 'resnet50', NUM_CLASSES)

# check head size after replace
key = next((k for k in model_check.state_dict() if 'fc.weight' in k), None)
print(f'head size after replace: {model_check.state_dict()[key].shape[0]}')
print(f'expected: {NUM_CLASSES} (3)')

# check test loader
test_loader, _ = make_test_loader(THAI_TEST)
print(f'test images: {len(test_loader.dataset.samples)}')
print(f'classes: {test_loader.dataset.class_names}')

## Cell 7 — S4: PV→Thai (Zero-shot, eval only)
| Scenario | Train | Test | Purpose |
|----------|-------|------|---------|
| S4 | PlantVillage | Thai | Zero-shot Thai · domain gap baseline |

> **Note:** the original S4 uses a randomly initialised head, so it is not used in the paper — the paper uses **S4-fair v2**. *Superseded by S4-fair v2; not reported.*

In [ ]:
# ── 7.1 S4: PV → Thai (zero-shot — load S1, replace head 38→3) ──────────────
print('Scenario S4: PV → Thai (zero-shot, eval only)')

for arch, label in zip(ARCHITECTURES, ARCH_LABELS):
    print(f'\n  [{arch}]')
    s1_path = f'{MODELS_DIR}/S1_{arch}.pt'
    if not Path(s1_path).exists():
        print(f'  [WARNING] S1 checkpoint not found — skip'); continue

    save_path = f'{MODELS_DIR}/S4_{arch}.pt'

    torch.manual_seed(SEED)  # fix seed before replace_head
    model = build_model(arch, num_classes=38)
    model.load_state_dict(torch.load(s1_path, map_location=DEVICE))
    model = replace_head(model, arch, NUM_CLASSES)  # NUM_CLASSES=3
    model.to(DEVICE).eval()

    # save checkpoint
    if not Path(save_path).exists():
        torch.save(model.state_dict(), save_path)
        print(f'  saved → {save_path}')

    test_loader, _ = make_test_loader(THAI_TEST)
    preds, labels_arr = evaluate(model, test_loader)
    metrics = compute_metrics(preds, labels_arr, CLASS_NAMES)
    RESULTS.append({'Scenario':'S4','Label':'PV→Thai (zero-shot)',
                    'Architecture':label,'Arch_id':arch,
                    'Train':'PlantVillage','Test':'Thai',**metrics})
    print(f'  → S4 Acc: {metrics["acc"]:.2f}% | F1: {metrics["f1"]:.2f}%')
    del model; torch.cuda.empty_cache()

pd.DataFrame(RESULTS).to_csv(f'{TABLES_DIR}/results_S4.csv', index=False)
print('\n[OK] S4 complete')

In [ ]:
# Re-evaluate S5 for every architecture with the new loader
import pandas as pd

S5_RECORDS_NEW = []
test_loader_new, _ = make_test_loader(THAI_TEST)

for arch, label in zip(ARCHITECTURES, ARCH_LABELS):
    pt = f'{MODELS_DIR}/S5_{arch}.pt'
    if not Path(pt).exists():
        print(f'[WARNING] {arch} checkpoint not found — skip')
        continue

    model_r = build_model(arch, num_classes=MAX_CLASSES_S5)
    model_r.load_state_dict(torch.load(pt, map_location=DEVICE))
    model_r = SelectiveHead(model_r, THAI_IDX_S5).to(DEVICE)
    model_r.eval()

    preds, labels_arr = evaluate(model_r, test_loader_new)
    m = compute_metrics(preds, labels_arr, CLASS_NAMES)
    print(f'  [{arch}] Acc: {m["acc"]:.2f}% | F1: {m["f1"]:.2f}%')

    S5_RECORDS_NEW.append({
        'Scenario': 'S5',
        'Label': 'Multi→Thai (std aug)',
        'Architecture': label,
        'Arch_id': arch,
        'acc': m['acc'], 'f1': m['f1'],
        'precision': m['precision'], 'recall': m['recall']
    })

    del model_r; torch.cuda.empty_cache()

# overwrite the existing CSV
df_s5_new = pd.DataFrame(S5_RECORDS_NEW)
df_s5_new.to_csv(f'{TABLES_DIR}/results_S5.csv', index=False)
print(f'\n[OK] S5 re-evaluated and saved ({len(S5_RECORDS_NEW)} rows)')

## Cell 8 — S5: Multi→Thai (Multi-source, standard aug)
| Scenario | Train | Test | Purpose |
|----------|-------|------|---------|
| S5 | PV+PlantDoc+SLD (standard aug) | Thai | Multi-source zero-shot |

In [ ]:
# check that make_multisource_loaders is available
try:
    tr_loader, vl_loader, union = make_multisource_loaders(aug_mode='standard')
    print(f'[OK] ready')
    print(f'Union classes: {len(union)}')
    print(f'Train samples: {len(tr_loader.dataset)}')
    print(f'Val samples: {len(vl_loader.dataset)}')
except NameError:
    print('[ERROR] not run yet — run the cell that defines MultiSourceDataset first')

In [ ]:
# ── 8.1 S5: Multi-source + standard aug → Thai ──────────────────────────────
# Train: PV+PlantDoc+SLD standard aug
# Eval:  SelectiveHead — select logits by Thai class index in the union
# Thai test set uses only 3 classes: Healthy, RedRot, Rust
print('Scenario S5: Multi-source → Thai (standard aug)')
print()

UNION_CLASSES_S5, _ = get_union_classes()
THAI_IDX_S5 = get_thai_indices_in_union(UNION_CLASSES_S5, CLASS_NAMES)
MAX_CLASSES_S5 = len(UNION_CLASSES_S5)
print(f'Union classes: {MAX_CLASSES_S5}')  # must be 70
print(f'Thai class indices in union: '
      f'{dict(zip(CLASS_NAMES, THAI_IDX_S5))}')  # must not contain -1

assert MAX_CLASSES_S5 == 70, f'[ERROR] wrong union: got {MAX_CLASSES_S5}, expected 70'
assert -1 not in THAI_IDX_S5, f'[ERROR] wrong Thai indices: contains -1'
print('Scenario S5: Multi-source → Thai (standard aug)')
print()

UNION_CLASSES_S5, _ = get_union_classes()
THAI_IDX_S5 = get_thai_indices_in_union(UNION_CLASSES_S5, CLASS_NAMES)
MAX_CLASSES_S5 = len(UNION_CLASSES_S5)

# check before running
print(f'Union classes: {MAX_CLASSES_S5}')
print(f'Thai class indices: {dict(zip(CLASS_NAMES, THAI_IDX_S5))}')
assert MAX_CLASSES_S5 == 70, f'[ERROR] stop: Union={MAX_CLASSES_S5}, expected 70 — check DATASET_PATHS["SLD"]'
assert -1 not in THAI_IDX_S5, f'[ERROR] stop: Thai indices contain -1'
print('[OK] Union correct — start training')
print()

for arch, label in zip(ARCHITECTURES, ARCH_LABELS):
    print(f'\n  [{arch}]')
    save_path = f'{MODELS_DIR}/S5_{arch}.pt'

    if Path(save_path).exists():
        print('  checkpoint found — loading')
        model = build_model(arch, num_classes=MAX_CLASSES_S5)
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    else:
        # use make_multisource_loaders instead of ConcatDataset
        tr_loader, vl_loader, _ = make_multisource_loaders(aug_mode='standard')
        model, _, _ = train_model(arch, tr_loader, vl_loader,
                                  list(range(MAX_CLASSES_S5)),
                                  save_path=save_path, verbose=True)

    # SelectiveHead: map union logits → Thai 3 classes
    model_sel = SelectiveHead(model, THAI_IDX_S5).to(DEVICE)
    model_sel.eval()

    test_loader, _ = make_test_loader(THAI_TEST)
    preds, labels_arr = evaluate(model_sel, test_loader)
    metrics = compute_metrics(preds, labels_arr, CLASS_NAMES)
    RESULTS.append({'Scenario':'S5','Label':'Multi→Thai (standard aug)',
                    'Architecture':label,'Arch_id':arch,
                    'Train':'PV+PlantDoc+SLD (standard)','Test':'Thai',**metrics})
    print(f'  → S5 Acc: {metrics["acc"]:.2f}% | F1: {metrics["f1"]:.2f}%')
    del model, model_sel; torch.cuda.empty_cache()

pd.DataFrame(RESULTS).to_csv(f'{TABLES_DIR}/results_S5.csv', index=False)
print('\n[OK] S5 complete')

## Cell 9 — S6: Multi+F1–F7+TTA→Thai
| Scenario | Train | Test | Purpose |
|----------|-------|------|---------|
| S6 | PV+PlantDoc+SLD (F1–F7 TropAug) + TTA | Thai | Tropical-aware augmentation + TTA |

In [ ]:
# check S6 before running
_, _, UNION_CLASSES_S6 = make_cached_loaders(aug_mode='tropical')
MAX_CLASSES_S6 = len(UNION_CLASSES_S6)
THAI_IDX_S6 = get_thai_indices_in_union(UNION_CLASSES_S6, CLASS_NAMES)

print(f'Union S5: {MAX_CLASSES_S5} | Union S6: {MAX_CLASSES_S6}')
print(f'Thai IDX S5: {dict(zip(CLASS_NAMES, THAI_IDX_S5))}')
print(f'Thai IDX S6: {dict(zip(CLASS_NAMES, THAI_IDX_S6))}')
print(f'Union matches: {MAX_CLASSES_S5 == MAX_CLASSES_S6}')
print(f'Thai IDX matches: {THAI_IDX_S5 == THAI_IDX_S6}')

assert MAX_CLASSES_S5 == MAX_CLASSES_S6, f'[ERROR] union mismatch: S5={MAX_CLASSES_S5} S6={MAX_CLASSES_S6}'
assert THAI_IDX_S5 == THAI_IDX_S6, f'[ERROR] Thai indices mismatch'
print('[OK] S5 and S6 use the same union — comparable')

In [ ]:
import os
print(f'MAX_CLASSES_S6: {MAX_CLASSES_S6}')
print(f'THAI_IDX_S6: {dict(zip(CLASS_NAMES, THAI_IDX_S6))}')

arch  = 'convnext_base'
label = 'ConvNeXt-B'
save_path = f'{MODELS_DIR}/S6_{arch}.pt'

print(f'[{arch}] — S6')

if Path(save_path).exists():
    sd = torch.load(save_path, map_location='cpu')
    key = next((k for k in sd if any(x in k for x in
         ['fc.weight','head.weight','head.fc.weight','classifier.weight'])), None)
    saved_sz = sd[key].shape[0] if key else 0
    has_nan  = any(v.isnan().any() for v in sd.values())

    if saved_sz != MAX_CLASSES_S6 or has_nan:
        os.remove(save_path)
        print(f'  checkpoint invalid → retrain')
    else:
        print(f'  checkpoint OK (head={saved_sz}) — skip training')
        model     = build_model(arch, num_classes=MAX_CLASSES_S6)
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))
        model_sel = SelectiveHead(model, THAI_IDX_S6).to(DEVICE)
        model_sel.eval()
        test_loader, _ = make_test_loader(THAI_TEST)
        preds, labels_arr, probs = tta_predict(model_sel, test_loader, n_aug=8)
        metrics = compute_metrics(preds, labels_arr, CLASS_NAMES)
        if not any(r['Scenario']=='S6' and r['Arch_id']==arch for r in RESULTS):
            RESULTS.append({
                'Scenario':'S6',
                'Label':'Multi+F1–F7+TTA→Thai (proposed)',
                'Architecture':label, 'Arch_id':arch,
                'Train':'PV+PlantDoc+SLD+TropAug(F1-F7)',
                'Test':'Thai', **metrics})
        S6_RECORDS.append({'arch':arch,'label':label,
                           'preds':preds,'labels':labels_arr,'probs':probs})
        print(f'  → S6 Acc: {metrics["acc"]:.2f}% | F1: {metrics["f1"]:.2f}%')
        del model, model_sel; torch.cuda.empty_cache()

if not Path(save_path).exists():
    # persistent_workers=False prevents worker crashes
    tr_loader, vl_loader, _ = make_cached_loaders(aug_mode='tropical')
    model, best_val, history = train_model_s6(
        arch, tr_loader, vl_loader,
        list(range(MAX_CLASSES_S6)),
        save_path=save_path, verbose=True)

    model_sel = SelectiveHead(model, THAI_IDX_S6).to(DEVICE)
    model_sel.eval()
    test_loader, _ = make_test_loader(THAI_TEST)
    preds, labels_arr, probs = tta_predict(model_sel, test_loader, n_aug=8)
    metrics = compute_metrics(preds, labels_arr, CLASS_NAMES)
    RESULTS.append({
        'Scenario':'S6',
        'Label':'Multi+F1–F7+TTA→Thai (proposed)',
        'Architecture':label, 'Arch_id':arch,
        'Train':'PV+PlantDoc+SLD+TropAug(F1-F7)',
        'Test':'Thai', **metrics})
    S6_RECORDS.append({'arch':arch,'label':label,
                       'preds':preds,'labels':labels_arr,'probs':probs})
    print(f'  → S6 Acc (8-view TTA): {metrics["acc"]:.2f}% | F1: {metrics["f1"]:.2f}%')
    del model, model_sel; torch.cuda.empty_cache()

In [ ]:
import torch
import pandas as pd

s6_archs = [
    ('resnet50', 'ResNet-50'),
    ('efficientnet_b4', 'EfficientNet-B4'),
    ('densenet121', 'DenseNet-121'),
    ('vit_b16', 'ViT-B/16'),
    ('convnext_base', 'ConvNeXt-B'),
]

s6_records = []
test_loader, _ = make_test_loader(THAI_TEST)

for arch, label in s6_archs:
    pt = f'{MODELS_DIR}/S6_{arch}.pt'
    if not Path(pt).exists():
        print(f'[WARNING] {arch}: checkpoint not found — skip')
        continue

    print(f'  [{arch}]')
    model = build_model(arch, num_classes=MAX_CLASSES_S6)
    model.load_state_dict(torch.load(pt, map_location=DEVICE))
    model_sel = SelectiveHead(model, THAI_IDX_S6).to(DEVICE)
    model_sel.eval()

    preds, labels_arr, probs = tta_predict(model_sel, test_loader, n_aug=8)
    metrics = compute_metrics(preds, labels_arr, CLASS_NAMES)

    s6_records.append({
        'Scenario': 'S6',
        'Label': 'Multi+F1–F7+TTA→Thai (proposed)',
        'Architecture': label, 'Arch_id': arch,
        'Train': 'PV+PlantDoc+SLD+TropAug(F1-F7)',
        'Test': 'Thai', **metrics})

    print(f'  → Acc={metrics["acc"]:.2f}% | F1={metrics["f1"]:.2f}% | '
          f'Precision={metrics["precision"]:.2f}% | Recall={metrics["recall"]:.2f}%')
    del model, model_sel; torch.cuda.empty_cache()

df_s6 = pd.DataFrame(s6_records)
df_s6.to_csv(f'{TABLES_DIR}/results_S6.csv', index=False)
print(f'\n[OK] results_S6.csv saved ({len(df_s6)} rows)')
print(df_s6[['Scenario','Architecture','acc','f1']].to_string(index=False))

In [ ]:
# evaluate S6 efficientnet_b4
arch  = 'efficientnet_b4'
label = 'EfficientNet-B4'
pt    = f'{MODELS_DIR}/S6_{arch}.pt'

if not Path(pt).exists():
    print(f'[ERROR] checkpoint not found')
else:
    model = build_model(arch, num_classes=MAX_CLASSES_S6)
    model.load_state_dict(torch.load(pt, map_location=DEVICE))
    model_sel = SelectiveHead(model, THAI_IDX_S6).to(DEVICE)
    model_sel.eval()

    test_loader, _ = make_test_loader(THAI_TEST)
    preds, labels_arr, probs = tta_predict(model_sel, test_loader, n_aug=8)
    metrics = compute_metrics(preds, labels_arr, CLASS_NAMES)

    # remove the old row with precision=0 and add the new one
    df_results = df_results[~((df_results['Scenario']=='S6') &
                               (df_results['Arch_id']==arch))]
    new_row = pd.DataFrame([{
        'Scenario':'S6', 'Label':'Multi+F1–F7+TTA→Thai (proposed)',
        'Architecture':label, 'Arch_id':arch,
        'Train':'PV+PlantDoc+SLD+TropAug(F1-F7)', 'Test':'Thai',
        **metrics}])
    df_results = pd.concat([df_results, new_row], ignore_index=True)
    RESULTS = df_results.to_dict('records')

    # save
    df_results.to_csv(f'{TABLES_DIR}/results_all.csv', index=False)
    print(f'→ S6 Acc: {metrics["acc"]:.2f}% | F1: {metrics["f1"]:.2f}%')
    print(f'Total rows: {len(df_results)}')
    del model, model_sel; torch.cuda.empty_cache()

In [ ]:
# evaluate S6 efficientnet_b4 and add it to df_results immediately
arch  = 'efficientnet_b4'
label = 'EfficientNet-B4'
pt    = f'{MODELS_DIR}/S6_{arch}.pt'

model = build_model(arch, num_classes=MAX_CLASSES_S6)
model.load_state_dict(torch.load(pt, map_location=DEVICE))
model_sel = SelectiveHead(model, THAI_IDX_S6).to(DEVICE)
model_sel.eval()

test_loader, _ = make_test_loader(THAI_TEST)
preds, labels_arr, probs = tta_predict(model_sel, test_loader, n_aug=8)
metrics = compute_metrics(preds, labels_arr, CLASS_NAMES)

# add to df_results
new_row = pd.DataFrame([{
    'Scenario':'S6', 'Label':'Multi+F1–F7+TTA→Thai (proposed)',
    'Architecture':label, 'Arch_id':arch,
    'Train':'PV+PlantDoc+SLD+TropAug(F1-F7)', 'Test':'Thai',
    **metrics}])
df_results = pd.concat([df_results, new_row], ignore_index=True)
RESULTS = df_results.to_dict('records')

# save results_all.csv and results_S6.csv
df_results.to_csv(f'{TABLES_DIR}/results_all.csv', index=False)
df_results[df_results['Scenario']=='S6'].to_csv(
    f'{TABLES_DIR}/results_S6.csv', index=False)

print(f'→ Acc={metrics["acc"]:.2f}% | F1={metrics["f1"]:.2f}%')
print(f'Total rows: {len(df_results)}')
print(df_results.groupby('Scenario')['Arch_id'].count())
del model, model_sel; torch.cuda.empty_cache()

## Cell 10 — Ablation Study (Config A–D)
| Config | TropAug | TTA | Multi-source | Purpose |
|--------|---------|-----|-------------|---------|
| A Baseline | ✗ | ✗ | ✗ | Starting point · S4 score |
| B +TropAug | ✓ | ✗ | ✗ | Aug contribution |
| C +TTA     | ✓ | ✓ | ✗ | TTA contribution |
| D Full(S6) | ✓ | ✓ | ✓ | Full system |

In [ ]:
import pandas as pd

# load from the cleaned results_all.csv
df_results = pd.read_csv(f'{TABLES_DIR}/results_all.csv')
RESULTS = df_results.to_dict('records')

print(f'[OK] df_results loaded: {len(df_results)} rows')
print(df_results.groupby('Scenario')['Arch_id'].count())
print()
print(df_results.groupby('Scenario')['acc'].max().to_string())

In [ ]:
import os
for cfg in ['B', 'C']:
    pt = f'{MODELS_DIR}/abl_{cfg}_{best_arch_id}.pt'
    if Path(pt).exists():
        os.remove(pt)
        print(f'Deleted abl_{cfg}_{best_arch_id}.pt')

In [ ]:
# ── 10.1 Ablation Study (Config A–D) ─────────────────────────────────────────
import os

# ── get the best architecture from S6 ─────────────────────────────────────────
df_s6 = df_results[df_results['Scenario']=='S6']
best_arch_label = df_s6.loc[df_s6['acc'].idxmax(), 'Architecture']
best_arch_id    = df_s6.loc[df_s6['acc'].idxmax(), 'Arch_id']
print(f'Best architecture (S6): {best_arch_label} ({best_arch_id})')

# ── get S4, S5, S6 results of the best arch ──────────────────────────────────
row_s4 = df_results[(df_results['Scenario']=='S4') &
                    (df_results['Arch_id']==best_arch_id)].iloc[0]
row_s5 = df_results[(df_results['Scenario']=='S5') &
                    (df_results['Arch_id']==best_arch_id)].iloc[0]
row_s6 = df_results[(df_results['Scenario']=='S6') &
                    (df_results['Arch_id']==best_arch_id)].iloc[0]

# ── Ablation uses S4/S5/S6 results directly — no retraining ──────────────────
# Config A = S4 (PV→Thai zero-shot, no aug, no TTA, single-source)
# Config B = S5 (Multi-source, standard aug, no TTA)
# Config C = S6 no TTA (Multi-source, TropAug, no TTA)  ← re-evaluated
# Config D = S6 (Multi-source, TropAug, TTA) = proposed

ablation_rows = []

# Config A
ablation_rows.append({
    'Config': 'A: Baseline (S4)',
    'Tropical_Aug': '✗', 'TTA': '✗', 'Multi_source': '✗',
    'acc': row_s4['acc'], 'f1': row_s4['f1'],
    'precision': row_s4['precision'], 'recall': row_s4['recall'],
    'Δ_vs_A': 0.0,
})
print(f'  Config A → Acc: {row_s4["acc"]:.2f}% | Δ: +0.00%')

# Config B = S5
ablation_rows.append({
    'Config': 'B: +Multi-source (S5)',
    'Tropical_Aug': '✗', 'TTA': '✗', 'Multi_source': '✓',
    'acc': row_s5['acc'], 'f1': row_s5['f1'],
    'precision': row_s5['precision'], 'recall': row_s5['recall'],
    'Δ_vs_A': round(row_s5['acc'] - row_s4['acc'], 2),
})
print(f'  Config B → Acc: {row_s5["acc"]:.2f}% | Δ: {ablation_rows[-1]["Δ_vs_A"]:+.2f}%')

# Config C = S6 no TTA
print(f'\n  Config C: +TropAug (no TTA) — evaluate S6 model...')
model_c = build_model(best_arch_id, num_classes=MAX_CLASSES_S6)
model_c.load_state_dict(torch.load(f'{MODELS_DIR}/S6_{best_arch_id}.pt', map_location=DEVICE))
model_c = SelectiveHead(model_c, THAI_IDX_S6).to(DEVICE)
model_c.eval()
test_loader, _ = make_test_loader(THAI_TEST)
preds_c, labels_c = evaluate(model_c, test_loader)
metrics_c = compute_metrics(preds_c, labels_c, CLASS_NAMES)
ablation_rows.append({
    'Config': 'C: +TropAug (no TTA)',
    'Tropical_Aug': '✓', 'TTA': '✗', 'Multi_source': '✓',
    'acc': metrics_c['acc'], 'f1': metrics_c['f1'],
    'precision': metrics_c['precision'], 'recall': metrics_c['recall'],
    'Δ_vs_A': round(metrics_c['acc'] - row_s4['acc'], 2),
})
print(f'  Config C → Acc: {metrics_c["acc"]:.2f}% | Δ: {ablation_rows[-1]["Δ_vs_A"]:+.2f}%')
del model_c; torch.cuda.empty_cache()

# Config D = S6 full
ablation_rows.append({
    'Config': 'D: Full S6 (proposed)',
    'Tropical_Aug': '✓', 'TTA': '✓', 'Multi_source': '✓',
    'acc': row_s6['acc'], 'f1': row_s6['f1'],
    'precision': row_s6['precision'], 'recall': row_s6['recall'],
    'Δ_vs_A': round(row_s6['acc'] - row_s4['acc'], 2),
})
print(f'  Config D → Acc: {row_s6["acc"]:.2f}% | Δ: {ablation_rows[-1]["Δ_vs_A"]:+.2f}%')

# ── Summary ───────────────────────────────────────────────────────────────────
df_ablation = pd.DataFrame(ablation_rows)
print('\n[OK] Ablation Study complete')
print(df_ablation[['Config','Tropical_Aug','TTA','Multi_source',
                   'acc','f1','Δ_vs_A']].to_string(index=False))

df_ablation.to_excel(f'{TABLES_DIR}/ablation_study.xlsx', index=False)
df_ablation.to_csv(f'{TABLES_DIR}/ablation_study.csv', index=False)
print(f'\nSaved → {TABLES_DIR}/ablation_study.xlsx')

## Cell 10b — Systematic Degradation Analysis (SDA) + Tropical Degradation Severity Score (TDSS)

| Factor | Code | Augmentation | Tropical Phenomenon |
|--------|------|-------------|--------------------|
| F1/F2 | Lighting | RandomBrightnessContrast | Strong sunlight / deep shade |
| F3 | Blur | MotionBlur kernel 7–15 | Motion / wind |
| F4 | Occlusion | CoarseDropout 1–4 holes | Overlapping leaves |
| F5 | Rain | RandomRain heavy | Raindrops on leaves |
| F6 | Fog | RandomFog coef 0.15–0.30 | Haze from high humidity |
| F7 | JPEG | ImageCompression 60–85 | Low-quality smartphone camera |

**SDA Protocol:**
- Load the best S6 model (trained, no retraining)
- Use the Thai test set (3-class filtered)
- Evaluate 8 conditions: Clean + F1–F7, one at a time (p=1.0)
- Measure ΔAcc_fi = Acc(Clean) − Acc(+Fi) → degradation impact per factor

**TDSS (Tropical Degradation Severity Score):**
```
TDSS = Σ (wi × ΔAcc_fi)  where wi = normalized weight per factor
```
→ A single scalar summarising how difficult the tropical domain is for a given dataset

In [ ]:
# ── Cell 10b: Systematic Degradation Analysis (SDA) + TDSS ──────────────────
import torch, numpy as np, pandas as pd
import albumentations as A
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from albumentations.pytorch import ToTensorV2
from pathlib import Path
from torch.utils.data import DataLoader
from PIL import Image
from sklearn.metrics import f1_score

# ── 10b.1 Define degradation conditions ──────────────────────────────────────
DEGRADATION_CONDITIONS = {
    'Clean':        [],
    'F1_Sunlight':  [A.RandomBrightnessContrast(
                        brightness_limit=(0.3, 0.5),
                        contrast_limit=(0.0, 0.2), p=1.0)],
    'F2_Shadow':    [A.RandomBrightnessContrast(
                        brightness_limit=(-0.4, -0.1),
                        contrast_limit=(-0.2, 0.0), p=1.0)],
    'F3_Blur':      [A.MotionBlur(blur_limit=(7, 15), p=1.0)],
    'F4_Occlusion': [A.CoarseDropout(
                        num_holes_range=(1, 4),
                        hole_height_range=(20, 60),
                        hole_width_range=(20, 60),
                        fill_value=0, p=1.0)],
    'F5_Rain':      [A.RandomRain(
                        rain_type='heavy', drop_length=20,
                        drop_width=1, blur_value=3, p=1.0)],
    'F6_Fog':       [A.RandomFog(
                        fog_coef_range=(0.15, 0.30),
                        alpha_coef=0.1, p=1.0)],
    'F7_JPEG':      [A.ImageCompression(
                        quality_range=(60, 85),
                        compression_type='jpeg', p=1.0)],
    'All_F1F7':     [
        A.RandomBrightnessContrast(
            brightness_limit=(0.3, 0.5), contrast_limit=(0.0, 0.2), p=1.0),
        A.MotionBlur(blur_limit=(7, 15), p=1.0),
        A.CoarseDropout(
            num_holes_range=(1, 4),
            hole_height_range=(20, 60),
            hole_width_range=(20, 60),
            fill_value=0, p=1.0),
        A.RandomRain(
            rain_type='heavy', drop_length=20,
            drop_width=1, blur_value=3, p=1.0),
        A.RandomFog(
            fog_coef_range=(0.15, 0.30),
            alpha_coef=0.1, p=1.0),
        A.ImageCompression(
            quality_range=(60, 85),
            compression_type='jpeg', p=1.0),
    ],
}

FACTOR_LABELS = {
    'Clean':        'Clean\n(Baseline)',
    'F1_Sunlight':  'F1: Sunlight',
    'F2_Shadow':    'F2: Shadow',
    'F3_Blur':      'F3: Motion\nBlur',
    'F4_Occlusion': 'F4: Leaf\nOcclusion',
    'F5_Rain':      'F5: Rain\nDroplets',
    'F6_Fog':       'F6: Fog/\nHaze',
    'F7_JPEG':      'F7: JPEG\nCompression',
    'All_F1F7':     'All Factors\n(Combined)',
}

# ── 10b.2 Load best S6 model ──────────────────────────────────────────────────
print(f'Loading best S6 model: {best_arch_label} ({best_arch_id})')
model_sda = build_model(best_arch_id, num_classes=MAX_CLASSES_S6)
model_sda.load_state_dict(
    torch.load(f'{MODELS_DIR}/S6_{best_arch_id}.pt', map_location=DEVICE))
model_sda = SelectiveHead(model_sda, THAI_IDX_S6).to(DEVICE)
model_sda.eval()

# ── 10b.3 DegradedDataset ─────────────────────────────────────────────────────
class DegradedDataset(torch.utils.data.Dataset):
    def __init__(self, root, class_names, extra_augs):
        self.class_names = class_names
        self.samples     = []
        for label_idx, cls in enumerate(class_names):
            cls_dir = Path(root) / cls
            if not cls_dir.exists(): continue
            for ext in ('*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG'):
                for p in cls_dir.glob(ext):
                    self.samples.append((str(p), label_idx))
        self.alb_transform = A.Compose(
            [A.Resize(IMG_SIZE, IMG_SIZE)] +
            extra_augs +
            [A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        img = np.array(Image.open(path).convert('RGB'))
        return self.alb_transform(image=img)['image'], label

# ── 10b.4 Evaluate ────────────────────────────────────────────────────────────
sda_rows  = []
acc_clean = None

for cond_name, extra_augs in DEGRADATION_CONDITIONS.items():
    ds = DegradedDataset(THAI_TEST, CLASS_NAMES, extra_augs)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True,
                        prefetch_factor=2, persistent_workers=False)
    preds_arr, labels_arr = evaluate(model_sda, loader)
    acc = (preds_arr == labels_arr).mean() * 100
    f1  = f1_score(labels_arr, preds_arr, average='macro', zero_division=0) * 100
    if cond_name == 'Clean':
        acc_clean = acc
    delta = round(acc_clean - acc, 2) if acc_clean is not None else 0.0
    sda_rows.append({
        'Condition': cond_name,
        'Label':     FACTOR_LABELS[cond_name],
        'Acc':       round(acc, 2),
        'F1':        round(f1, 2),
        'DeltaAcc':  delta,
    })
    print(f'  {cond_name:<15} Acc={acc:.2f}%  F1={f1:.2f}%  ΔAcc={delta:+.2f}%')

df_sda = pd.DataFrame(sda_rows)
df_sda.to_excel(f'{TABLES_DIR}/systematic_degradation_analysis.xlsx', index=False)
df_sda.to_csv(f'{TABLES_DIR}/systematic_degradation_analysis.csv', index=False)
print(f'\n[OK] SDA saved')

# ── 10b.5 TDSS ────────────────────────────────────────────────────────────────
individual_factors = ['F1_Sunlight','F2_Shadow','F3_Blur',
                      'F4_Occlusion','F5_Rain','F6_Fog','F7_JPEG']
df_factors = df_sda[df_sda['Condition'].isin(individual_factors)].copy()
df_factors['DeltaAcc_abs'] = df_factors['DeltaAcc'].abs()
total_impact = df_factors['DeltaAcc_abs'].sum()
df_factors['Weight'] = df_factors['DeltaAcc_abs'] / (total_impact + 1e-8)
df_factors['Contribution'] = df_factors['Weight'] * df_factors['DeltaAcc_abs']
TDSS = df_factors['Contribution'].sum()

print()
print('='*55)
print('  TROPICAL DEGRADATION SEVERITY SCORE (TDSS)')
print('='*55)
print(f'  TDSS = {TDSS:.4f}')
print()
print(f'  {"Factor":<15} {"ΔAcc":>8} {"Weight":>8} {"Contribution":>12}')
print(f'  {"-"*15} {"-"*8} {"-"*8} {"-"*12}')
for _, row in df_factors.iterrows():
    print(f'  {row["Condition"]:<15} {row["DeltaAcc"]:>7.2f}% '
          f'{row["Weight"]:>7.3f}  {row["Contribution"]:>11.4f}')
print(f'  {"-"*15} {"-"*8} {"-"*8} {"-"*12}')
print(f'  {"TDSS":<15} {"":>8} {"":>8} {TDSS:>11.4f}')
print('='*55)

import json as _json
tdss_result = {
    'TDSS': round(float(TDSS), 4),
    'dataset': 'Thai Local Sugarcane (3-class)',
    'model': best_arch_label,
    'acc_clean': round(float(acc_clean), 2),
    'factors': df_factors[['Condition','Acc','F1','DeltaAcc',
                            'Weight','Contribution']].to_dict('records')
}
with open(f'{TABLES_DIR}/tdss_result.json', 'w') as f:
    _json.dump(tdss_result, f, indent=2)
print(f'[OK] TDSS saved')

# ── 10b.6 Figure (CEA style + vivid) ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(COL2, 4.2))
fig.patch.set_facecolor('white')

conds  = [r['Condition'] for r in sda_rows]
labels = [r['Label']     for r in sda_rows]
accs   = [r['Acc']       for r in sda_rows]
deltas = [r['DeltaAcc']  for r in sda_rows]
x_pos  = np.arange(len(conds))

colors_bar = []
for c in conds:
    if c == 'Clean':      colors_bar.append('#4CAF50')
    elif c == 'All_F1F7': colors_bar.append('#9C27B0')
    else:                 colors_bar.append('#2196F3')

bars = axes[0].bar(x_pos, accs, color=colors_bar, alpha=0.85,
                   edgecolor='white', linewidth=0.8, zorder=2)
axes[0].axhline(y=accs[0], color='#4CAF50', linestyle='--',
                alpha=0.8, linewidth=1.0)

for bar, acc, delta in zip(bars, accs, deltas):
    axes[0].text(bar.get_x() + bar.get_width()/2.,
                 bar.get_height() + 0.3,
                 f'{acc:.1f}%', ha='center', va='bottom',
                 fontsize=6.5, fontweight='bold')
    if delta > 0:
        axes[0].text(bar.get_x() + bar.get_width()/2.,
                     max(bar.get_height() - 4, 2),
                     f'−{delta:.1f}', ha='center', va='top',
                     fontsize=6, color='#E53935', fontweight='bold')

axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(
    [r['Label'] for r in sda_rows],
    fontsize=6.5, ha='right', rotation=40)
axes[0].set_ylabel('Accuracy (%)', fontsize=9)
axes[0].set_title('(a) Systematic Degradation Analysis (SDA)\n'
                  'Accuracy under each tropical degradation factor (p=1.0)',
                  fontsize=9, fontweight='bold')
axes[0].set_ylim(0, max(accs) * 1.20)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].grid(axis='y', alpha=0.25, linestyle='--', linewidth=0.6, zorder=0)
axes[0].tick_params(axis='x', pad=2)

patch_clean  = mpatches.Patch(color='#4CAF50', label='Clean baseline')
patch_single = mpatches.Patch(color='#2196F3', label='Single factor (F1–F7)')
patch_all    = mpatches.Patch(color='#9C27B0', label='All factors combined')
axes[0].legend(handles=[patch_clean, patch_single, patch_all],
               fontsize=6.5, loc='upper right')

fac_names   = df_factors['Condition'].str.replace('_', ' ').tolist()
fac_deltas  = df_factors['DeltaAcc'].abs().tolist()
fac_weights = df_factors['Weight'].tolist()
colors_pie  = ['#FF5722','#FF9800','#FFC107',
               '#8BC34A','#00BCD4','#9C27B0','#E91E63']
wedges, texts, autotexts = axes[1].pie(
    fac_weights,
    labels=[f'{n}\n(ΔAcc={d:.1f}%)' for n, d in zip(fac_names, fac_deltas)],
    autopct='%1.1f%%', colors=colors_pie,
    startangle=90, pctdistance=0.72,
    textprops={'fontsize': 6.5})
for at in autotexts:
    at.set_fontsize(6.5); at.set_fontweight('bold')
axes[1].set_title(f'(b) TDSS Factor Weight Breakdown\n'
                  f'TDSS = {TDSS:.4f}',
                  fontsize=9, fontweight='bold')

plt.suptitle(f'SDA — {best_arch_label} on Thai Sugarcane (3-class)',
             fontsize=10, fontweight='bold', y=1.02)
plt.tight_layout(pad=0.5)

sda_fig_path = f'{FIGURES_DIR}/fig_sda_tdss.pdf'
plt.savefig(sda_fig_path, dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(sda_fig_path.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'[OK] SDA+TDSS figure saved')

del model_sda; torch.cuda.empty_cache()
print(f'\n[OK] Cell 10b complete — TDSS={TDSS:.4f}')


## Cell 10c — Domain Adaptation Baseline Comparison (CORAL · AdaBN)

Compares **Tropical-Aware Augmentation (S6)** with standard DA methods:

| Method | Type | Retraining | Principle |
|--------|------|-----------|-----------|
| **S5** | Multi-source std aug | yes (done) | Baseline without DA |
| **AdaBN** | Test-time adaptation | no (post-process) | Replace BN stats with target stats |
| **CORAL** | Feature alignment | no (post-process) | Align feature covariance |
| **S6** | Training-time aug | yes (done) | Tropical-Aware Aug + TTA |

> AdaBN and CORAL run on the existing S5 checkpoint — **no retraining needed**, so they add little GPU time.

> **Note:** original DA baseline — the paper uses **A10 (corrected DA baselines)**. *Superseded by A10; not reported.*

In [ ]:
# ── Cell 10c: Domain Adaptation Baseline Comparison ──────────────────────────
import torch, torch.nn as nn
import numpy as np
import pandas as pd
from copy import deepcopy
from pathlib import Path
from torch.utils.data import DataLoader

DA_RESULTS = []

# ── 10c.1 AdaBN ──────────────────────────────────────────────────────────────
def apply_adabn(model, target_loader, device=DEVICE):
    model_ada = deepcopy(model)
    for m in model_ada.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.reset_running_stats()
            m.training = True
            m.momentum  = None
    model_ada.eval()
    for m in model_ada.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.training = True
    with torch.no_grad():
        for imgs, _ in target_loader:
            model_ada(imgs.to(device))
    for m in model_ada.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.training = False
    return model_ada

# ── 10c.2 CORAL ───────────────────────────────────────────────────────────────
@torch.no_grad()
def extract_penultimate_features(model, arch, loader, device=DEVICE):
    features = []
    def hook_fn(module, input, output):
        features.append(output.detach().cpu())

    if arch == 'resnet50':          hook_layer = model.base.avgpool
    elif arch == 'efficientnet_b4': hook_layer = model.base.global_pool
    elif arch == 'densenet121':     hook_layer = model.base.features
    elif arch == 'vit_b16':         hook_layer = model.base.norm
    elif arch == 'convnext_base':   hook_layer = model.base.head.norm
    else: raise ValueError(f'Unknown arch: {arch}')

    handle = hook_layer.register_forward_hook(hook_fn)
    model.eval()
    for imgs, _ in loader:
        model(imgs.to(device))
    handle.remove()

    feats = torch.cat(features, dim=0)
    if feats.dim() > 2:
        feats = feats.flatten(1)
    return feats

def coral_loss(source_feats, target_feats):
    d = source_feats.size(1)
    Cs = torch.cov(source_feats.T)
    Ct = torch.cov(target_feats.T)
    return torch.norm(Cs - Ct, p='fro') ** 2 / (4 * d * d)

def apply_coral_whitening(model, arch, source_loader, target_loader, device=DEVICE):
    src_feats = extract_penultimate_features(model, arch, source_loader, device)
    tgt_feats = extract_penultimate_features(model, arch, target_loader, device)
    coral_dist = coral_loss(src_feats, tgt_feats)
    Cs = torch.cov(src_feats.T).float()
    Ct = torch.cov(tgt_feats.T).float()
    eps = 1e-5
    try:
        Us, Ss, _ = torch.linalg.svd(Cs)
        Ws = Us @ torch.diag(1.0 / torch.sqrt(Ss + eps)) @ Us.T
        Ut, St, _ = torch.linalg.svd(Ct)
        Wt = Ut @ torch.diag(torch.sqrt(St + eps)) @ Ut.T
        transform = (Wt @ Ws).to(device)
        aligned = True
    except Exception as e:
        print(f'  [WARNING] CORAL SVD failed: {e}')
        transform = None
        aligned = False
    return transform, coral_dist.item(), aligned

# ── 10c.3 Run DA baselines ────────────────────────────────────────────────────
print('Domain Adaptation Baseline Comparison')
print('='*55)
print(f'Architecture: {best_arch_label} ({best_arch_id})')
print(f'Test set: Thai Local 3-class (Healthy/RedRot/Rust)')
print()

s5_path = f'{MODELS_DIR}/S5_{best_arch_id}.pt'
if not Path(s5_path).exists():
    print(f'[WARNING] S5 checkpoint not found')
else:
    model_s5_base = build_model(best_arch_id, num_classes=MAX_CLASSES_S5)
    model_s5_base.load_state_dict(torch.load(s5_path, map_location=DEVICE))
    model_s5_sel = SelectiveHead(model_s5_base, THAI_IDX_S5).to(DEVICE)
    model_s5_sel.eval()

    test_loader_da, _ = make_test_loader(THAI_TEST)

    # ── S5 Baseline ───────────────────────────────────────────────────────────
    print('S5 — Multi-source + standard aug (no DA):')
    preds_s5, labels_s5 = evaluate(model_s5_sel, test_loader_da)
    m_s5 = compute_metrics(preds_s5, labels_s5, CLASS_NAMES)
    print(f'  Acc: {m_s5["acc"]:.2f}%  F1: {m_s5["f1"]:.2f}%')
    DA_RESULTS.append({'Method':'S5 (no DA)',
                       'Acc': m_s5['acc'], 'F1': m_s5['f1'],
                       'DA_type': 'None', 'Retrain': 'Yes (done)'})

    # ── AdaBN ─────────────────────────────────────────────────────────────────
    print('\nAdaBN — Adaptive Batch Normalisation:')
    has_bn = any(isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))
                 for m in model_s5_base.modules())
    if not has_bn:
        print(f'  [WARNING] {best_arch_label} has no BN layers → AdaBN N/A')
        DA_RESULTS.append({'Method':'AdaBN', 'Acc': float('nan'),
                           'F1': float('nan'), 'DA_type': 'Test-time',
                           'Retrain': 'No',
                           'Note': f'N/A — {best_arch_label} has no BN'})
    else:
        model_ada = apply_adabn(model_s5_sel, test_loader_da)
        preds_ada, labels_ada = evaluate(model_ada, test_loader_da)
        m_ada = compute_metrics(preds_ada, labels_ada, CLASS_NAMES)
        delta_ada = round(m_ada['acc'] - m_s5['acc'], 2)
        print(f'  Acc: {m_ada["acc"]:.2f}%  F1: {m_ada["f1"]:.2f}%')
        print(f'  ΔAcc vs S5: {delta_ada:+.2f}%')
        DA_RESULTS.append({'Method':'AdaBN', 'Acc': m_ada['acc'],
                           'F1': m_ada['f1'], 'DA_type': 'Test-time',
                           'Retrain': 'No', 'DeltaVsS5': delta_ada})
        del model_ada; torch.cuda.empty_cache()

    # ── CORAL ─────────────────────────────────────────────────────────────────
    print('\nCORAL — Deep CORAL (post-hoc, feature distance reported):')
    source_loader_da, _ = make_test_loader_raw(DATASET_PATHS['SLD'])
    print(f'  Source: {len(source_loader_da.dataset.samples)} imgs | '
          f'{source_loader_da.dataset.classes}')
    try:
        coral_tfm, coral_dist, aligned = apply_coral_whitening(
            model_s5_sel, best_arch_id, source_loader_da, test_loader_da)
        print(f'  CORAL distance (source↔target): {coral_dist:.4f}')
        if aligned:
            preds_coral, labels_coral = evaluate(model_s5_sel, test_loader_da)
            m_coral = compute_metrics(preds_coral, labels_coral, CLASS_NAMES)
            print(f'  Acc: {m_coral["acc"]:.2f}%  F1: {m_coral["f1"]:.2f}%')
            print(f'  Note: post-hoc approx — true CORAL requires retrain')
            DA_RESULTS.append({'Method':'CORAL (post-hoc)',
                               'Acc': m_coral['acc'], 'F1': m_coral['f1'],
                               'DA_type': 'Feature alignment', 'Retrain': 'No',
                               'CORAL_dist': round(coral_dist, 4)})
    except Exception as e:
        print(f'  [ERROR] CORAL error: {e}')
        DA_RESULTS.append({'Method':'CORAL', 'Acc': float('nan'),
                           'DA_type': 'Feature alignment', 'Retrain': 'No',
                           'Note': str(e)})

    # ── S6 (ours) ─────────────────────────────────────────────────────────────
    print('\nS6 (Ours) — Multi-source + TropAug + TTA:')
    row_s6 = df_results[(df_results['Scenario']=='S6') &
                        (df_results['Arch_id']==best_arch_id)]
    if not row_s6.empty:
        acc_s6 = row_s6.iloc[0]['acc']
        f1_s6  = row_s6.iloc[0]['f1']
        print(f'  Acc: {acc_s6:.2f}%  F1: {f1_s6:.2f}%')
        DA_RESULTS.append({'Method':'S6 — TropAug+TTA (ours)',
                           'Acc': acc_s6, 'F1': f1_s6,
                           'DA_type': 'Training-time aug',
                           'Retrain': 'Yes (done)'})

    # ── Summary ───────────────────────────────────────────────────────────────
    print()
    print('='*55)
    print('Summary — DA Baseline Comparison')
    print('='*55)
    df_da = pd.DataFrame(DA_RESULTS)
    print(df_da[['Method','Acc','F1','DA_type','Retrain']].to_string(index=False))

    df_da.to_excel(f'{TABLES_DIR}/da_baseline_comparison.xlsx', index=False)
    df_da.to_csv(f'{TABLES_DIR}/da_baseline_comparison.csv', index=False)
    print(f'\n[OK] DA comparison saved')

    # ── Figure (CEA style) ────────────────────────────────────────────────────
    df_plot = df_da.dropna(subset=['Acc']).copy()
    methods = df_plot['Method'].tolist()
    accs    = df_plot['Acc'].tolist()
    f1s     = df_plot['F1'].tolist()
    x_pos   = np.arange(len(methods))
    bar_w   = 0.35

    colors_method = []
    for m in methods:
        if 'ours' in m.lower(): colors_method.append('#E53935')
        elif 'AdaBN' in m:     colors_method.append('#1E88E5')
        elif 'CORAL' in m:     colors_method.append('#43A047')
        else:                  colors_method.append('#757575')

    fig, ax = plt.subplots(figsize=(COL2, 3.2))
    fig.patch.set_facecolor('white')
    b1 = ax.bar(x_pos - bar_w/2, accs, bar_w, color=colors_method,
                alpha=0.9, edgecolor='white', linewidth=0.8, label='Accuracy')
    b2 = ax.bar(x_pos + bar_w/2, f1s, bar_w, color=colors_method,
                alpha=0.55, edgecolor='white', linewidth=0.8,
                hatch='//', label='Macro-F1')

    for bar, val in zip(b1, accs):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{val:.1f}%', ha='center', va='bottom',
                fontsize=8, fontweight='bold')

    ax.set_xticks(x_pos)
    ax.set_xticklabels(methods, fontsize=8, ha='center')
    ax.set_ylabel('Score (%)', fontsize=9)
    ax.set_title(
        f'Domain Adaptation Baseline Comparison — {best_arch_label}\n'
        f'Thai Sugarcane (3-class: Healthy/RedRot/Rust)',
        fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(accs)*1.2)
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.25, linestyle='--', linewidth=0.6)

    plt.tight_layout(pad=0.5)
    da_fig = f'{FIGURES_DIR}/fig_da_comparison.pdf'
    plt.savefig(da_fig, dpi=300, bbox_inches='tight', format='pdf')
    plt.savefig(da_fig.replace('.pdf','.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'[OK] DA comparison figure saved')

    del model_s5_base, model_s5_sel
    torch.cuda.empty_cache()
    print('\n[OK] Cell 10c complete')

## Cell 11 — Statistical Significance Test
McNemar's test: **S4 (no adapt) vs S6 (proposed)** · Bootstrap 95% CI ΔAcc S6−S4


> **Note:** uses the original S4 (random head) — not used in the paper. *Not reported.*

In [ ]:
# ── 11.1 McNemar's test: S4 vs S6 ────────────────────────────────────────────
print("McNemar's Test: S4 (zero-shot, no adapt) vs S6 (proposed +F1–F7+TTA)")
print()

test_loader, _ = make_test_loader(THAI_TEST)
true_labels = np.array([lbl for _, lbl in test_loader.dataset.samples])

# ── S4: load from checkpoint ──────────────────────────────────────────────────
model_s4 = build_model(best_arch_id, num_classes=NUM_CLASSES)
model_s4.load_state_dict(
    torch.load(f'{MODELS_DIR}/S4_{best_arch_id}.pt', map_location=DEVICE))
model_s4.to(DEVICE).eval()
preds_s4, _ = evaluate(model_s4, test_loader)
del model_s4; torch.cuda.empty_cache()

# ── S6: SelectiveHead + 8-view TTA ───────────────────────────────────────────
model_s6 = build_model(best_arch_id, num_classes=MAX_CLASSES_S6)
model_s6.load_state_dict(
    torch.load(f'{MODELS_DIR}/S6_{best_arch_id}.pt', map_location=DEVICE))
model_s6 = SelectiveHead(model_s6, THAI_IDX_S6).to(DEVICE)
model_s6.eval()
preds_s6, _, _ = tta_predict(model_s6, test_loader, n_aug=8)
del model_s6; torch.cuda.empty_cache()

# ── Align ─────────────────────────────────────────────────────────────────────
n = min(len(preds_s4), len(preds_s6), len(true_labels))
preds_s4    = preds_s4[:n]
preds_s6    = preds_s6[:n]
true_labels = true_labels[:n]

correct_s4 = (preds_s4 == true_labels)
correct_s6 = (preds_s6 == true_labels)
b = ((correct_s4) & (~correct_s6)).sum()
c = ((~correct_s4) & (correct_s6)).sum()

table     = [[correct_s4.sum(), b],
             [c, (~correct_s4 & ~correct_s6).sum()]]
result_mc = mcnemar(table, exact=False, correction=True)

print(f'Contingency table:')
print(f'  S4-correct S6-wrong (b) = {b}')
print(f'  S4-wrong   S6-correct (c) = {c}')
print(f'McNemar chi2 = {result_mc.statistic:.4f}')
print(f'p-value      = {result_mc.pvalue:.6f}')
print(f'Result: {"Significant (p<0.05)" if result_mc.pvalue < 0.05 else "Not significant (p>=0.05)"}')

# ── 11.2 Bootstrap 95% CI ΔAcc (S6 − S4) ────────────────────────────────────
print('\nBootstrap 95% CI for ΔAcc (S6 − S4):')
N_BOOT = 2000
rng    = np.random.default_rng(SEED)
delta_boots = np.array([
    (preds_s6[idx := rng.integers(0,n,size=n)] == true_labels[idx]).mean()*100 -
    (preds_s4[idx] == true_labels[idx]).mean()*100
    for _ in range(N_BOOT)
])
ci_low, ci_high = np.percentile(delta_boots, [2.5, 97.5])
obs_delta = (correct_s6.mean() - correct_s4.mean()) * 100
print(f'Observed ΔAcc (S6−S4): {obs_delta:+.2f}%')
print(f'95% CI: [{ci_low:+.2f}%, {ci_high:+.2f}%]')
print(f'Result: {"CI excludes zero → significantly positive" if ci_low > 0 else "CI includes zero"}')

stat_result = {
    'comparison':          'S4 (zero-shot, PV→Thai) vs S6 (proposed +F1-F7+TTA)',
    'test':                'McNemar (exact=False, correction=True)',
    'mcnemar_chi2':        round(result_mc.statistic, 4),
    'mcnemar_pvalue':      round(result_mc.pvalue, 6),
    'significant_p05':     bool(result_mc.pvalue < 0.05),
    'b_s4correct_s6wrong': int(b),
    'c_s4wrong_s6correct': int(c),
    'observed_delta_acc':  round(obs_delta, 2),
    'ci_95_low':           round(ci_low, 2),
    'ci_95_high':          round(ci_high, 2),
    'n_bootstrap':         N_BOOT,
    'n_samples':           int(n),
}
with open(f'{TABLES_DIR}/statistical_test.json', 'w') as f:
    json.dump(stat_result, f, indent=2)
print(f'\n[OK] Statistical test saved')

## Cell 12 — Split Sensitivity (S4 vs S6, 3 splits)

> **Note:** uses the original S4 (random head) — not used in the paper. *Not reported.*

In [ ]:
# ── 12.1 Split sensitivity: S4 vs S6 across 75/80/90 ────────────────────────
print('Split Sensitivity Analysis: S4 (no adapt) vs S6 (proposed)')
split_rows = []

for split_key, paths in THAI_SPLITS.items():
    test_path = paths['test']
    if not Path(test_path).exists():
        print(f'  [WARNING] split {split_key} test not found — skip'); continue

    test_loader_sp, _ = make_test_loader(test_path)

    for arch, label in zip(ARCHITECTURES, ARCH_LABELS):
        # S4: load from the S4 checkpoint
        s4_path = f'{MODELS_DIR}/S4_{arch}.pt'
        if Path(s4_path).exists():
            model_s4 = build_model(arch, num_classes=NUM_CLASSES)
            model_s4.load_state_dict(torch.load(s4_path, map_location=DEVICE))
            model_s4.to(DEVICE).eval()
            p4, l4 = evaluate(model_s4, test_loader_sp)
            m4 = compute_metrics(p4, l4, CLASS_NAMES)
            del model_s4; torch.cuda.empty_cache()
        else:
            print(f'  [WARNING] S4_{arch}.pt not found — skip')
            m4 = {'acc':0.,'f1':0.}

        # S6: SelectiveHead + TTA
        s6_path = f'{MODELS_DIR}/S6_{arch}.pt'
        if Path(s6_path).exists():
            model_s6 = build_model(arch, num_classes=MAX_CLASSES_S6)
            model_s6.load_state_dict(torch.load(s6_path, map_location=DEVICE))
            model_s6 = SelectiveHead(model_s6, THAI_IDX_S6).to(DEVICE)
            model_s6.eval()
            p6, l6, _ = tta_predict(model_s6, test_loader_sp)
            m6 = compute_metrics(p6, l6, CLASS_NAMES)
            del model_s6; torch.cuda.empty_cache()
        else:
            print(f'  [WARNING] S6_{arch}.pt not found — skip')
            m6 = {'acc':0.,'f1':0.}

        delta = round(m6['acc'] - m4['acc'], 2)
        split_rows.append({
            'Split':        f'{split_key}/{100-int(split_key)}',
            'Architecture': label,
            'S4_Acc':       m4['acc'], 'S6_Acc': m6['acc'],
            'ΔAcc (S6-S4)': delta,
            'S4_F1':        m4['f1'],  'S6_F1': m6['f1'],
        })
        print(f'  Split {split_key} | {label:<22} | '
              f'S4: {m4["acc"]:.2f}% → S6: {m6["acc"]:.2f}% (Δ={delta:+.2f}%)')

df_split = pd.DataFrame(split_rows)
df_split.to_excel(f'{TABLES_DIR}/split_sensitivity.xlsx', index=False)
df_split.to_csv(f'{TABLES_DIR}/split_sensitivity.csv', index=False)
print(f'\n[OK] Split sensitivity saved')

## Cell 13 — Grad-CAM Visualization

In [ ]:
# ── 13.1 Grad-CAM — S6 proposed model (CEA style) ────────────────────────────
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

model_cam_base = build_model(best_arch_id, num_classes=MAX_CLASSES_S6)
model_cam_base.load_state_dict(
    torch.load(f'{MODELS_DIR}/S6_{best_arch_id}.pt', map_location=DEVICE))
model_cam = SelectiveHead(model_cam_base, THAI_IDX_S6).to(DEVICE)
model_cam.eval()

def get_target_layer(model, arch):
    m = model.base if hasattr(model, 'base') else model
    if arch == 'resnet50':        return [m.layer4[-1]]
    if arch == 'efficientnet_b4': return [m.blocks[-1][-1].conv_pw]
    if arch == 'densenet121':     return [m.features.denseblock4.denselayer16.conv2]
    if arch == 'vit_b16':         return [m.blocks[-1].attn.proj]
    if arch == 'convnext_base':   return [m.stages[-1].blocks[-1].conv_dw]
    raise ValueError(f'Unknown arch: {arch}')

cam = GradCAM(model=model_cam,
              target_layers=get_target_layer(model_cam, best_arch_id))
thai_path = Path(THAI_TEST)

fig, axes = plt.subplots(len(CLASS_NAMES), 4,
                          figsize=(COL2, COL2 * 0.32 * len(CLASS_NAMES)))
fig.patch.set_facecolor('white')
fig.suptitle(f'Grad-CAM Visualization — S6 Proposed ({best_arch_label})',
             fontsize=10, fontweight='bold', y=1.01)

for row_i, cls in enumerate(CLASS_NAMES):
    cls_imgs = list((thai_path/cls).glob('*.[jp][pn]g'))[:2]
    if len(cls_imgs) < 2:
        cls_imgs += list((thai_path/cls).glob('*.JPG'))[:2-len(cls_imgs)]

    for col_i, img_path in enumerate(cls_imgs):
        img_pil = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        img_np  = np.array(img_pil) / 255.0
        input_t = VAL_TRANSFORM(image=np.array(img_pil))['image']\
                    .unsqueeze(0).to(DEVICE)
        gcam    = cam(input_tensor=input_t, targets=None)[0]
        cam_img = show_cam_on_image(img_np.astype(np.float32),
                                    gcam, use_rgb=True)

        axes[row_i][col_i*2].imshow(img_np)
        axes[row_i][col_i*2].set_title(f'{cls} — Original',
                                        fontsize=8, fontweight='bold', pad=3)
        axes[row_i][col_i*2].axis('off')

        axes[row_i][col_i*2+1].imshow(cam_img)
        axes[row_i][col_i*2+1].set_title(f'{cls} — Grad-CAM',
                                          fontsize=8, fontweight='bold', pad=3)
        axes[row_i][col_i*2+1].axis('off')

plt.tight_layout(pad=0.3, h_pad=0.5, w_pad=0.3)
cam_path = f'{FIGURES_DIR}/fig_gradcam_S6.pdf'
plt.savefig(cam_path, dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(cam_path.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'[OK] Grad-CAM saved → {cam_path}')
del model_cam, model_cam_base; torch.cuda.empty_cache()

## Cell 14 — Confusion Matrix (S4, S5, S6)

In [ ]:
# ── 14.1 Confusion matrices: S4, S5, S6 (CEA style) ─────────────────────────
SHOW_SCENARIOS = ['S4','S5','S6']
SHOW_LABELS    = ['PV→Thai (zero-shot)',
                  'Multi→Thai (standard aug)',
                  'Multi+F1–F7+TTA (proposed)']

fig, axes = plt.subplots(1, 3, figsize=(COL2, 3.2))
fig.patch.set_facecolor('white')
fig.suptitle(f'Confusion Matrices — {best_arch_label}',
             fontsize=10, fontweight='bold', y=1.02)

for idx, (s_id, s_label) in enumerate(zip(SHOW_SCENARIOS, SHOW_LABELS)):
    ax = axes[idx]
    row = df_results[(df_results['Scenario']==s_id) &
                     (df_results['Arch_id']==best_arch_id)]
    if row.empty:
        ax.axis('off'); ax.set_title(f'{s_id}: no data'); continue

    test_loader, _ = make_test_loader(THAI_TEST)

    if s_id == 'S4':
        model_c = build_model(best_arch_id, num_classes=NUM_CLASSES)
        model_c.load_state_dict(
            torch.load(f'{MODELS_DIR}/S4_{best_arch_id}.pt', map_location=DEVICE))
        model_c.to(DEVICE).eval()
        preds_c, labels_c = evaluate(model_c, test_loader)

    elif s_id == 'S5':
        model_c = build_model(best_arch_id, num_classes=MAX_CLASSES_S5)
        model_c.load_state_dict(
            torch.load(f'{MODELS_DIR}/S5_{best_arch_id}.pt', map_location=DEVICE))
        model_c = SelectiveHead(model_c, THAI_IDX_S5).to(DEVICE)
        model_c.eval()
        preds_c, labels_c = evaluate(model_c, test_loader)

    else:  # S6
        model_c = build_model(best_arch_id, num_classes=MAX_CLASSES_S6)
        model_c.load_state_dict(
            torch.load(f'{MODELS_DIR}/S6_{best_arch_id}.pt', map_location=DEVICE))
        model_c = SelectiveHead(model_c, THAI_IDX_S6).to(DEVICE)
        model_c.eval()
        preds_c, labels_c, _ = tta_predict(model_c, test_loader)

    del model_c; torch.cuda.empty_cache()

    cm   = confusion_matrix(labels_c, preds_c, labels=list(range(NUM_CLASSES)))
    cm_n = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8) * 100
    acc  = round((preds_c == labels_c).mean() * 100, 1)

    sns.heatmap(cm_n, ax=ax,
                annot=True, fmt='.1f',
                annot_kws={'size': 8, 'weight': 'bold'},
                cmap='Blues',
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES,
                cbar=False,
                linewidths=0.4,
                linecolor='#dddddd',
                vmin=0, vmax=100)
    ax.set_title(f'{s_id}: {s_label}\nAcc={acc}%',
                 fontsize=9, fontweight='bold', pad=6)
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('True', fontsize=8)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout(pad=0.5)
cm_path = f'{FIGURES_DIR}/fig_confusion_matrices.pdf'
plt.savefig(cm_path, dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(cm_path.replace('.pdf','.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'[OK] Confusion matrices saved → {cm_path}')

## Cell 15 — Inference Latency Benchmark

In [ ]:
# ── 15.2 Plot: Accuracy–Latency tradeoff (CEA style) ─────────────────────────
fig, ax = plt.subplots(figsize=(COL2, 4.0))
fig.patch.set_facecolor('white')

colors = ['#1565C0','#2E7D32','#E65100','#6A1B9A','#B71C1C']

offsets = {
    'ResNet-50':       (-12, 10),
    'EfficientNet-B4': (-75, -14),
    'DenseNet-121':    (-20, 10),
    'ViT-B/16':        (-20, -24),
    'ConvNeXt-B':      (-72, 10),
}

for i, row in df_latency.iterrows():
    ax.scatter(row['Latency_mean_ms'], row['S6_Acc'],
               s=row['Params_M']*10,
               color=colors[i], alpha=0.88,
               edgecolors='white', linewidths=2, zorder=3)
    ox, oy = offsets.get(row['Architecture'], (8, 6))
    ax.annotate(row['Architecture'],
                (row['Latency_mean_ms'], row['S6_Acc']),
                textcoords='offset points',
                xytext=(ox, oy),
                fontsize=8, fontweight='bold',
                color=colors[i])

ax.set_xlim(45, 125)
ax.set_ylim(55, 82)
ax.set_xlabel('CPU Inference Latency (ms/image)', fontsize=9)
ax.set_ylabel('S6 Accuracy on Thai Dataset (%)', fontsize=9)
ax.set_title('Accuracy–Latency Tradeoff — S6 Proposed\n'
             '(bubble size ∝ parameter count)',
             fontsize=10, fontweight='bold')
ax.tick_params(labelsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.6)

plt.tight_layout(pad=0.5)
lat_fig_path = f'{FIGURES_DIR}/fig_latency_tradeoff.pdf'
plt.savefig(lat_fig_path, dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(lat_fig_path.replace('.pdf','.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'[OK] Latency benchmark saved → {lat_fig_path}')

## Cell 16 — CEA Publication-Ready Figures

In [ ]:
# ── CEA Publication Style ─────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'sans-serif',
    'font.sans-serif':    ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':          9,
    'axes.titlesize':     10,
    'axes.labelsize':     9,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
    'legend.fontsize':    8,
    'legend.frameon':     True,
    'legend.edgecolor':   '#cccccc',
    'figure.dpi':         300,
    'axes.linewidth':     0.8,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'xtick.direction':    'out',
    'ytick.direction':    'out',
    'xtick.major.width':  0.8,
    'ytick.major.width':  0.8,
    'savefig.bbox':       'tight',
    'savefig.dpi':        300,
})

# CEA single column = 8.9cm = 3.5in | double column = 18.4cm = 7.24in
COL1 = 3.5   # inches
COL2 = 7.24  # inches

# ── 16.1 Figure 2: Accuracy heatmap ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(COL2, 3.5))
fig.patch.set_facecolor('white')

pivot_acc = df_results.pivot_table(
    index='Architecture', columns='Scenario',
    values='acc', aggfunc='first')
pivot_acc = pivot_acc[['S1','S2','S3','S4','S5','S6']]

sns.heatmap(pivot_acc, ax=ax,
            annot=True, fmt='.1f',
            annot_kws={'size': 8, 'weight': 'bold'},
            cmap='RdYlGn', center=50,
            linewidths=0.4, linecolor='#dddddd',
            cbar_kws={'label': 'Accuracy (%)', 'shrink': 0.8,
                      'aspect': 20})
ax.set_title('Accuracy (%) Across Scenarios and Architectures',
             fontsize=10, fontweight='bold', pad=8)
ax.set_xlabel('Scenario', fontsize=9)
ax.set_ylabel('Architecture', fontsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=8)

scenario_labels_x = ['S1\n(PV→PV)',
                     'S2\n(PV→PlantDoc)',
                     'S3\n(PV→SLD)',
                     'S4\n(PV→Thai\nzero-shot)',
                     'S5\n(Multi→Thai\nstd aug)',
                     'S6\n(Multi+TropAug\n+TTA)']
ax.set_xticklabels(scenario_labels_x, fontsize=8, rotation=0)

plt.tight_layout(pad=0.5)
path_heatmap = f'{FIGURES_DIR}/fig2_acc_heatmap.pdf'
plt.savefig(path_heatmap, dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(path_heatmap.replace('.pdf','.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'[OK] Figure 2 (Accuracy heatmap) saved')

# ── 16.2 Figure 3: Ablation bar chart ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(COL2, 3.2))
fig.patch.set_facecolor('white')

configs = df_ablation['Config'].tolist()
accs    = df_ablation['acc'].tolist()
f1s     = df_ablation['f1'].tolist()
deltas  = df_ablation['Δ_vs_A'].tolist()
x_pos   = np.arange(len(configs))
bar_w   = 0.35

# CEA-friendly colors — printable in greyscale
colors_a = ['#BDBDBD','#66BB6A','#42A5F5','#EF5350']

# (a) Accuracy & F1
bars1 = axes[0].bar(x_pos - bar_w/2, accs, bar_w,
                    color=colors_a, alpha=0.92,
                    edgecolor='white', linewidth=0.8)
bars2 = axes[0].bar(x_pos + bar_w/2, f1s, bar_w,
                    color=colors_a, alpha=0.55,
                    edgecolor='white', linewidth=0.8, hatch='//')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(configs, rotation=20, ha='right', fontsize=8)
axes[0].set_ylabel('Score (%)', fontsize=9)
axes[0].set_title('(a) Ablation Study — Accuracy & Macro-F1',
                  fontsize=10, fontweight='bold')
axes[0].legend([bars1[0], bars2[0]], ['Accuracy', 'Macro-F1'],
               fontsize=8, loc='upper left')
axes[0].set_ylim(0, 108)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
for bar, val in zip(bars1, accs):
    axes[0].text(bar.get_x()+bar.get_width()/2., val+0.8,
                 f'{val:.1f}', ha='center', va='bottom',
                 fontsize=8, fontweight='bold')

# (b) ΔAcc vs Config A
bar_c = ['#BDBDBD' if d <= 0 else '#66BB6A' for d in deltas]
b = axes[1].bar(x_pos, deltas, color=bar_c,
                edgecolor='white', linewidth=0.8, alpha=0.92)
axes[1].axhline(0, color='#333333', linewidth=0.8, linestyle='--')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(configs, rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('ΔAcc vs Config A (%)', fontsize=9)
axes[1].set_title('(b) Improvement over Baseline (Config A)',
                  fontsize=10, fontweight='bold')
axes[1].set_ylim(0, 62)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
for bar, val in zip(b, deltas):
    axes[1].text(bar.get_x()+bar.get_width()/2.,
                 val + 0.8,
                 f'{val:+.1f}%', ha='center', va='bottom',
                 fontsize=8, fontweight='bold')

plt.tight_layout(pad=0.5)
path_abl = f'{FIGURES_DIR}/fig3_ablation.pdf'
plt.savefig(path_abl, dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(path_abl.replace('.pdf','.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'[OK] Figure 3 (Ablation) saved')

## Cell 17 — Export Full Results Table (CEA format)

In [ ]:
# ── 17.1 Compile and export ──────────────────────────────────────────────────
df_main = df_results[['Scenario','Label','Architecture',
                       'acc','f1','precision','recall']].copy()
df_main.columns = ['Scenario','Description','Architecture',
                   'Accuracy (%)','Macro F1 (%)','Precision (%)','Recall (%)']

# ΔAcc relative to S4, only for S4-S6 (all tested on Thai data)
# S1-S3 are tested on other datasets → ΔAcc is meaningless → shown as '-'
s4_accs = df_main[df_main['Scenario']=='S4'].groupby('Architecture')['Accuracy (%)'].first()

def calc_delta(r):
    if r['Scenario'] in ['S4','S5','S6']:
        base = s4_accs.get(r['Architecture'], None)
        if base is not None:
            return round(r['Accuracy (%)'] - base, 2)
    return None

df_main['ΔAcc vs S4 (%)'] = df_main.apply(calc_delta, axis=1)

excel_path = f'{TABLES_DIR}/main_results_S1-S6.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_main.to_excel(writer, sheet_name='Main Results S1-S6', index=False)
    if 'df_ablation' in dir():
        df_ablation.to_excel(writer, sheet_name='Ablation Study', index=False)
    if 'df_latency' in dir():
        df_latency.to_excel(writer, sheet_name='Latency Benchmark', index=False)
    if 'df_sda' in dir():
        df_sda.to_excel(writer, sheet_name='SDA Analysis', index=False)
    if 'df_da' in dir():
        df_da.to_excel(writer, sheet_name='DA Baseline', index=False)
    if 'stat_result' in dir():
        pd.DataFrame([stat_result]).to_excel(
            writer, sheet_name='Statistical Test S4vsS6', index=False)

print(f'[OK] All results saved → {excel_path}')
print()
print('='*65)
print('  EXPERIMENT SUMMARY — Phase 1 (S1–S6) Complete')
print('='*65)
best_per = df_main.groupby('Scenario').apply(
    lambda g: g.loc[g['Accuracy (%)'].idxmax()])
print(best_per[['Scenario','Description','Architecture',
                'Accuracy (%)','Macro F1 (%)','ΔAcc vs S4 (%)']].to_string(index=False))

---
# ADD-ON EXPERIMENTS
**Every time a new runtime starts (including after a Colab disconnect)**, first run these 10 cells above (no training):

`0.1 Mount Drive` → `1.1 Kaggle API` → `1.2 Download` → `1.3 Path mapping` → `1.4 Copy Thai` → `1.5 Filter SLD` → `3.1 Build model` → `4.1 Albumentations transforms` → `def train_one_epoch` → `RAM optimization: cache`

**Skip all other cells above** (2.x, 5.1–17.1, Pre-flight) — some of them retrain models or overwrite earlier results.

Then always run **A0 → A0.1 → A1 → A2 → A3** (if already run, results are loaded from Drive in a few minutes), followed by the cells you need.

# Add-on Experiments — extend Phase 1 without re-running everything

These cells use the S1/S4/S5/S6 checkpoints × 5 architectures already saved in `MODELS_DIR` by Phase 1.

| Part | Training? | Approx. time on A100 |
|---|---|---|
| A0–A10 | no, inference only | ~2–3 h in total (SDA v2 is the longest) |
| B1 linear probe + foundation models | linear head only | ~1 h (mostly feature extraction) |
| B2 fine-tune source ablation | yes | ~2–4 h (2 architectures) |
| B3 label-efficiency | yes, small data | ~1 h (ResNet-50) |
| B4 seeds (optional) | yes | ~5–8 h |

Every cell **skips work that is already done** (it checks the result files first), so it can be re-run after a Colab disconnect.

In [ ]:
# ── A0. Config & imports ────────────────────────────────────────────────────
import subprocess, sys
for _p in ['imagehash', 'open_clip_torch']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)

import os, json, math, random, time, hashlib
from pathlib import Path
from copy import deepcopy
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import cv2
from concurrent.futures import ThreadPoolExecutor
from statsmodels.stats.multitest import multipletests

ADDON_DIR = Path(f'{BASE_DRIVE}/addon')
PRED_DIR, ATAB, AFIG, AMOD, AFEAT = [ADDON_DIR / s for s in
                                     ('preds', 'tables', 'figures', 'models', 'features')]
for _d in (PRED_DIR, ATAB, AFIG, AMOD, AFEAT):
    _d.mkdir(parents=True, exist_ok=True)

ARCH_LABEL = dict(zip(ARCHITECTURES, ARCH_LABELS))
EVAL_ARCHS = list(ARCHITECTURES)                 # inference-only → use all 5
# chosen in advance (not from Thai test results): ResNet-50 = the deployed model, ConvNeXt-B = largest modern CNN
KEY_ARCHS  = ['resnet50', 'convnext_base']
BN_ARCHS   = ['resnet50', 'efficientnet_b4', 'densenet121']
RINGSPOT   = 'RingSpot'

# union label space must match S5/S6 training (raw SLD 5 classes → union 70)
DATASET_PATHS['SLD'] = f'{DATA_DIR}/sld'
SLD_FILTER = f'{DATA_DIR}/sld_filtered'          # SLD 3 classes (created in cell 1.5)
UNION_CLASSES, _ = get_union_classes()
THAI_IDX = get_thai_indices_in_union(UNION_CLASSES, CLASS_NAMES)
N_UNION  = len(UNION_CLASSES)
assert N_UNION == 70 and -1 not in THAI_IDX, (N_UNION, THAI_IDX)
NON_THAI_IDX = [i for i in range(N_UNION) if i not in THAI_IDX]

# copy the Thai images (current split) from Drive to local disk — repeated reads from Drive are very slow
import shutil
_thai_split_dir = Path(THAI_TEST).parent
LOCAL_THAI = Path(f'/content/thai_local_{_thai_split_dir.name}')
if not LOCAL_THAI.exists():
    t0 = time.time(); shutil.copytree(_thai_split_dir, LOCAL_THAI)
    print(f'copied Thai split → {LOCAL_THAI} ({time.time() - t0:.0f}s)')
THAI_TEST_L, THAI_TRAIN_L = str(LOCAL_THAI / 'test'), str(LOCAL_THAI / 'train')
assert Path(THAI_TEST_L).exists() and Path(THAI_TRAIN_L).exists()
print(f'[OK] Union={N_UNION} | Thai idx={dict(zip(CLASS_NAMES, THAI_IDX))}')
print(f'   Output → {ADDON_DIR}')

In [ ]:
# ── A0.1 Shared helpers ─────────────────────────────────────────────────────
IMG_EXT = {'.jpg', '.jpeg', '.png'}

def list_images(root, classes, label_offset=0):
    out = []
    for li, c in enumerate(classes):
        d = Path(root) / c
        if not d.exists():
            print(f'  [WARNING] not found: {d}'); continue
        out += [(str(p), li + label_offset) for p in sorted(d.iterdir())
                if p.suffix.lower() in IMG_EXT]
    return out

def seed_transform(tfm, s):
    """Make albumentations draw the same random parameters every time for the same image (important for TTA/SDA)"""
    s = int(s) % (2**31 - 1)
    if hasattr(tfm, 'set_random_seed'):
        tfm.set_random_seed(s)
    random.seed(s); np.random.seed(s)

class ListDataset(Dataset):
    """(path,label) list → tensor. Has .samples / .alb_transform, so it works with the existing tta_predict"""
    def __init__(self, samples, transform=None, seed=None, return_index=False):
        self.samples = list(samples)
        self.alb_transform = transform if transform is not None else VAL_TRANSFORM
        self.seed, self.return_index = seed, return_index
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, l = self.samples[i]
        img = np.array(Image.open(p).convert('RGB'))
        if self.seed is not None:
            seed_transform(self.alb_transform, self.seed + i)
        x = self.alb_transform(image=img)['image']
        return (x, l, i) if self.return_index else (x, l)

def make_loader(samples, transform=None, bs=None, seed=None, shuffle=False,
                workers=None, return_index=False):
    return DataLoader(ListDataset(samples, transform, seed, return_index),
                      batch_size=bs or BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS if workers is None else workers,
                      pin_memory=True)

def ckpt_path(scn, arch):
    return f'{MODELS_DIR}/{scn}_{arch}.pt'

def n_out(scn):
    return {'S1': 38, 'S4': NUM_CLASSES}.get(scn, N_UNION)

def load_ckpt(scn, arch, path=None):
    m = build_model(arch, num_classes=n_out(scn), pretrained=False)
    m.load_state_dict(torch.load(path or ckpt_path(scn, arch), map_location=DEVICE))
    return m.eval()

def to_thai(z):
    """union logits (70) → 3 Thai classes (same as SelectiveHead); returned unchanged if already 3"""
    return z if z.shape[-1] == NUM_CLASSES else z[..., THAI_IDX]

@torch.no_grad()
def infer_logits(model, loader, amp=False):
    model.eval(); outs = []
    for batch in loader:
        x = batch[0].to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp and DEVICE.type == 'cuda'):
            outs.append(model(x).float().cpu())
    return torch.cat(outs).numpy()

def softmax_np(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

def probs_from_views(z_views, T=1.0):
    """z_views: (V,N,C) or (N,C) → mean softmax across views (same as tta_predict)"""
    z_views = np.asarray(z_views)
    if z_views.ndim == 2:
        return softmax_np(z_views, T)
    return softmax_np(z_views, T).mean(0)

def basic_metrics(p, y):
    from sklearn.metrics import f1_score
    pred = p.argmax(1)
    pcf1 = f1_score(y, pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    return {'acc': (pred == y).mean() * 100,
            'f1': f1_score(y, pred, average='macro', zero_division=0) * 100,
            **{f'F1_{c}': v * 100 for c, v in zip(CLASS_NAMES, pcf1)}}

def boot_ci(correct, n_boot=2000, seed=SEED):
    correct = np.asarray(correct, dtype=float)
    rng = np.random.default_rng(seed); n = len(correct)
    bs = correct[rng.integers(0, n, size=(n_boot, n))].mean(1) * 100
    return np.percentile(bs, [2.5, 97.5])

def paired_boot_ci(c1, c2, n_boot=2000, seed=SEED):
    """Paired CI of (acc2 − acc1) on the same set of images"""
    c1, c2 = np.asarray(c1, float), np.asarray(c2, float)
    rng = np.random.default_rng(seed); n = len(c1)
    idx = rng.integers(0, n, size=(n_boot, n))
    d = (c2[idx].mean(1) - c1[idx].mean(1)) * 100
    return np.percentile(d, [2.5, 97.5])

def mcnemar_p(c1, c2):
    from statsmodels.stats.contingency_tables import mcnemar
    c1, c2 = np.asarray(c1, bool), np.asarray(c2, bool)
    b, c = int((c1 & ~c2).sum()), int((~c1 & c2).sum())
    tab = [[int((c1 & c2).sum()), b], [c, int((~c1 & ~c2).sum())]]
    exact = (b + c) < 25
    r = mcnemar(tab, exact=exact, correction=not exact)
    return float(r.pvalue), b, c

def save_table(df, name):
    df.to_csv(ATAB / f'{name}.csv', index=False)
    print(f'saved {ATAB / name}.csv  ({len(df)} rows)')
    return df

TAU = 0.70   # confidence threshold used by the deployed system (A5, A6, C1)

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def feature_extractor(base, arch):
    m = deepcopy(base)
    if arch == 'resnet50':      m.fc = nn.Identity()
    elif arch == 'densenet121': m.classifier = nn.Identity()
    else:                       m.reset_classifier(0)
    return m.eval()

print('[OK] helpers ready')

## A1 — Duplicate and data-leakage audit (run before anything else)
Checks 4 things: (1) byte-identical files within the Thai data, (2) near-duplicates between Thai train and test,
(3) near-duplicates with **different labels** (label noise), (4) Thai images near-duplicate with SLD (if any, transfer results would be inflated)

Uses a 64-bit perceptual hash; Hamming ≤ 4 is treated as near-duplicate — **inspect the image pairs saved by this cell visually** before deciding to exclude them.

In [ ]:
# ── A1. Duplicate / leakage audit ───────────────────────────────────────────
import imagehash
from concurrent.futures import ThreadPoolExecutor

DUP_THR = 4          # pHash Hamming ≤ 4 (of 64 bits)
DHASH_THR = 8        # must also pass dHash ≤ 8 → fewer false positives among similar-looking sugarcane leaves
APPLY_EXCLUSIONS = True   # set False if visual inspection shows the pairs are not real duplicates, then re-run A1–A2

def _phash(path):
    try:
        h = imagehash.phash(Image.open(path).convert('RGB'), hash_size=8)
        return np.packbits(h.hash.flatten())
    except Exception:   # unreadable → random hash from path (will not match other images)
        print(f'  [WARNING] unreadable: {path}')
        return np.frombuffer(hashlib.md5(path.encode()).digest()[:8], np.uint8).copy()

def _dhash(path):
    try:
        return np.packbits(imagehash.dhash(Image.open(path).convert('RGB'), hash_size=8).hash.flatten())
    except Exception:
        return np.frombuffer(hashlib.md5((path + 'd').encode()).digest()[:8], np.uint8).copy()

def hamming(a, b):
    return int(np.unpackbits(np.bitwise_xor(a, b)).sum())

def confirm(pairs, DA, DB, thr=DHASH_THR):
    """Keep only pairs whose dHash is also close"""
    return [(i, j, d) for i, j, d in pairs if hamming(DA[i], DB[j]) <= thr]

def _md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def hash_all(paths, fn, workers=16):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(fn, paths))

def near_dups(HA, HB, thr=DUP_THR, same=False, chunk=512):
    """Return a list of (i, j, dist) for pairs with Hamming ≤ thr"""
    HA, HB = np.asarray(HA, np.uint8), np.asarray(HB, np.uint8)
    out = []
    for i0 in range(0, len(HA), chunk):
        x = np.bitwise_xor(HA[i0:i0 + chunk, None, :], HB[None, :, :])
        d = np.unpackbits(x, axis=-1).sum(-1)
        ii, jj = np.where(d <= thr)
        for i, j in zip(ii, jj):
            gi = i0 + i
            if same and gi >= j:
                continue
            out.append((gi, int(j), int(d[i, j])))
    return out

audit_file = ATAB / 'A1_dup_pairs.csv'
THAI_ALL_CLASSES = sorted(d.name for d in Path(THAI_TEST_L).iterdir() if d.is_dir())
thai_all = ([(p, c, 'train') for c in THAI_ALL_CLASSES for p, _ in list_images(THAI_TRAIN_L, [c])] +
            [(p, c, 'test')  for c in THAI_ALL_CLASSES for p, _ in list_images(THAI_TEST_L, [c])])
df_thai_all = pd.DataFrame(thai_all, columns=['path', 'cls', 'split'])
print(f'Thai (80 split) all classes: {len(df_thai_all)} images | {THAI_ALL_CLASSES}')

if audit_file.exists():
    df_pairs = pd.read_csv(audit_file)
    print('loaded existing audit')
else:
    t0 = time.time()
    df_thai_all['md5'] = hash_all(df_thai_all.path, _md5)
    H_thai = np.stack(hash_all(df_thai_all.path, _phash))
    D_thai = np.stack(hash_all(df_thai_all.path, _dhash))
    sld_paths = [p for p, _ in list_images(DATASET_PATHS['SLD'],
                 sorted(d.name for d in Path(DATASET_PATHS['SLD']).iterdir() if d.is_dir()))]
    H_sld = np.stack(hash_all(sld_paths, _phash))
    D_sld = np.stack(hash_all(sld_paths, _dhash))
    rows = []
    for i, j, d in confirm(near_dups(H_thai, H_thai, same=True), D_thai, D_thai):
        a, b = df_thai_all.iloc[i], df_thai_all.iloc[j]
        rows.append({'kind': 'thai-thai', 'path_a': a.path, 'cls_a': a.cls, 'split_a': a.split,
                     'path_b': b.path, 'cls_b': b.cls, 'split_b': b.split, 'hamming': d,
                     'exact_md5': a.md5 == b.md5})
    for i, j, d in confirm(near_dups(H_thai, H_sld), D_thai, D_sld):
        a = df_thai_all.iloc[i]
        rows.append({'kind': 'thai-sld', 'path_a': a.path, 'cls_a': a.cls, 'split_a': a.split,
                     'path_b': sld_paths[j], 'cls_b': Path(sld_paths[j]).parent.name,
                     'split_b': 'sld', 'hamming': d, 'exact_md5': False})
    df_pairs = pd.DataFrame(rows, columns=['kind', 'path_a', 'cls_a', 'split_a', 'path_b',
                                           'cls_b', 'split_b', 'hamming', 'exact_md5'])
    df_pairs.to_csv(audit_file, index=False)
    print(f'⏱ {time.time() - t0:.0f}s')

tt = df_pairs[df_pairs.kind == 'thai-thai']
cross_split = tt[tt.split_a != tt.split_b]
cross_label = tt[tt.cls_a != tt.cls_b]
thai_sld = df_pairs[df_pairs.kind == 'thai-sld']
summary_A1 = pd.DataFrame([
    {'check': 'Thai near-dup pairs (all)',                 'n': len(tt)},
    {'check': '  of which exact byte-identical',           'n': int(tt.exact_md5.sum())},
    {'check': 'Thai train↔test near-dup pairs',             'n': len(cross_split)},
    {'check': 'Thai near-dup pairs with DIFFERENT labels', 'n': len(cross_label)},
    {'check': 'Thai↔SLD near-dup pairs',                    'n': len(thai_sld)},
    {'check': '  Thai test images affected (Thai↔SLD)',     'n': thai_sld[thai_sld.split_a == 'test'].path_a.nunique()},
])
print(summary_A1.to_string(index=False))
save_table(summary_A1, 'A1_summary')

# save example pairs for visual inspection
def show_pairs(df, title, fname, k=12):
    if len(df) == 0:
        return
    df = df.sort_values('hamming').head(k)
    fig, ax = plt.subplots(len(df), 2, figsize=(5, 2.4 * len(df)))
    ax = np.atleast_2d(ax)
    for r, (_, row) in enumerate(df.iterrows()):
        for c, key in enumerate(['a', 'b']):
            ax[r, c].imshow(Image.open(row[f'path_{key}']).convert('RGB'))
            ax[r, c].set_title(f"{row[f'cls_{key}']}/{row[f'split_{key}']} d={row.hamming}", fontsize=7)
            ax[r, c].axis('off')
    fig.suptitle(title, fontsize=9); plt.tight_layout()
    plt.savefig(AFIG / fname, dpi=120); plt.close(fig)
    print(f'figure {AFIG / fname}')

show_pairs(cross_split, 'Thai train↔test near-duplicates', 'A1_train_test_pairs.png')
show_pairs(cross_label, 'Near-duplicates with different labels', 'A1_cross_label_pairs.png')
show_pairs(thai_sld, 'Thai↔SLD near-duplicates', 'A1_thai_sld_pairs.png')

# sets to exclude (adjustable after inspecting the pairs)
LEAK_TRAIN = set(cross_split.apply(lambda r: r.path_a if r.split_a == 'train' else r.path_b, axis=1)) if len(cross_split) else set()
SLD_OVERLAP_THAI = set(thai_sld.path_a)
if not APPLY_EXCLUSIONS:
    LEAK_TRAIN, SLD_OVERLAP_THAI, tt = set(), set(), tt.iloc[0:0]
if len(tt) > 0.05 * len(df_thai_all):
    print('[WARNING] near-duplicates exceed 5% — inspect the pairs in A1_*.png first; if they are false positives, lower DUP_THR or set APPLY_EXCLUSIONS=False')
print(f'→ Thai-train images near-duplicate with test, removed from label-efficiency/calibration: {len(LEAK_TRAIN)} images')
print(f'→ Thai images near-duplicate with SLD (excluded from evaluation): {len(SLD_OVERLAP_THAI)} images')

## A2 — Evaluation sets
* **thai_test** (600) — the original test set, used for the main results
* **thai_full** (~3,000) — S4–S6 never use Thai labels, so they can be evaluated on all Thai images (train+test of the 80 split) → confidence intervals about 2× narrower; **replaces the earlier "split sensitivity"**, whose test sets overlap across splits
* **thai_calib** — 10% of Thai train, used to fit the temperature (uses labels, so not a main result)
* **ringspot** — Thai RingSpot images, used as an "unknown disease" to test rejection (OOD)

In [ ]:
# ── A2. Evaluation manifests ────────────────────────────────────────────────
def _drop(samples, bad):
    return [s for s in samples if s[0] not in bad]

thai_test  = _drop(list_images(THAI_TEST_L,  CLASS_NAMES), SLD_OVERLAP_THAI)
thai_train = _drop(list_images(THAI_TRAIN_L, CLASS_NAMES), SLD_OVERLAP_THAI | LEAK_TRAIN)

# thai_full = test + train with internal near-duplicates removed (one copy kept)
_dup_drop = set()
for _, r in tt.iterrows():
    if r.cls_a in CLASS_NAMES and r.cls_b in CLASS_NAMES:
        _dup_drop.add(r.path_b)
thai_full = _drop(thai_test + _drop(list_images(THAI_TRAIN_L, CLASS_NAMES), SLD_OVERLAP_THAI), _dup_drop)

ringspot = ([(p, -1) for p, _ in list_images(THAI_TEST_L, [RINGSPOT])] +
            [(p, -1) for p, _ in list_images(THAI_TRAIN_L, [RINGSPOT])])

_rng = np.random.default_rng(SEED)
_y = np.array([l for _, l in thai_train])
_calib_idx = np.concatenate([_rng.choice(np.where(_y == c)[0], size=max(1, int(0.1 * (_y == c).sum())), replace=False)
                             for c in range(NUM_CLASSES)])
thai_calib = [thai_train[i] for i in sorted(_calib_idx)]

EVAL_SETS = {'thai_test': thai_test, 'thai_full': thai_full, 'ringspot': ringspot}
for name, s in {**EVAL_SETS, 'thai_train': thai_train, 'thai_calib': thai_calib}.items():
    pd.DataFrame(s, columns=['path', 'label']).to_csv(PRED_DIR / f'manifest_{name}.csv', index=False)
    print(f'  {name:<11} {len(s):>5} images')

## A3 — Export logits of every checkpoint (run once; all analyses use these files)
* clean (no TTA) for S4/S5/S6 × 5 architectures
* TTA with 8 views stored per view, for S5 and S6 (**seeded**, because the original TTA used random JPEG/brightness, so repeated runs differed)
* all 70 class logits are kept (used for OOD)

In [ ]:
# ── A3. Export logits ───────────────────────────────────────────────────────
TTA_SEED = 2024

def pred_file(scn, arch, tset, kind='clean'):
    return PRED_DIR / f'{scn}_{arch}_{tset}_{kind}.npy'

def export_logits(scn, arch, model=None, sets=EVAL_SETS, tta=None, force=False):
    tta = (scn != 'S4') if tta is None else tta
    todo = [(t, k) for t in sets for k in (['clean', 'tta'] if tta else ['clean'])
            if force or not pred_file(scn, arch, t, k).exists()]
    if not todo:
        return
    model = model if model is not None else load_ckpt(scn, arch).to(DEVICE)
    for tset, kind in todo:
        samples = sets[tset]
        if kind == 'clean':
            z = infer_logits(model, make_loader(samples))
        else:
            z = np.stack([infer_logits(model, make_loader(samples, transform=tfm, seed=TTA_SEED + 1000 * v))
                          for v, tfm in enumerate(TTA_TRANSFORMS)])
        np.save(pred_file(scn, arch, tset, kind), z.astype(np.float32))
        print(f'   saved {scn} {arch} {tset} {kind} {z.shape}')
    del model; torch.cuda.empty_cache()

for arch in EVAL_ARCHS:
    for scn in ['S4', 'S5', 'S6']:
        if not Path(ckpt_path(scn, arch)).exists():
            print(f'[WARNING] missing {scn}_{arch}.pt'); continue
        print(f'[{scn} | {arch}]')
        export_logits(scn, arch)
print('[OK] A3 done')

def get_logits(scn, arch, tset, kind='clean'):
    return np.load(pred_file(scn, arch, tset, kind))

def labels_of(tset):
    return np.array([l for _, l in EVAL_SETS[tset]])

CONFIGS = {   # config name → (scenario, kind)
    'S4':        ('S4', 'clean'),
    'S5':        ('S5', 'clean'),
    'S5+TTA':    ('S5', 'tta'),
    'S6-noTTA':  ('S6', 'clean'),
    'S6+TTA':    ('S6', 'tta'),
}

def get_probs(cfg, arch, tset, T=1.0):
    scn, kind = CONFIGS[cfg]
    return probs_from_views(to_thai(get_logits(scn, arch, tset, kind)), T)

## A4 — Main table with 95% CI + McNemar (Holm) + per-view TTA analysis
* Compared with the earlier numbers in `results_all.csv` to confirm they match (an earlier cell once gave S5 ≈ 33%, which contradicted the table)
* Separates the effects of (a) multi-source (S4→S5), (b) TropAug (S5→S6-noTTA), (c) TTA (S6-noTTA→S6+TTA and S5→S5+TTA)

In [ ]:
# ── A4. Main table, CIs, paired tests ───────────────────────────────────────
from statsmodels.stats.multitest import multipletests

rows = []
for tset in ['thai_test', 'thai_full']:
    y = labels_of(tset)
    for arch in EVAL_ARCHS:
        for cfg in CONFIGS:
            try:
                p = get_probs(cfg, arch, tset)
            except FileNotFoundError:
                continue
            m = basic_metrics(p, y)
            lo, hi = boot_ci(p.argmax(1) == y)
            rows.append({'set': tset, 'arch': ARCH_LABEL[arch], 'config': cfg, **m,
                         'acc_lo': lo, 'acc_hi': hi, 'n': len(y)})
df_main = save_table(pd.DataFrame(rows).round(2), 'A4_main_table')
print(df_main[df_main.set == 'thai_test'].pivot(index='arch', columns='config', values='acc').round(2))

# compare with the earlier numbers
try:
    old = pd.read_csv(f'{TABLES_DIR}/results_all.csv')
    m = {'S4': 'S4', 'S5': 'S5', 'S6+TTA': 'S6'}
    chk = []
    for cfg, scn in m.items():
        for arch in EVAL_ARCHS:
            o = old[(old.Scenario == scn) & (old.Arch_id == arch)]
            n = df_main[(df_main.set == 'thai_test') & (df_main.config == cfg) & (df_main.arch == ARCH_LABEL[arch])]
            if len(o) and len(n):
                chk.append({'config': cfg, 'arch': arch, 'old_acc': o.acc.iloc[0], 'new_acc': n.acc.iloc[0],
                            'diff': round(n.acc.iloc[0] - o.acc.iloc[0], 2)})
    df_chk = save_table(pd.DataFrame(chk), 'A4_check_vs_old')
    print(df_chk.to_string(index=False))
    print(f'Note: thai_test now has {len(thai_test)} images (originally 600) — fewer means A1 removed images duplicated in SLD')
    print('Note: S6+TTA may differ slightly because the original TTA did not fix the seed — use the new (seeded) numbers in the paper')
except FileNotFoundError:
    print('[WARNING] results_all.csv not found')

PAIRS = [('S4', 'S5', 'multi-source'), ('S5', 'S6-noTTA', 'TropAug'),
         ('S6-noTTA', 'S6+TTA', 'TTA on S6'), ('S5', 'S5+TTA', 'TTA on S5'),
         ('S5', 'S6+TTA', 'full S6 vs S5')]
rows = []
for tset in ['thai_test', 'thai_full']:
    y = labels_of(tset)
    for arch in EVAL_ARCHS:
        for a, b, what in PAIRS:
            try:
                ca = get_probs(a, arch, tset).argmax(1) == y
                cb = get_probs(b, arch, tset).argmax(1) == y
            except FileNotFoundError:
                continue
            pv, nb, nc = mcnemar_p(ca, cb)
            lo, hi = paired_boot_ci(ca, cb)
            rows.append({'set': tset, 'arch': ARCH_LABEL[arch], 'comparison': f'{a} → {b}', 'effect': what,
                         'delta_acc': (cb.mean() - ca.mean()) * 100, 'ci_lo': lo, 'ci_hi': hi,
                         'b(a✓b✗)': nb, 'c(a✗b✓)': nc, 'p_mcnemar': pv})
df_pairs_t = pd.DataFrame(rows)
for tset in df_pairs_t.set.unique():
    msk = df_pairs_t.set == tset
    df_pairs_t.loc[msk, 'p_holm'] = multipletests(df_pairs_t.loc[msk, 'p_mcnemar'], method='holm')[1]
df_pairs_t['sig_holm'] = df_pairs_t.p_holm < 0.05
save_table(df_pairs_t.round(4), 'A4_paired_tests')
print(df_pairs_t[df_pairs_t.set == 'thai_full']
      .pivot(index='arch', columns='effect', values='delta_acc').round(2))
print('\nDirection of the effect across 5 architectures (a partial substitute for replicates until seeds are available):')
print(df_pairs_t.groupby(['set', 'effect']).delta_acc.agg(['mean', 'std', lambda s: (s > 0).sum()]).round(2))

# TTA per view
VIEW_NAMES = ['orig', 'hflip', 'vflip', 'h+vflip', 'jpeg', 'jpeg+hflip', 'bright', 'shadow']
rows = []
for scn in ['S5', 'S6']:
    for arch in EVAL_ARCHS:
        try:
            zv = to_thai(get_logits(scn, arch, 'thai_full', 'tta'))
        except FileNotFoundError:
            continue
        y = labels_of('thai_full')
        r = {'scenario': scn, 'arch': ARCH_LABEL[arch],
             'all8': (probs_from_views(zv).argmax(1) == y).mean() * 100}
        for v, nm in enumerate(VIEW_NAMES):
            r[f'only_{nm}'] = (softmax_np(zv[v]).argmax(1) == y).mean() * 100
            keep = [i for i in range(len(zv)) if i != v]
            r[f'drop_{nm}'] = (probs_from_views(zv[keep]).argmax(1) == y).mean() * 100
        r['flips_only(4)'] = (probs_from_views(zv[:4]).argmax(1) == y).mean() * 100
        rows.append(r)
df_tta = save_table(pd.DataFrame(rows).round(2), 'A4_tta_per_view')
print(df_tta[['scenario', 'arch', 'all8', 'flips_only(4)'] + [f'only_{v}' for v in VIEW_NAMES]].to_string(index=False))

## A5 — Calibration & selective prediction
* ECE (15 bins), Brier, NLL before/after temperature scaling (fitted on **thai_calib**, separate from test)
* Risk–coverage + AURC, and results at the 0.70 threshold used by the deployed system → evidence for choosing 0.70

In [ ]:
# ── A5. Calibration & selective prediction ──────────────────────────────────
def ece_score(p, y, n_bins=15):
    conf, pred = p.max(1), p.argmax(1)
    acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e * 100

def brier(p, y):
    oh = np.eye(p.shape[1])[y]
    return ((p - oh) ** 2).sum(1).mean()

def nll(p, y):
    return -np.log(p[np.arange(len(y)), y] + 1e-12).mean()

def fit_temperature(z_views, y, iters=200):
    """z_views: (V,N,C) or (N,C); fit T to minimise the NLL of the mean softmax"""
    z = torch.tensor(np.asarray(z_views, np.float32))
    if z.ndim == 2:
        z = z[None]
    yt = torch.tensor(y)
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=iters)
    def closure():
        opt.zero_grad()
        p = torch.softmax(z / logT.exp(), -1).mean(0)
        loss = F.nll_loss(torch.log(p + 1e-12), yt)
        loss.backward(); return loss
    opt.step(closure)
    T = logT.detach().exp().item()
    if not 0.05 <= T <= 20:
        print(f'  [WARNING] T={T:.3g} is abnormal (model near chance on the calib set?) → clipped to [0.05, 20]')
    return float(np.clip(T, 0.05, 20))

def risk_coverage(p, y):
    conf = p.max(1); correct = (p.argmax(1) == y)
    order = np.argsort(-conf)
    cum_err = np.cumsum(~correct[order]) / np.arange(1, len(y) + 1)
    cov = np.arange(1, len(y) + 1) / len(y)
    return cov, cum_err, cum_err.mean()   # AURC

# indices of thai_calib within thai_full (uses exported logits, no model re-run)
_full_paths = {p: i for i, (p, _) in enumerate(thai_full)}
_calib_in_full = [_full_paths[p] for p, _ in thai_calib if p in _full_paths]
y_cal = labels_of('thai_full')[_calib_in_full]
y_te = labels_of('thai_test')
rows, curves = [], {}
for arch in EVAL_ARCHS:
    for cfg in ['S5', 'S5+TTA', 'S6-noTTA', 'S6+TTA']:
        scn, kind = CONFIGS[cfg]
        try:
            z_full = to_thai(get_logits(scn, arch, 'thai_full', kind))
            z_te = to_thai(get_logits(scn, arch, 'thai_test', kind))
        except FileNotFoundError:
            continue
        z_cal = z_full[..., _calib_in_full, :] if z_full.ndim == 3 else z_full[_calib_in_full]
        T = fit_temperature(z_cal, y_cal)
        for setting, t in [('raw', 1.0), ('TS(Thai-10%)', T)]:
            p = probs_from_views(z_te, t)
            cov, risk, aurc = risk_coverage(p, y_te)
            keep = p.max(1) >= TAU
            rows.append({'arch': ARCH_LABEL[arch], 'config': cfg, 'setting': setting, 'T': round(t, 3),
                         'acc': (p.argmax(1) == y_te).mean() * 100, 'ECE': ece_score(p, y_te),
                         'Brier': brier(p, y_te), 'NLL': nll(p, y_te), 'AURC': aurc,
                         'mean_conf': p.max(1).mean() * 100,
                         f'coverage@{TAU}': keep.mean() * 100,
                         f'sel_acc@{TAU}': (p.argmax(1)[keep] == y_te[keep]).mean() * 100 if keep.any() else np.nan,
                         'risk@cov80': risk[int(0.8 * len(risk)) - 1] * 100})
            curves[(arch, cfg, setting)] = (cov, risk, p)
df_cal = save_table(pd.DataFrame(rows).round(3), 'A5_calibration')
print(df_cal[df_cal.arch.isin([ARCH_LABEL[a] for a in KEY_ARCHS])].to_string(index=False))

# Figure: reliability + risk–coverage for the deployed config (ResNet-50 S6+TTA) and ConvNeXt S5
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for k, (arch, cfg) in enumerate([(KEY_ARCHS[0], 'S6+TTA'), (KEY_ARCHS[-1], 'S5')]):
    for setting, ls in [('raw', '-'), ('TS(Thai-10%)', '--')]:
        if (arch, cfg, setting) not in curves:
            continue
        cov, risk, p = curves[(arch, cfg, setting)]
        conf, acc = p.max(1), (p.argmax(1) == y_te)
        bins = np.linspace(0, 1, 11); mids, accs = [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.sum() >= 5:
                mids.append(conf[m].mean()); accs.append(acc[m].mean())
        axes[k].plot(mids, accs, ls, marker='o', ms=3, label=setting)
        axes[2].plot(cov, risk * 100, ls, label=f'{ARCH_LABEL[arch]} {cfg} {setting}')
    axes[k].plot([0, 1], [0, 1], 'k:', lw=0.8)
    axes[k].set_title(f'Reliability — {ARCH_LABEL[arch]} {cfg}', fontsize=9)
    axes[k].set_xlabel('confidence'); axes[k].set_ylabel('accuracy'); axes[k].legend(fontsize=7)
axes[2].set_xlabel('coverage'); axes[2].set_ylabel('selective risk (%)')
axes[2].set_title('Risk–coverage (Thai test)', fontsize=9); axes[2].legend(fontsize=6)
plt.tight_layout(); plt.savefig(AFIG / 'A5_calibration.png', dpi=300); plt.show()

## A6 — Rejecting an unknown disease (Thai RingSpot = real near-OOD)
The system tells users that Ring Spot is not supported, but the model always predicts one of the 3 classes → measure how well it can be separated.

Scores compared: MSP, entropy, energy (3 classes), max-logit, and 2 scores that use the **70-class head** the model already has but does not use:
energy-70 and *sugarcane margin* = max(sugarcane-class logit) − max(logit of the other 67 classes)

In [ ]:
# ── A6. Unknown-disease (RingSpot) rejection ────────────────────────────────
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.special import logsumexp

def ood_scores(z70_views):
    z = np.asarray(z70_views)
    z = z.mean(0) if z.ndim == 3 else z           # TTA: average logits
    z3 = z[:, THAI_IDX]
    p3 = softmax_np(z3)
    return {'MSP': p3.max(1),
            'neg_entropy': (p3 * np.log(p3 + 1e-12)).sum(1),
            'energy3': logsumexp(z3, axis=1),
            'max_logit3': z3.max(1),
            'energy70': logsumexp(z, axis=1),
            'sugarcane_margin': z3.max(1) - z[:, NON_THAI_IDX].max(1)}

def fpr_at_95tpr(s_id, s_ood):
    thr = np.percentile(s_id, 5)                  # keep 95% of ID
    return (s_ood >= thr).mean() * 100

rows, dist_rows = [], []
for arch in EVAL_ARCHS:
    for cfg in ['S5', 'S6-noTTA', 'S6+TTA']:
        scn, kind = CONFIGS[cfg]
        try:
            z_id = get_logits(scn, arch, 'thai_test', kind)
            z_od = get_logits(scn, arch, 'ringspot', kind)
        except FileNotFoundError:
            continue
        s_id, s_od = ood_scores(z_id), ood_scores(z_od)
        for name in s_id:
            sc = np.concatenate([s_id[name], s_od[name]])
            lab = np.r_[np.ones(len(s_id[name])), np.zeros(len(s_od[name]))]
            rows.append({'arch': ARCH_LABEL[arch], 'config': cfg, 'score': name,
                         'AUROC': roc_auc_score(lab, sc) * 100,
                         'AUPR_unknown': average_precision_score(1 - lab, -sc) * 100,
                         'FPR95': fpr_at_95tpr(s_id[name], s_od[name])})
        p_id, p_od = probs_from_views(to_thai(z_id)), probs_from_views(to_thai(z_od))
        dist_rows.append({'arch': ARCH_LABEL[arch], 'config': cfg,
                          f'known_rejected@{TAU}': (p_id.max(1) < TAU).mean() * 100,
                          f'ringspot_rejected@{TAU}': (p_od.max(1) < TAU).mean() * 100,
                          'ringspot_mean_conf': p_od.max(1).mean() * 100,
                          **{f'ringspot→{c}': (p_od.argmax(1) == i).mean() * 100 for i, c in enumerate(CLASS_NAMES)}})
df_ood = save_table(pd.DataFrame(rows).round(2), 'A6_ood_ringspot')
df_ood_dist = save_table(pd.DataFrame(dist_rows).round(2), 'A6_ringspot_predictions')
print(df_ood.pivot_table(index=['arch', 'config'], columns='score', values='AUROC').round(1))
print(df_ood_dist.to_string(index=False))

## A7 — SDA v2: multiple severities, 3 repetitions, S5 vs S6
Fixes 3 problems of the original SDA:
1. **Circularity** — the original measured only S6, which was trained with F1–F7 → S5 is now measured as well; the difference = robustness benefit of TropAug
2. **Single severity** — 5 ImageNet-C-style severities, reporting mCE / relative mCE (reference = S5 ResNet-50)
3. **Single random draw** — 3 repetitions with fixed seeds; every model sees **the same** degraded images (paired) → CIs

Severities 2–3 are close to the values used in the original SDA.

In [ ]:
# ── A7. SDA v2 ──────────────────────────────────────────────────────────────
SDA_ARCHS = list(EVAL_ARCHS)      # reduce to KEY_ARCHS for a faster run
SDA_REPS = 3
SEV = [1, 2, 3, 4, 5]
FACTORS = ['F1_Sunlight', 'F2_Shadow', 'F3_Blur', 'F4_Occlusion', 'F5_Rain', 'F6_Fog', 'F7_JPEG']

def _dropout(n, h):
    kw = dict(num_holes_range=(n, n), hole_height_range=(h // 2, h), hole_width_range=(h // 2, h), p=1.0)
    try:
        return A.CoarseDropout(fill=0, **kw)
    except TypeError:
        return A.CoarseDropout(fill_value=0, **kw)

def make_factor(f, s):
    i = s - 1
    if f == 'F1_Sunlight':
        b = [0.1, 0.2, 0.3, 0.4, 0.5][i]
        return [A.RandomBrightnessContrast(brightness_limit=(b, b), contrast_limit=(0, 0), p=1.0)]
    if f == 'F2_Shadow':
        b = [0.1, 0.2, 0.3, 0.4, 0.5][i]
        return [A.RandomBrightnessContrast(brightness_limit=(-b, -b), contrast_limit=(0, 0), p=1.0)]
    if f == 'F3_Blur':
        k = [3, 7, 11, 15, 21][i]
        return [A.MotionBlur(blur_limit=(k, k), p=1.0)]
    if f == 'F4_Occlusion':
        n, h = [(1, 40), (2, 50), (3, 60), (4, 70), (5, 80)][i]
        return [_dropout(n, h)]
    if f == 'F5_Rain':
        rt, w, bv = [('drizzle', 1, 3), ('default', 1, 3), ('heavy', 1, 3),
                     ('torrential', 1, 3), ('torrential', 2, 5)][i]
        return [A.RandomRain(rain_type=rt, drop_width=w, blur_value=bv, p=1.0)]
    if f == 'F6_Fog':
        c = [0.1, 0.2, 0.3, 0.45, 0.6][i]
        return [A.RandomFog(fog_coef_range=(c, c), alpha_coef=0.1, p=1.0)]
    if f == 'F7_JPEG':
        q = [85, 70, 50, 30, 15][i]
        return [A.ImageCompression(quality_range=(q, q), compression_type='jpeg', p=1.0)]
    raise ValueError(f)

def degraded_tfm(extra):
    return A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), *extra,
                      A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])

# example images for each severity (for the appendix)
_ex = np.array(Image.open(thai_test[0][0]).convert('RGB'))
fig, ax = plt.subplots(len(FACTORS), 5, figsize=(10, 2 * len(FACTORS)))
for r, f in enumerate(FACTORS):
    for c, s in enumerate(SEV):
        t = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), *make_factor(f, s)]); seed_transform(t, 0)
        ax[r, c].imshow(t(image=_ex)['image']); ax[r, c].axis('off')
        ax[r, c].set_title(f'{f} s{s}', fontsize=6)
plt.tight_layout(); plt.savefig(AFIG / 'A7_severity_examples.png', dpi=200); plt.close(fig)

sda_npz = ATAB / 'A7_sda_v2_correct.npz'
if sda_npz.exists():
    _d = np.load(sda_npz, allow_pickle=True)
    CORR, CONDS, MODELS_SDA = _d['corr'], list(_d['conds']), list(_d['models'])
    print('loaded SDA v2 results')
else:
    MODELS_SDA = [(scn, arch) for arch in SDA_ARCHS for scn in ('S5', 'S6')
                  if Path(ckpt_path(scn, arch)).exists()]
    nets = {k: SelectiveHead(load_ckpt(*k), THAI_IDX).to(DEVICE).eval() for k in MODELS_SDA}
    CONDS = [('Clean', 0, 0)] + [(f, s, r) for f in FACTORS for s in SEV for r in range(SDA_REPS)]
    y = labels_of('thai_test')
    CORR = np.zeros((len(MODELS_SDA), len(CONDS), len(y)), bool)
    t0 = time.time()
    for ci, (f, s, r) in enumerate(CONDS):
        tfm = VAL_TRANSFORM if f == 'Clean' else degraded_tfm(make_factor(f, s))
        seed = 100_000 * (r + 1) + 1000 * s + 10 * (FACTORS.index(f) if f in FACTORS else 0)
        loader = make_loader(thai_test, transform=tfm, seed=seed, return_index=True)
        with torch.no_grad():
            for x, lab, idx in loader:
                x = x.to(DEVICE, non_blocking=True)
                for mi, k in enumerate(MODELS_SDA):
                    CORR[mi, ci, idx.numpy()] = (nets[k](x).argmax(1).cpu() == lab).numpy()
        if ci % 15 == 0:
            print(f'  {ci + 1}/{len(CONDS)} {f} s{s} r{r} | {time.time() - t0:.0f}s')
    np.savez_compressed(sda_npz, corr=CORR, conds=np.array(CONDS, dtype=object),
                        models=np.array(MODELS_SDA, dtype=object))
    del nets; torch.cuda.empty_cache()

MODELS_SDA = [tuple(m) for m in MODELS_SDA]
CONDS = [tuple(c) for c in CONDS]
acc_of = lambda mi, mask: CORR[mi][mask].mean() * 100

rows = []
for mi, (scn, arch) in enumerate(MODELS_SDA):
    clean = CORR[mi, 0].mean() * 100
    for f in FACTORS:
        for s in SEV:
            msk = [i for i, c in enumerate(CONDS) if c[0] == f and c[1] == s]
            reps = [CORR[mi, i].mean() * 100 for i in msk]
            rows.append({'scenario': scn, 'arch': ARCH_LABEL[arch], 'factor': f, 'severity': s,
                         'acc': np.mean(reps), 'acc_sd_reps': np.std(reps), 'clean': clean,
                         'delta': clean - np.mean(reps)})
df_sda2 = save_table(pd.DataFrame(rows).round(3), 'A7_sda_v2_long')

# mCE / relative mCE (Hendrycks & Dietterich 2019) — ref = S5 ResNet-50
ref = df_sda2[(df_sda2.scenario == 'S5') & (df_sda2.arch == 'ResNet-50')]
rows = []
for (scn, arch), g in df_sda2.groupby(['scenario', 'arch']):
    r = {'scenario': scn, 'arch': arch, 'clean_acc': g.clean.iloc[0]}
    ces, rces = [], []
    for f in FACTORS:
        gf, rf = g[g.factor == f], ref[ref.factor == f]
        err, err_ref = (100 - gf.acc).sum(), (100 - rf.acc).sum()
        rel, rel_ref = (gf.delta).sum(), (rf.delta).sum()
        r[f'CE_{f}'] = err / err_ref * 100
        r[f'meanΔ_{f}'] = gf.delta.mean()
        ces.append(err / err_ref * 100)
        rces.append(rel / rel_ref * 100 if abs(rel_ref) > 1e-6 else np.nan)
    r['mCE'] = np.mean(ces); r['rel_mCE'] = np.nanmean(rces)
    rows.append(r)
df_mce = save_table(pd.DataFrame(rows).round(2), 'A7_mCE')
print(df_mce[['scenario', 'arch', 'clean_acc', 'mCE', 'rel_mCE'] + [f'meanΔ_{f}' for f in FACTORS]].to_string(index=False))

# Robustness benefit of TropAug: (ΔS5 − ΔS6) per factor with a paired bootstrap CI over images
rows = []
y = labels_of('thai_test'); n = len(y); rng = np.random.default_rng(SEED)
B = rng.integers(0, n, size=(2000, n))
for arch in SDA_ARCHS:
    if ('S5', arch) not in MODELS_SDA or ('S6', arch) not in MODELS_SDA:
        continue
    i5, i6 = MODELS_SDA.index(('S5', arch)), MODELS_SDA.index(('S6', arch))
    for f in FACTORS:
        msk = [i for i, c in enumerate(CONDS) if c[0] == f]
        # per-image: clean correctness − mean correctness under the factor (averaged over severities/reps)
        d5 = CORR[i5, 0].astype(float) - CORR[i5, msk].mean(0)
        d6 = CORR[i6, 0].astype(float) - CORR[i6, msk].mean(0)
        gain = (d5 - d6)
        bs = gain[B].mean(1) * 100
        rows.append({'arch': ARCH_LABEL[arch], 'factor': f, 'drop_S5': d5.mean() * 100, 'drop_S6': d6.mean() * 100,
                     'robustness_gain': gain.mean() * 100,
                     'ci_lo': np.percentile(bs, 2.5), 'ci_hi': np.percentile(bs, 97.5)})
df_gain = save_table(pd.DataFrame(rows).round(2), 'A7_tropaug_robustness_gain')
print(df_gain.pivot(index='factor', columns='arch', values='robustness_gain'))

# Figure: accuracy vs severity (S5 solid, S6 dashed) for KEY_ARCHS
fig, axes = plt.subplots(2, 4, figsize=(13, 6)); axes = axes.ravel()
for k, f in enumerate(FACTORS):
    for arch in KEY_ARCHS:
        for scn, ls in [('S5', '-'), ('S6', '--')]:
            g = df_sda2[(df_sda2.factor == f) & (df_sda2.arch == ARCH_LABEL[arch]) & (df_sda2.scenario == scn)]
            if len(g):
                axes[k].plot([0] + SEV, [g.clean.iloc[0]] + list(g.acc), ls, marker='o', ms=3,
                             label=f'{ARCH_LABEL[arch]} {scn}')
    axes[k].set_title(f, fontsize=9); axes[k].set_xlabel('severity'); axes[k].set_ylim(30, 100)
axes[0].set_ylabel('accuracy (%)'); axes[0].legend(fontsize=6)
g = df_gain[df_gain.arch.isin([ARCH_LABEL[a] for a in KEY_ARCHS])]
for j, arch in enumerate(g.arch.unique()):
    gg = g[g.arch == arch]
    axes[7].errorbar(np.arange(len(gg)) + 0.15 * j, gg.robustness_gain,
                     yerr=[gg.robustness_gain - gg.ci_lo, gg.ci_hi - gg.robustness_gain], fmt='o', ms=3, label=arch)
axes[7].axhline(0, color='k', lw=0.6); axes[7].set_xticks(range(len(FACTORS)))
axes[7].set_xticklabels([f.split('_')[0] for f in FACTORS], fontsize=7)
axes[7].set_title('TropAug robustness gain (ΔS5 − ΔS6)', fontsize=9); axes[7].legend(fontsize=6)
plt.tight_layout(); plt.savefig(AFIG / 'A7_sda_v2.png', dpi=300); plt.show()

## A8 — Automatic image-quality metrics → errors on real images (first step of N1)
No manual labels needed yet: compute proxies of the capture conditions for each Thai image and test whether they relate to misclassification.

| Metric | Proxy for |
|---|---|
| `overexposed` (% pixels V>0.98), `specular` (very bright + low saturation) | strong sun, reflections, water drops (F1, F5) |
| `underexposed` (% V<0.05), `mean_V` | shade (F2) |
| `lap_var` (log) | blur (F3) |
| `green_frac` (ExG) | leaf fraction in the image / background |
| `border_edge`, `edge_density` | cluttered background, several leaves (F4) |
| `jpeg_blockiness` | compression (F7) |

Exploratory analysis — the template created by this cell can then be used to label the actual conditions by hand (N1) for confirmation.

In [ ]:
# ── A8. Image-quality proxies, error analysis, domain-shift stats ───────────
def jpeg_blockiness(gray):
    g = gray.astype(np.float32)
    dh = np.abs(np.diff(g, axis=1)); dv = np.abs(np.diff(g, axis=0))
    bh = dh[:, 7::8].mean() / (dh.mean() + 1e-6)
    bv = dv[7::8, :].mean() / (dv.mean() + 1e-6)
    return float((bh + bv) / 2)

def image_proxies(path):
    im = Image.open(path).convert('RGB'); im.thumbnail((512, 512))
    rgb = np.asarray(im)
    a = rgb.astype(np.float32) / 255
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    s, v = hsv[..., 1] / 255, hsv[..., 2] / 255
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    R, G, Bc = a[..., 0], a[..., 1], a[..., 2]
    exg = 2 * G - R - Bc
    edges = cv2.Canny(gray, 100, 200) > 0
    h, w = gray.shape; bh, bw = int(0.2 * h), int(0.2 * w)
    border = np.ones_like(edges); border[bh:h - bh, bw:w - bw] = False
    rg, yb = R - G, 0.5 * (R + G) - Bc
    return {'mean_V': v.mean(), 'overexposed': (v > 0.98).mean() * 100, 'underexposed': (v < 0.05).mean() * 100,
            'specular': ((v > 0.9) & (s < 0.15)).mean() * 100, 'mean_S': s.mean(),
            'log_lap_var': np.log1p(cv2.Laplacian(gray, cv2.CV_64F).var()),
            'green_frac': (exg > 0.05).mean() * 100, 'edge_density': edges.mean() * 100,
            'border_edge': edges[border].mean() * 100,
            'colorfulness': np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2),
            'jpeg_blockiness': jpeg_blockiness(gray)}

PROXIES = ['mean_V', 'overexposed', 'underexposed', 'specular', 'mean_S', 'log_lap_var',
           'green_frac', 'edge_density', 'border_edge', 'colorfulness', 'jpeg_blockiness']

prox_file = ATAB / 'A8_proxies_thai_full.csv'
if prox_file.exists():
    df_prox = pd.read_csv(prox_file)
else:
    with ThreadPoolExecutor(16) as ex:
        feats = list(ex.map(image_proxies, [p for p, _ in thai_full]))
    df_prox = pd.DataFrame(feats); df_prox.insert(0, 'path', [p for p, _ in thai_full])
    df_prox.insert(1, 'label', [CLASS_NAMES[l] for _, l in thai_full])
    df_prox.to_csv(prox_file, index=False)

# per-image correctness on thai_full
y_full = labels_of('thai_full')
for arch in KEY_ARCHS:
    for cfg in ['S5', 'S6-noTTA', 'S6+TTA']:
        try:
            df_prox[f'correct_{cfg}_{arch}'] = (get_probs(cfg, arch, 'thai_full').argmax(1) == y_full).astype(int)
        except FileNotFoundError:
            pass

import statsmodels.formula.api as smf
rows = []
Z = df_prox[PROXIES].apply(lambda c: (c - c.mean()) / (c.std() + 1e-9))
for col in [c for c in df_prox.columns if c.startswith('correct_')]:
    d = pd.concat([Z, df_prox[['label', col]]], axis=1).rename(columns={col: 'correct'})
    if d.correct.nunique() < 2:
        continue
    try:
        fit = smf.logit('correct ~ ' + ' + '.join(PROXIES) + ' + C(label)', data=d).fit(disp=0, maxiter=200)
    except Exception as e:
        print(f'[WARNING] {col}: {e}'); continue
    ci = fit.conf_int()
    for v in PROXIES:
        rows.append({'model': col.replace('correct_', ''), 'proxy': v,
                     'OR_per_SD': np.exp(fit.params[v]), 'OR_lo': np.exp(ci.loc[v, 0]),
                     'OR_hi': np.exp(ci.loc[v, 1]), 'p': fit.pvalues[v]})
df_logit = pd.DataFrame(rows)
if len(df_logit):
    df_logit['p_holm'] = df_logit.groupby('model').p.transform(lambda s: multipletests(s, method='holm')[1])
save_table(df_logit.round(4), 'A8_error_logit')
print('OR < 1 = a 1 SD increase in the metric lowers the odds of a correct prediction')
if len(df_logit):
    print(df_logit[df_logit.p_holm < 0.05].round(3).to_string(index=False))

# accuracy by quartile of each metric (for plotting)
rows = []
for v in PROXIES:
    q = pd.qcut(df_prox[v].rank(method='first'), 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    for col in [c for c in df_prox.columns if c.startswith('correct_')]:
        for qq, g in df_prox.groupby(q):
            rows.append({'proxy': v, 'quartile': qq, 'model': col.replace('correct_', ''), 'acc': g[col].mean() * 100, 'n': len(g)})
save_table(pd.DataFrame(rows).round(2), 'A8_acc_by_quartile')

# Domain shift: Thai vs each source (Cohen's d) — for §3.2, alongside/instead of MMD
def sample_paths(root, k=400, seed=SEED):
    ps = [str(p) for p in Path(root).rglob('*') if p.suffix.lower() in IMG_EXT]
    rng = np.random.default_rng(seed)
    return list(rng.choice(ps, size=min(k, len(ps)), replace=False))

shift_file = ATAB / 'A8_domain_shift_proxies.csv'
if not shift_file.exists():
    parts = []
    for name, root in [('PlantVillage', DATASET_PATHS['PlantVillage']), ('PlantDoc', DATASET_PATHS['PlantDoc']),
                       ('SLD', SLD_FILTER)]:
        with ThreadPoolExecutor(16) as ex:
            f = pd.DataFrame(list(ex.map(image_proxies, sample_paths(root))))
        f['dataset'] = name; parts.append(f)
    t = df_prox.sample(min(400, len(df_prox)), random_state=SEED)[PROXIES].copy(); t['dataset'] = 'Thai'
    pd.concat(parts + [t]).to_csv(shift_file, index=False)
df_shift = pd.read_csv(shift_file)
rows = []
th = df_shift[df_shift.dataset == 'Thai']
for src in ['PlantVillage', 'PlantDoc', 'SLD']:
    sr = df_shift[df_shift.dataset == src]
    for v in PROXIES:
        sp = np.sqrt((th[v].var() + sr[v].var()) / 2) + 1e-9
        rows.append({'source': src, 'proxy': v, 'thai_mean': th[v].mean(), 'src_mean': sr[v].mean(),
                     'cohen_d(Thai−src)': (th[v].mean() - sr[v].mean()) / sp})
df_d = save_table(pd.DataFrame(rows).round(3), 'A8_domain_shift_cohen_d')
print(df_d.pivot(index='proxy', columns='source', values='cohen_d(Thai−src)').round(2))

# Template for manual condition labelling (N1) — no predictions shown (blind), 20% double-labelled to compute κ
tmpl = pd.DataFrame({'path': [p for p, _ in thai_full]}).sample(frac=1, random_state=SEED).reset_index(drop=True)
tmpl.insert(0, 'img_id', [f'T{i:05d}' for i in range(len(tmpl))])
tmpl['double_annotate'] = (np.random.default_rng(SEED).random(len(tmpl)) < 0.2).astype(int)
for c in ['sun_glare', 'hard_shadow', 'water_droplets', 'motion_blur', 'out_of_focus', 'occlusion',
          'multi_leaf', 'cluttered_background', 'far_distance', 'low_light', 'notes', 'annotator']:
    tmpl[c] = ''
tmpl.to_csv(ATAB / 'N1_condition_annotation_template.csv', index=False)
print(f'N1 template → {ATAB / "N1_condition_annotation_template.csv"} (enter 0/1 per condition)')

## A9 — Quantitative Grad-CAM (comparable with Xiang et al. 2026)
* **border ratio** = share of CAM mass in the outer 20% border of the image (high = attends to background/edges)
* **leaf ratio** = share of CAM mass on green pixels (ExG) ≈ attends to the leaf
* compares S5 vs S6, and correct vs incorrect predictions

In [ ]:
# ── A9. Quantitative Grad-CAM ───────────────────────────────────────────────
from pytorch_grad_cam import GradCAM
from scipy.stats import mannwhitneyu

def cam_layer(base, arch):
    if arch == 'resnet50':        return [base.layer4[-1]]
    if arch == 'convnext_base':   return [base.stages[-1]]
    if arch == 'densenet121':     return [base.features.denseblock4]
    if arch == 'efficientnet_b4': return [base.conv_head]
    raise ValueError(f'{arch}: ViT needs reshape_transform — skipped')

_mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1); _std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
_border = np.ones((IMG_SIZE, IMG_SIZE), bool); _b = int(0.2 * IMG_SIZE)
_border[_b:-_b, _b:-_b] = False

cam_file = ATAB / 'A9_gradcam_per_image.csv'
if cam_file.exists():
    df_cam = pd.read_csv(cam_file)
else:
    rows = []
    for arch in KEY_ARCHS:
        for scn in ['S5', 'S6']:
            if not Path(ckpt_path(scn, arch)).exists():
                continue
            base = load_ckpt(scn, arch).to(DEVICE)
            model = SelectiveHead(base, THAI_IDX).to(DEVICE).eval()
            with GradCAM(model=model, target_layers=cam_layer(base, arch)) as cam:
                for x, lab, idx in make_loader(thai_test, bs=32, return_index=True):
                    g = cam(input_tensor=x.to(DEVICE), targets=None)   # (B,H,W); None = predicted class
                    with torch.no_grad():
                        pred = model(x.to(DEVICE)).argmax(1).cpu().numpy()
                    raw = (x * _std + _mean).clamp(0, 1).numpy()   # (B,3,H,W)
                    exg = 2 * raw[:, 1] - raw[:, 0] - raw[:, 2]
                    for k in range(len(lab)):
                        m = g[k] / (g[k].sum() + 1e-9)
                        rows.append({'scenario': scn, 'arch': ARCH_LABEL[arch], 'img': thai_test[int(idx[k])][0],
                                     'label': int(lab[k]), 'pred': int(pred[k]), 'correct': int(pred[k] == lab[k]),
                                     'border_ratio': float(m[_border].sum()), 'leaf_ratio': float(m[exg[k] > 0.05].sum())})
            del base, model; torch.cuda.empty_cache()
    df_cam = pd.DataFrame(rows); df_cam.to_csv(cam_file, index=False)

summ = df_cam.groupby(['arch', 'scenario', 'correct'])[['border_ratio', 'leaf_ratio']].agg(['mean', 'std']).round(3)
print(summ)
rows = []
for arch in df_cam.arch.unique():
    a = df_cam[(df_cam.arch == arch) & (df_cam.scenario == 'S5')]
    b = df_cam[(df_cam.arch == arch) & (df_cam.scenario == 'S6')]
    for met in ['border_ratio', 'leaf_ratio']:
        if len(a) and len(b):
            rows.append({'arch': arch, 'metric': met, 'S5': a[met].mean(), 'S6': b[met].mean(),
                         'p_MWU(S5 vs S6)': mannwhitneyu(a[met], b[met]).pvalue,
                         'p_MWU(correct vs wrong, S6)': mannwhitneyu(b[b.correct == 1][met], b[b.correct == 0][met]).pvalue
                         if b.correct.nunique() == 2 else np.nan})
save_table(pd.DataFrame(rows).round(4), 'A9_gradcam_summary')

## A10 — Corrected DA baselines
**AdaBN −17.84% is most likely a problem of how it was run, not of the method:** the original cell 10c estimated BN statistics from a test loader that was **not shuffled**
(FilteredImageFolder is ordered by class), so batches of 256 were [Healthy 200 + RedRot 56], [RedRot 144 + Rust 112], [Rust 88], and
`momentum=None` averages the statistics of each batch with **equal weight** → biased mean/variance (the last batch of 88 images gets 1/3 of the weight, and between-batch variance is lost).

**The original CORAL had no effect:** the transform was computed but never applied → results identical to S5 in every digit.

This cell runs: AdaBN in 4 variants (including the original, to confirm the cause), TENT, and a correct CORAL/moment-matching (Sun et al. 2016).

In [ ]:
# ── A10. Corrected DA baselines ─────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def adabn(model, samples, shuffle=True, bs=128, name_filter=None):
    m = deepcopy(model).eval()
    bns = [mod for n, mod in m.named_modules()
           if isinstance(mod, nn.modules.batchnorm._BatchNorm) and (name_filter is None or name_filter(n))]
    for b in bns:
        b.reset_running_stats(); b.momentum = None; b.train()
    with torch.no_grad():
        for x, _ in make_loader(samples, bs=bs, shuffle=shuffle, workers=4):
            m(x.to(DEVICE))
    for b in bns:
        b.eval()
    return m

def acc_on(model, samples):
    y = np.array([l for _, l in samples])
    return (infer_logits(model, make_loader(samples)).argmax(1) == y).mean() * 100

def tent(model, adapt_samples, eval_samples, lr=1e-4, bs=64, epochs=1, online=False):
    """TENT (Wang et al. 2021): adapt only BN/LN affine parameters by entropy minimization.
    online=True → adapt on eval_samples batch by batch (standard TENT protocol)"""
    m = deepcopy(model)
    params = []
    for mod in m.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            mod.track_running_stats = False; mod.running_mean = None; mod.running_var = None
            params += [p for p in mod.parameters()]
        elif isinstance(mod, nn.LayerNorm):
            params += [p for p in mod.parameters()]
    for p in m.parameters():
        p.requires_grad = False
    for p in params:
        p.requires_grad = True
    m.train()
    for mod in m.modules():          # disable dropout / drop-path
        if isinstance(mod, nn.Dropout) or mod.__class__.__name__ == 'DropPath':
            mod.eval()
    opt = torch.optim.Adam(params, lr=lr)
    y_eval = np.array([l for _, l in eval_samples]); preds = np.full(len(y_eval), -1)
    loops = [(eval_samples, True)] if online else [(adapt_samples, False)] * epochs + [(eval_samples, True)]
    for samples, record in loops:
        update = online or not record
        for x, _, idx in make_loader(samples, bs=bs, shuffle=True, workers=4, return_index=True):
            with torch.set_grad_enabled(update):
                out = m(x.to(DEVICE))
            if record:
                preds[idx.numpy()] = out.argmax(1).detach().cpu().numpy()
            if update:
                pr = out.softmax(1)
                loss = -(pr * out.log_softmax(1)).sum(1).mean()
                opt.zero_grad(); loss.backward(); opt.step()
    return (preds == y_eval).mean() * 100

def coral_transform(Xs, Xt, lam=1.0):
    """Sun et al. (2016) CORAL: whiten the source, then re-colour with the target covariance"""
    Xs = torch.tensor(Xs, dtype=torch.float64); Xt = torch.tensor(Xt, dtype=torch.float64)
    mus, mut = Xs.mean(0), Xt.mean(0)
    Cs = torch.cov((Xs - mus).T) + lam * torch.eye(Xs.shape[1], dtype=torch.float64)
    Ct = torch.cov((Xt - mut).T) + lam * torch.eye(Xt.shape[1], dtype=torch.float64)
    ws, Vs = torch.linalg.eigh(Cs); wt, Vt = torch.linalg.eigh(Ct)
    Cs_ih = Vs @ torch.diag(ws.clamp_min(1e-12).rsqrt()) @ Vs.T
    Ct_h = Vt @ torch.diag(wt.clamp_min(1e-12).sqrt()) @ Vt.T
    return ((Xs - mus) @ Cs_ih @ Ct_h + mut).numpy()

def linear_eval(Xs, ys, Xte, yte):
    sc = StandardScaler().fit(Xs)
    clf = LogisticRegression(max_iter=3000, C=1.0).fit(sc.transform(Xs), ys)
    return (clf.predict(sc.transform(Xte)) == yte).mean() * 100

sld3 = list_images(SLD_FILTER, CLASS_NAMES)   # the only source with 3-class sugarcane labels
y_te = labels_of('thai_test')
da_file = ATAB / 'A10_da_baselines.csv'
rows = pd.read_csv(da_file).to_dict('records') if da_file.exists() else []
done = {(r['arch'], r['method']) for r in rows}

def add(arch, method, acc, note=''):
    rows.append({'arch': ARCH_LABEL[arch], 'method': method, 'acc': round(acc, 2), 'note': note})
    pd.DataFrame(rows).to_csv(da_file, index=False)
    print(f'  {ARCH_LABEL[arch]:<16} {method:<42} {acc:6.2f}%')

for arch in EVAL_ARCHS:
    if not Path(ckpt_path('S5', arch)).exists() or (ARCH_LABEL[arch], 'S5 (no DA)') in done:
        continue
    base = load_ckpt('S5', arch).to(DEVICE)
    sel = SelectiveHead(base, THAI_IDX).to(DEVICE).eval()
    add(arch, 'S5 (no DA)', acc_on(sel, thai_test))
    if arch in BN_ARCHS:
        add(arch, 'AdaBN — test, class-sorted batches (original)', acc_on(adabn(sel, thai_test, shuffle=False, bs=256), thai_test), 'transductive; reproduces cell 10c')
        add(arch, 'AdaBN — test, shuffled', acc_on(adabn(sel, thai_test, shuffle=True), thai_test), 'transductive')
        add(arch, 'AdaBN — Thai-train unlabeled, shuffled', acc_on(adabn(sel, thai_train, shuffle=True), thai_test), 'non-transductive')
        last = {'resnet50': 'layer4', 'densenet121': 'denseblock4', 'efficientnet_b4': 'blocks.6'}[arch]
        add(arch, f'AdaBN — Thai-train, last stage only ({last})',
            acc_on(adabn(sel, thai_train, name_filter=lambda n: last in n), thai_test))
    else:
        add(arch, 'AdaBN', float('nan'), 'N/A — LayerNorm architecture')
    add(arch, 'TENT — online on test (standard)', tent(sel, None, thai_test, online=True))
    add(arch, 'TENT — offline, adapt on Thai-train', tent(sel, thai_train, thai_test, online=False))
    # Feature-level: linear classifier on S5 features trained on SLD (3 classes), then aligned to Thai-train (no labels used)
    fe = feature_extractor(base, arch).to(DEVICE)
    Xs = infer_logits(fe, make_loader(sld3)); ys = np.array([l for _, l in sld3])
    Xt = infer_logits(fe, make_loader(thai_train)); Xte = infer_logits(fe, make_loader(thai_test))
    add(arch, 'Linear on S5 features, SLD labels (no align)', linear_eval(Xs, ys, Xte, y_te))
    Xs_mm = (Xs - Xs.mean(0)) / (Xs.std(0) + 1e-6) * Xt.std(0) + Xt.mean(0)
    add(arch, '  + diagonal moment matching', linear_eval(Xs_mm, ys, Xte, y_te))
    add(arch, '  + CORAL (Sun et al. 2016)', linear_eval(coral_transform(Xs, Xt), ys, Xte, y_te))
    del base, sel, fe; torch.cuda.empty_cache()

df_da = save_table(pd.DataFrame(rows), 'A10_da_baselines')
print(df_da.pivot_table(index='method', columns='arch', values='acc', sort=False).round(2))

## B1 — Foundation models + linear probe + **corrected source ablation** (only a linear head is trained)
* Features are extracted once from the same source pool (PV+PD+SLD, 59k images, union of 70 classes), then a linear head is trained → Thai 3 classes, in the same way as S5
* Backbones: DINOv2-B/14, CLIP ViT-B/16, BioCLIP, ImageNet ResNet-50, ImageNet ConvNeXt-B and **PV ResNet-50 (S1), frozen**
* Source subsets (separating diversity from the number of images):
  `SLD` · `PV+SLD` · `PD+SLD` · `PV+PD+SLD` · and 3 equal-size subsets `PVsub+SLD` / `PD+SLD` / `PVhalf+PDhalf+SLD`
* **"S4-fair"** = frozen PV ResNet-50 (S1) + a head trained on SLD → replaces the original S4, which had a random head
* CLIP / BioCLIP zero-shot with prompts (no training data at all)

In [ ]:
# ── B1.0 Source cache (shared with B2/B4) ────────────────────────────────────
import timm
SOURCES = ['PlantVillage', 'PlantDoc', 'SLD']
SRC = CachedMultiSourceDataset([DATASET_PATHS[s] for s in SOURCES])        # 224×224 uint8 in RAM (~9 GB)
assert SRC.union_classes == UNION_CLASSES, 'union does not match S5/S6'
SRC_TAGS = np.concatenate([[s] * len(AlbumentationsDataset(DATASET_PATHS[s]).samples) for s in SOURCES])
SRC_Y = np.asarray(SRC.labels)
assert len(SRC_TAGS) == len(SRC_Y)
IDX = {s: np.where(SRC_TAGS == s)[0] for s in SOURCES}
print({s: len(v) for s, v in IDX.items()})

def subset_indices(name, seed=SEED):
    rng = np.random.default_rng(seed); nPD = len(IDX['PlantDoc'])
    PV, PD, SLD = IDX['PlantVillage'], IDX['PlantDoc'], IDX['SLD']
    return {'SLD': SLD,
            'PV+SLD': np.r_[PV, SLD],
            'PD+SLD': np.r_[PD, SLD],
            'PV+PD+SLD': np.r_[PV, PD, SLD],
            'PVsub+SLD': np.r_[rng.choice(PV, nPD, replace=False), SLD],
            'PVhalf+PDhalf+SLD': np.r_[rng.choice(PV, nPD // 2, replace=False),
                                       rng.choice(PD, nPD - nPD // 2, replace=False), SLD]}[name]

SUBSETS = ['SLD', 'PV+SLD', 'PD+SLD', 'PV+PD+SLD', 'PVsub+SLD', 'PVhalf+PDhalf+SLD']
VARIABLE_SUBSETS = {'PVsub+SLD', 'PVhalf+PDhalf+SLD'}   # depend on the sampling seed

# ── helpers for training from the cache (used in B2 and B4) ──
class CacheSubset(Dataset):
    def __init__(self, idx, tfm):
        self.idx, self.tfm = np.asarray(idx), tfm
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        return self.tfm(image=SRC.images[j])['image'], int(SRC_Y[j])

def loaders_from_idx(idx, aug_mode='standard', seed=SEED, val_frac=0.15):
    rng = np.random.default_rng(seed); idx = rng.permutation(idx); nv = max(1, int(len(idx) * val_frac))
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    return (DataLoader(CacheSubset(idx[nv:], get_train_transform(aug_mode)), batch_size=BATCH_SIZE, shuffle=True, **kw),
            DataLoader(CacheSubset(idx[:nv], VAL_TRANSFORM), batch_size=BATCH_SIZE, shuffle=False, **kw))

def train_or_load(tag, arch, idx, aug_mode, seed):
    path = AMOD / f'{tag}_{arch}_s{seed}.pt'
    if path.exists():
        m = build_model(arch, num_classes=N_UNION, pretrained=False)
        m.load_state_dict(torch.load(path, map_location=DEVICE)); return m.eval(), path
    set_all_seeds(seed)
    tr, vl = loaders_from_idx(idx, aug_mode, seed)
    t0 = time.time()
    m, best, _ = train_model(arch, tr, vl, list(range(N_UNION)), save_path=str(path), verbose=True)
    print(f'  ⏱ {tag} {arch} s{seed}: {(time.time() - t0) / 60:.1f} min | val {best:.2f}%')
    return m.eval(), path

In [ ]:
# ── B1.1 Feature extraction ─────────────────────────────────────────────────
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

def load_backbone(name):
    """Return (fn: tensor→features, mean, std, extra)"""
    if name == 'DINOv2-B14':
        m = timm.create_model('vit_base_patch14_dinov2.lvd142m', pretrained=True, num_classes=0, img_size=IMG_SIZE)
        m = m.to(DEVICE).eval(); return m, IMAGENET_MEAN, IMAGENET_STD, None
    if name in ('CLIP-B16', 'BioCLIP'):
        import open_clip
        tag = ('ViT-B-16', 'openai') if name == 'CLIP-B16' else ('hf-hub:imageomics/bioclip', None)
        m, _, _ = open_clip.create_model_and_transforms(tag[0], pretrained=tag[1]) if tag[1] else \
                  open_clip.create_model_and_transforms(tag[0])
        m = m.to(DEVICE).eval(); tok = open_clip.get_tokenizer(tag[0])
        enc = lambda x, m=m: F.normalize(m.encode_image(x), dim=-1)
        return enc, CLIP_MEAN, CLIP_STD, (m, tok)
    if name == 'ImageNet-R50':
        m = build_model('resnet50', num_classes=1000, pretrained=True); m.fc = nn.Identity()
        return m.eval(), IMAGENET_MEAN, IMAGENET_STD, None
    if name == 'ImageNet-ConvNeXtB':
        m = timm.create_model('convnext_base', pretrained=True, num_classes=0).to(DEVICE)
        return m.eval(), IMAGENET_MEAN, IMAGENET_STD, None
    if name == 'PV-R50 (S1)':
        return feature_extractor(load_ckpt('S1', 'resnet50'), 'resnet50').to(DEVICE), IMAGENET_MEAN, IMAGENET_STD, None
    raise ValueError(name)

BACKBONES = ['DINOv2-B14', 'CLIP-B16', 'BioCLIP', 'ImageNet-R50', 'ImageNet-ConvNeXtB', 'PV-R50 (S1)']

@torch.no_grad()
def feats_from_cache(fn, mean, std, bs=256):
    mu = torch.tensor(mean, device=DEVICE).view(1, 3, 1, 1); sd = torch.tensor(std, device=DEVICE).view(1, 3, 1, 1)
    out = []
    for i in range(0, len(SRC.images), bs):
        x = torch.from_numpy(SRC.images[i:i + bs]).to(DEVICE).permute(0, 3, 1, 2).float() / 255
        out.append(fn((x - mu) / sd).float().cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def feats_from_samples(fn, mean, std, samples):
    tfm = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(mean=mean, std=std), ToTensorV2()])
    out = []
    for x, _ in make_loader(samples, transform=tfm):
        out.append(fn(x.to(DEVICE)).float().cpu())
    return torch.cat(out).numpy()

def feat_path(bb, split):
    return AFEAT / f"{bb.replace(' ', '_').replace('(', '').replace(')', '')}_{split}.npy"

for bb in BACKBONES:
    need = [s for s in ['source', *EVAL_SETS] if not feat_path(bb, s).exists()]
    if not need:
        continue
    t0 = time.time()
    fn, mean, std, _ = load_backbone(bb)
    for split in need:
        X = feats_from_cache(fn, mean, std) if split == 'source' else feats_from_samples(fn, mean, std, EVAL_SETS[split])
        np.save(feat_path(bb, split), X.astype(np.float16))
    print(f'  {bb}: {need} | {time.time() - t0:.0f}s')
    del fn; torch.cuda.empty_cache()
print('[OK] features ready')

In [ ]:
# ── B1.2 Linear probes × source subsets ─────────────────────────────────────
def train_linear(X, y, n_cls=N_UNION, min_steps=3000, epochs=40, lr=3e-3, wd=1e-4, bs=512, seed=SEED):
    """At least min_steps steps for every subset → SLD-only (2.5k images) is not under-trained compared with PV+PD+SLD (59k)"""
    torch.manual_seed(seed)
    epochs = max(epochs, math.ceil(min_steps / math.ceil(len(X) / bs)))
    X = torch.as_tensor(X, dtype=torch.float32, device=DEVICE); y = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
    mu, sd = X.mean(0), X.std(0) + 1e-6
    head = nn.Linear(X.shape[1], n_cls).to(DEVICE)
    nn.init.zeros_(head.weight); nn.init.zeros_(head.bias)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    steps = epochs * math.ceil(len(X) / bs)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    for _ in range(epochs):
        perm = torch.randperm(len(X), device=DEVICE)
        for i in range(0, len(X), bs):
            b = perm[i:i + bs]
            loss = F.cross_entropy(head((X[b] - mu) / sd), y[b], label_smoothing=0.1)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    @torch.no_grad()
    def predict(Z):
        Z = torch.as_tensor(np.asarray(Z, np.float32), device=DEVICE)
        return head((Z - mu) / sd).cpu().numpy()
    return predict

N_SEEDS_PROBE = 3
probe_file = ATAB / 'B1_linear_probe_subsets.csv'
rows = pd.read_csv(probe_file).to_dict('records') if probe_file.exists() else []
done = {(r['backbone'], r['subset'], r['seed']) for r in rows}
for bb in BACKBONES:
    Xs = np.load(feat_path(bb, 'source')).astype(np.float32)
    Xev = {t: np.load(feat_path(bb, t)).astype(np.float32) for t in ['thai_test', 'thai_full']}
    for sub in SUBSETS:
        for s in range(N_SEEDS_PROBE):
            seed = SEED + s
            if (bb, sub, seed) in done:
                continue
            idx = subset_indices(sub, seed)
            pred = train_linear(Xs[idx], SRC_Y[idx], seed=seed)
            r = {'backbone': bb, 'subset': sub, 'seed': seed, 'n_train': len(idx)}
            for t, Xe in Xev.items():
                yy = labels_of(t)
                r[f'acc_{t}'] = (to_thai(pred(Xe)).argmax(1) == yy).mean() * 100
            rows.append(r)
        pd.DataFrame(rows).to_csv(probe_file, index=False)
    print(f'  {bb} done')
df_probe = pd.DataFrame(rows)
agg = df_probe.groupby(['backbone', 'subset']).agg(n=('n_train', 'first'),
                                                   acc_test=('acc_thai_test', 'mean'), sd_test=('acc_thai_test', 'std'),
                                                   acc_full=('acc_thai_full', 'mean'), sd_full=('acc_thai_full', 'std')).round(2).reset_index()
save_table(agg, 'B1_linear_probe_summary')
print(agg.pivot(index='backbone', columns='subset', values='acc_full')[SUBSETS])
print('\nS4-fair (PV-R50 frozen + SLD head):',
      agg[(agg.backbone == 'PV-R50 (S1)') & (agg.subset == 'SLD')][['acc_test', 'acc_full']].values)

In [ ]:
# ── B1.3 CLIP / BioCLIP zero-shot ───────────────────────────────────────────
TEMPLATES = ['a photo of a {}.', 'a close-up photo of a {}.', 'a field photo of a {}.',
             'a photo of a {}, a plant disease.', 'a smartphone photo of a {} in a farm.']
CLASS_TEXT = {'Healthy': 'healthy sugarcane leaf', 'RedRot': 'sugarcane leaf with red rot disease',
              'Rust': 'sugarcane leaf with rust disease'}
rows = []
for bb in ['CLIP-B16', 'BioCLIP']:
    _, _, _, (m, tok) = load_backbone(bb)
    with torch.no_grad():
        T = []
        for c in CLASS_NAMES:
            t = tok([tp.format(CLASS_TEXT[c]) for tp in TEMPLATES]).to(DEVICE)
            e = F.normalize(m.encode_text(t), dim=-1).mean(0)
            T.append(F.normalize(e, dim=0))
        T = torch.stack(T).float().cpu().numpy()
    for t in ['thai_test', 'thai_full']:
        Xe = np.load(feat_path(bb, t)).astype(np.float32)
        yy = labels_of(t); pr = (Xe @ T.T).argmax(1)
        lo, hi = boot_ci(pr == yy)
        rows.append({'model': f'{bb} zero-shot', 'set': t, 'acc': (pr == yy).mean() * 100, 'ci_lo': lo, 'ci_hi': hi})
    del m; torch.cuda.empty_cache()
print(save_table(pd.DataFrame(rows).round(2), 'B1_clip_zeroshot').to_string(index=False))

## B2 — Full fine-tuning source ablation (same pipeline as S5: `train_model`, standard aug, ImageNet init)
Confirms B1 with real fine-tuning for `KEY_ARCHS` → table **S4′ (SLD only) / PV+SLD / PD+SLD / PV+PD+SLD / equal-size**
`SLD` is the fair single-source baseline (sees the 3 sugarcane classes, like S5) and replaces the original S4.

In [ ]:
# ── B2. Fine-tune source ablation ───────────────────────────────────────────
B2_ARCHS = KEY_ARCHS
B2_SUBSETS = ['SLD', 'PD+SLD', 'PVsub+SLD', 'PVhalf+PDhalf+SLD', 'PV+SLD', 'PV+PD+SLD']  # ordered fast→slow
for arch in B2_ARCHS:
    for sub in B2_SUBSETS:
        tag = f'ABL-{sub}'
        if all(pred_file(f'{tag}-s{SEED}', arch, t).exists() for t in EVAL_SETS):
            continue
        print(f'[B2] {sub} | {arch}')
        m, _ = train_or_load(tag, arch, subset_indices(sub, SEED), 'standard', SEED)
        export_logits(f'{tag}-s{SEED}', arch, model=m.to(DEVICE), tta=False)

rows = []
for arch in B2_ARCHS:
    for sub in B2_SUBSETS:
        for t in ['thai_test', 'thai_full']:
            f = pred_file(f'ABL-{sub}-s{SEED}', arch, t)
            if not f.exists():
                continue
            yy = labels_of(t); c = to_thai(np.load(f)).argmax(1) == yy
            lo, hi = boot_ci(c)
            rows.append({'arch': ARCH_LABEL[arch], 'subset': sub, 'n_train_pool': len(subset_indices(sub, SEED)),
                         'set': t, 'acc': c.mean() * 100, 'ci_lo': lo, 'ci_hi': hi})
df_b2 = save_table(pd.DataFrame(rows).round(2), 'B2_finetune_source_ablation')
print(df_b2[df_b2.set == 'thai_full'].pivot(index='subset', columns='arch', values='acc').reindex(B2_SUBSETS))

# paired tests between subsets (thai_full) — main question: diversity or quantity?
B2_PAIRS = [('SLD', 'PV+SLD'), ('SLD', 'PD+SLD'), ('PV+SLD', 'PV+PD+SLD'), ('PD+SLD', 'PV+PD+SLD'),
            ('PVsub+SLD', 'PD+SLD'), ('PVsub+SLD', 'PVhalf+PDhalf+SLD'), ('PD+SLD', 'PVhalf+PDhalf+SLD')]
rows = []
yy = labels_of('thai_full')
for arch in B2_ARCHS:
    for a, b in B2_PAIRS:
        fa, fb = pred_file(f'ABL-{a}-s{SEED}', arch, 'thai_full'), pred_file(f'ABL-{b}-s{SEED}', arch, 'thai_full')
        if not (fa.exists() and fb.exists()):
            continue
        ca, cb = to_thai(np.load(fa)).argmax(1) == yy, to_thai(np.load(fb)).argmax(1) == yy
        pv, nb, nc = mcnemar_p(ca, cb); lo, hi = paired_boot_ci(ca, cb)
        rows.append({'arch': ARCH_LABEL[arch], 'from': a, 'to': b, 'delta': (cb.mean() - ca.mean()) * 100,
                     'ci_lo': lo, 'ci_hi': hi, 'p': pv})
df_b2p = pd.DataFrame(rows)
if len(df_b2p):
    df_b2p['p_holm'] = multipletests(df_b2p.p, method='holm')[1]
print(save_table(df_b2p.round(4), 'B2_paired').to_string(index=False))

## B3 — Label efficiency: how many labelled Thai images match multi-source training?
Uses the 2,400 Thai training images that were otherwise unused (images near-duplicate with test already removed in A1)
* 3 starting points: ImageNet · PV (S1) · Multi-source (S5)
* k = 5, 10, 25, 50, 100, 200, all images per class × 3 random draws
* the same number of steps for every setting (no early stopping on test)

In [ ]:
# ── B3. Label-efficiency curves ─────────────────────────────────────────────
LE_ARCHS = ['resnet50']              # add 'convnext_base' if time allows (~2× longer)
LE_K = [5, 10, 25, 50, 100, 200, 'all']
LE_INITS = ['imagenet', 'pv_s1', 's5_multi']
LE_DRAWS, LE_STEPS, LE_BS = 3, 400, 32

def load_arr(samples):
    return np.stack([np.array(Image.open(p).convert('RGB').resize((IMG_SIZE, IMG_SIZE))) for p, _ in samples])

TR_IMG = load_arr(thai_train); TR_Y = np.array([l for _, l in thai_train])
print(f'Thai train for LE: {len(TR_Y)} | per class {np.bincount(TR_Y)}')

def le_model(init, arch):
    if init == 'imagenet':
        return build_model(arch, num_classes=NUM_CLASSES, pretrained=True), None
    if init == 'pv_s1':
        return replace_head(load_ckpt('S1', arch).to(DEVICE), arch, NUM_CLASSES), None
    if init == 's5_multi':
        return load_ckpt('S5', arch).to(DEVICE), np.array(THAI_IDX)   # continue training on the original 70-class head
    raise ValueError(init)

def finetune(m, imgs, ys, ymap, steps=LE_STEPS, bs=LE_BS, seed=SEED):
    set_all_seeds(seed); rng = np.random.default_rng(seed)
    tfm = get_train_transform('standard')
    head = [p for n, p in m.named_parameters() if any(k in n for k in ('fc.', 'head', 'classifier'))]
    hid = {id(p) for p in head}
    body = [p for p in m.parameters() if id(p) not in hid]
    opt = torch.optim.AdamW([{'params': body, 'lr': 1e-4}, {'params': head, 'lr': 1e-3}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    scaler = torch.amp.GradScaler(device='cuda'); crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    m.train()
    for _ in range(steps):
        b = rng.choice(len(ys), size=bs, replace=len(ys) < bs)
        x = torch.stack([tfm(image=imgs[i])['image'] for i in b]).to(DEVICE)
        t = torch.as_tensor(ys[b] if ymap is None else ymap[ys[b]], device=DEVICE)
        with torch.autocast('cuda', dtype=torch.float16):
            loss = crit(m(x), t)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
    return m.eval()

le_file = ATAB / 'B3_label_efficiency_runs.csv'
rows = pd.read_csv(le_file).to_dict('records') if le_file.exists() else []
done = {(r['arch'], r['init'], str(r['k']), r['draw']) for r in rows}
y_te = labels_of('thai_test')
for arch in LE_ARCHS:
    for init in LE_INITS:
        for k in LE_K:
            for d in range(LE_DRAWS if k != 'all' else 1):
                if (ARCH_LABEL[arch], init, str(k), d) in done:
                    continue
                rng = np.random.default_rng(1000 + d)
                sel = np.arange(len(TR_Y)) if k == 'all' else np.concatenate(
                    [rng.choice(np.where(TR_Y == c)[0], size=k, replace=False) for c in range(NUM_CLASSES)])
                t0 = time.time()
                m, ymap = le_model(init, arch)
                m = finetune(m, TR_IMG[sel], TR_Y[sel], ymap, seed=SEED + d)
                z = to_thai(infer_logits(m, make_loader(thai_test)))
                acc = (z.argmax(1) == y_te).mean() * 100
                rows.append({'arch': ARCH_LABEL[arch], 'init': init, 'k': str(k), 'draw': d,
                             'n_labels': len(sel), 'acc': acc, 'sec': time.time() - t0})
                pd.DataFrame(rows).to_csv(le_file, index=False)
                print(f'  {arch} {init:<9} k={str(k):>4} d{d}: {acc:.2f}%  ({time.time() - t0:.0f}s)')
                del m; torch.cuda.empty_cache()

df_le = pd.DataFrame(rows)
s5_zero = {a: (get_probs('S5', a, 'thai_test').argmax(1) == y_te).mean() * 100 for a in LE_ARCHS}
agg = df_le.groupby(['arch', 'init', 'k'], sort=False).acc.agg(['mean', 'std', 'count']).round(2).reset_index()
save_table(agg, 'B3_label_efficiency_summary')
print(agg.pivot(index='k', columns='init', values='mean').reindex([str(k) for k in LE_K]))

fig, ax = plt.subplots(figsize=(6, 4))
kx = {str(k): (k if k != 'all' else int(np.bincount(TR_Y).min())) for k in LE_K}
for init in LE_INITS:
    g = agg[(agg.init == init) & (agg.arch == ARCH_LABEL[LE_ARCHS[0]])]
    ax.errorbar([kx[k] for k in g.k], g['mean'], yerr=g['std'].fillna(0), marker='o', ms=4, capsize=2, label=init)
ax.axhline(s5_zero[LE_ARCHS[0]], color='grey', ls='--', lw=1, label='S5 (0 Thai labels)')
ax.set_xscale('log'); ax.set_xlabel('Thai labelled images per class'); ax.set_ylabel('Accuracy on Thai test (%)')
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(AFIG / 'B3_label_efficiency.png', dpi=300); plt.show()

# number of Thai labels per class that ImageNet-init needs to match S5 (interpolated on a log axis)
for arch in LE_ARCHS:
    g = agg[(agg.init == 'imagenet') & (agg.arch == ARCH_LABEL[arch])]
    xs, ys_ = np.log([kx[k] for k in g.k]), g['mean'].values
    tgt = s5_zero[arch]
    above = np.where(ys_ >= tgt)[0]
    if len(above) == 0:
        print(f'{arch}: ImageNet-init does not reach S5 ({tgt:.1f}%) even with all images')
    elif above[0] == 0:
        print(f'{arch}: ImageNet-init ≥ S5 from k={g.k.iloc[0]}')
    else:
        i = above[0]; f = (tgt - ys_[i - 1]) / (ys_[i] - ys_[i - 1])
        print(f'{arch}: S5 (0 Thai labels, {tgt:.1f}%) ≈ ImageNet-init + {np.exp(xs[i - 1] + f * (xs[i] - xs[i - 1])):.0f} Thai images per class')

## B4 — (optional, long) Seeds for S5 vs S6 with the same pipeline
The original S5 was trained with `make_multisource_loaders` + `train_model`, but S6 used `make_cached_loaders` + `train_model_s6`
(neither function is defined in the notebook) → the S5/S6 difference is confounded with the difference in training procedure.

This cell trains both with **the same** loader, function and validation split; only `aug_mode` differs (S5c vs S6c); seeds are set in `B4_SEEDS` (one seed, 42, was run).
TropAug is slow because it runs on the CPU (RandomRain/Fog) — the original S6 ConvNeXt took ~490 s/epoch → start with ResNet-50.

In [ ]:
# ── B4. Controlled seeds (S5c vs S6c) ───────────────────────────────────────
B4_ARCHS, B4_SEEDS = ['convnext_base', 'resnet50'], [42]   # ConvNeXt = largest drop, ResNet = deployed model
ALL_IDX = np.arange(len(SRC_Y))
for arch in B4_ARCHS:
    for s in B4_SEEDS:
        for aug, tag in [('standard', 'ABL-PV+PD+SLD'), ('tropical', 'TROP-PV+PD+SLD')]:
            if all(pred_file(f'{tag}-s{s}', arch, t, k).exists() for t in EVAL_SETS for k in ('clean', 'tta')):
                continue
            print(f'[B4] {tag} {arch} seed {s}')
            m, _ = train_or_load(tag, arch, ALL_IDX, aug, s)
            export_logits(f'{tag}-s{s}', arch, model=m.to(DEVICE), tta=True)

rows = []
for arch in B4_ARCHS:
    for s in B4_SEEDS:
        for t in ['thai_test', 'thai_full']:
            yy = labels_of(t); r = {'arch': ARCH_LABEL[arch], 'seed': s, 'set': t}
            for nm, tag, kind in [('S5c', 'ABL-PV+PD+SLD', 'clean'), ('S5c+TTA', 'ABL-PV+PD+SLD', 'tta'),
                                  ('S6c-noTTA', 'TROP-PV+PD+SLD', 'clean'), ('S6c+TTA', 'TROP-PV+PD+SLD', 'tta')]:
                f = pred_file(f'{tag}-s{s}', arch, t, kind)
                if f.exists():
                    r[nm] = (probs_from_views(to_thai(np.load(f))).argmax(1) == yy).mean() * 100
            rows.append(r)
df_b4 = save_table(pd.DataFrame(rows).round(2), 'B4_seeds_runs')
if len(df_b4) and {'S5c', 'S6c-noTTA'} <= set(df_b4.columns):
    df_b4['TropAug_effect'] = df_b4['S6c-noTTA'] - df_b4['S5c']
    print(df_b4.groupby(['arch', 'set'])[['S5c', 'S5c+TTA', 'S6c-noTTA', 'S6c+TTA', 'TropAug_effect']]
          .agg(['mean', 'std']).round(2))

In [ ]:
# ── B4p. Paired test S5c vs S6c on thai_test / thai_full (inference only) ──
rows, chk = [], []
for arch in ['convnext_base', 'resnet50']:
    corr = {}
    for nm, tag in [('S5c', 'ABL-PV+PD+SLD'), ('S6c', 'TROP-PV+PD+SLD')]:
        m = build_model(arch, num_classes=N_UNION, pretrained=False)
        m.load_state_dict(torch.load(AMOD / f'{tag}_{arch}_s42.pt', map_location=DEVICE)); m = m.to(DEVICE).eval()
        for tset in ['thai_test', 'thai_full']:
            y = labels_of(tset)
            corr[(nm, tset)] = to_thai(infer_logits(m, make_loader(EVAL_SETS[tset]))).argmax(1) == y
            chk.append({'arch': arch, 'model': nm, 'set': tset, 'acc': corr[(nm, tset)].mean() * 100})
        del m; torch.cuda.empty_cache()
    for tset in ['thai_test', 'thai_full']:
        a, b = corr[('S5c', tset)], corr[('S6c', tset)]
        lo, hi = paired_boot_ci(a, b)
        pv, n10, n01 = mcnemar_p(a, b)
        rows.append({'arch': arch, 'set': tset, 'S5c': a.mean() * 100, 'S6c': b.mean() * 100,
                     'diff(S6c−S5c)': (b.mean() - a.mean()) * 100, 'diff_lo': lo, 'diff_hi': hi,
                     'S5c_only_correct': n10, 'S6c_only_correct': n01, 'mcnemar_p': pv})
print('Check against B4_seeds_runs (expected CNX 81.09/84.46→69.18/72.18, R50 75.83/77.65→69.18/72.84):')
print(pd.DataFrame(chk).round(2).to_string(index=False))
print(save_table(pd.DataFrame(rows).round(3), 'B4_paired_tropaug').to_string(index=False))

> **Note:** S4-fair v1 preprocessed SLD with cv2, which differs from training — the paper uses **v2** (next cell). *Superseded by v2; not reported.*

In [ ]:
# ── S4-fair for all 5 architectures: PlantVillage backbone (S1, frozen) + linear head trained on SLD ──
import math
if 'train_linear' not in globals():
    def train_linear(X, y, n_cls=N_UNION, min_steps=3000, epochs=40, lr=3e-3, wd=1e-4, bs=512, seed=SEED):
        torch.manual_seed(seed)
        epochs = max(epochs, math.ceil(min_steps / math.ceil(len(X) / bs)))
        X = torch.as_tensor(X, dtype=torch.float32, device=DEVICE); y = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
        mu, sd = X.mean(0), X.std(0) + 1e-6
        head = nn.Linear(X.shape[1], n_cls).to(DEVICE)
        nn.init.zeros_(head.weight); nn.init.zeros_(head.bias)
        opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
        steps = epochs * math.ceil(len(X) / bs)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=DEVICE)
            for i in range(0, len(X), bs):
                b = perm[i:i + bs]
                loss = F.cross_entropy(head((X[b] - mu) / sd), y[b], label_smoothing=0.1)
                opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        @torch.no_grad()
        def predict(Z):
            Z = torch.as_tensor(np.asarray(Z, np.float32), device=DEVICE)
            return head((Z - mu) / sd).cpu().numpy()
        return predict

_ds = AlbumentationsDataset(DATASET_PATHS['SLD'])
sld = [(p, UNION_CLASSES.index(_ds.classes[l])) for p, l in _ds.samples]
print('SLD classes:', _ds.classes, '| images:', len(sld))          # expected 5 classes, 2521 images

rows = []
for arch in ARCHITECTURES:
    if not Path(ckpt_path('S1', arch)).exists():
        print(f'[WARNING] S1_{arch}.pt not found'); continue
    fx = feature_extractor(load_ckpt('S1', arch), arch).to(DEVICE)
    Xs = infer_logits(fx, make_loader(sld)); ys = np.array([l for _, l in sld])
    Xe = {t: infer_logits(fx, make_loader(EVAL_SETS[t])) for t in ['thai_test', 'thai_full']}
    del fx; torch.cuda.empty_cache()
    for s in range(3):
        pred = train_linear(Xs, ys, seed=SEED + s)
        r = {'arch': ARCH_LABEL[arch], 'seed': SEED + s}
        for t, X in Xe.items():
            c = to_thai(pred(X)).argmax(1) == labels_of(t)
            r[f'acc_{t}'] = c.mean() * 100
            if s == 0:
                r[f'ci_lo_{t}'], r[f'ci_hi_{t}'] = boot_ci(c)
        rows.append(r)
    print(f'  {arch} done')
df_s4 = pd.DataFrame(rows)
save_table(df_s4.round(3), 'S4fair_runs')
agg = df_s4.groupby('arch', sort=False).agg(
    test_mean=('acc_thai_test', 'mean'), test_sd=('acc_thai_test', 'std'),
    test_lo=('ci_lo_thai_test', 'first'), test_hi=('ci_hi_thai_test', 'first'),
    full_mean=('acc_thai_full', 'mean'), full_sd=('acc_thai_full', 'std')).round(2).reset_index()
print(save_table(agg, 'S4fair_summary').to_string(index=False))
print('Check: ResNet-50 should be close to 61.8 (test) / 62.5 (full), as in B1 "PV-R50 (S1) | SLD"')

In [ ]:
# ── S4-fair v2: SLD preprocessed exactly as in S5 training (PIL resize in CachedMultiSourceDataset) ──
from PIL import Image as _PIL
_mu = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1); _sd = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
SLD_IMG = np.stack([np.array(_PIL.open(p).convert('RGB').resize((IMG_SIZE, IMG_SIZE))) for p, _ in sld])

@torch.no_grad()
def feats_pil(fx, imgs, bs=256):
    out = []
    for i in range(0, len(imgs), bs):
        x = torch.from_numpy(imgs[i:i + bs]).permute(0, 3, 1, 2).float() / 255
        out.append(fx(((x - _mu) / _sd).to(DEVICE)).float().cpu())
    return torch.cat(out).numpy()

rows = []
for arch in ARCHITECTURES:
    fx = feature_extractor(load_ckpt('S1', arch), arch).to(DEVICE)
    Xs = feats_pil(fx, SLD_IMG); ys = np.array([l for _, l in sld])
    Xe = {t: infer_logits(fx, make_loader(EVAL_SETS[t])) for t in ['thai_test', 'thai_full']}
    del fx; torch.cuda.empty_cache()
    for s in range(3):
        pred = train_linear(Xs, ys, seed=SEED + s)
        r = {'arch': ARCH_LABEL[arch], 'seed': SEED + s}
        for t, X in Xe.items():
            c = to_thai(pred(X)).argmax(1) == labels_of(t)
            r[f'acc_{t}'] = c.mean() * 100
            if s == 0:
                r[f'ci_lo_{t}'], r[f'ci_hi_{t}'] = boot_ci(c)
        rows.append(r)
    print(f'  {arch} done')
df_s4 = pd.DataFrame(rows)
save_table(df_s4.round(3), 'S4fair_v2_runs')
agg = df_s4.groupby('arch', sort=False).agg(
    test_mean=('acc_thai_test', 'mean'), test_sd=('acc_thai_test', 'std'),
    test_lo=('ci_lo_thai_test', 'first'), test_hi=('ci_hi_thai_test', 'first'),
    full_mean=('acc_thai_full', 'mean'), full_sd=('acc_thai_full', 'std')).round(2).reset_index()
print(save_table(agg, 'S4fair_v2_summary').to_string(index=False))
print('Check: ResNet-50 should give 61.76 (test) / 62.53 (full), matching B1 "PV-R50 (S1) | SLD"')

## C1 — Field images: export predictions + blinded labelling form
Set the field image folder (`FIELD_DIR` in the Config cell; plain image files or one sub-folder per class) and run.
* `C1_field_predictions.csv` — predictions of every model (**do not send this file to the experts**)
* `C1_field_blinded_form.csv` — sent to the 2 experts, who label without seeing predictions
* After the forms are filled in, run C1.2 → accuracy against blinded labels, Cohen's κ, ECE on real images

In [ ]:
# ── C1.1 Field export ───────────────────────────────────────────────────────
# FIELD_DIR: set in the Config cell
if Path(FIELD_DIR).exists():
    field = [(str(p), -1) for p in sorted(Path(FIELD_DIR).rglob('*')) if p.suffix.lower() in IMG_EXT]
    FIELD_SET = {'field': field}
    print(f'field images: {len(field)}')
    out = pd.DataFrame({'path': [p for p, _ in field]})
    for arch in KEY_ARCHS:
        for scn in ['S5', 'S6']:
            if Path(ckpt_path(scn, arch)).exists():
                export_logits(scn, arch, sets=FIELD_SET, tta=True)
        for cfg in ['S5', 'S6-noTTA', 'S6+TTA']:
            p = get_probs(cfg, arch, 'field')
            out[f'{cfg}_{arch}_pred'] = [CLASS_NAMES[i] for i in p.argmax(1)]
            out[f'{cfg}_{arch}_conf'] = p.max(1).round(4)
    out.to_csv(ATAB / 'C1_field_predictions.csv', index=False)
    form = out[['path']].sample(frac=1, random_state=SEED).reset_index(drop=True)
    form.insert(0, 'img_id', [f'F{i:04d}' for i in range(len(form))])
    form[['img_id', 'path']].to_csv(ATAB / 'C1_field_KEY_do_not_share.csv', index=False)
    form['file_name'] = form.path.map(lambda p: Path(p).name)
    for c in ['expert1_label', 'expert1_certainty_1to5', 'expert2_label', 'expert2_certainty_1to5', 'notes']:
        form[c] = ''
    form.drop(columns='path').to_csv(ATAB / 'C1_field_blinded_form.csv', index=False)
    print('Label options: Healthy / RedRot / Rust / RingSpot / Other-disease / Not-sugarcane / Cannot-tell')
else:
    print(f'[WARNING] {FIELD_DIR} not found — skipping C1')

In [ ]:
# ── C1.2 After the experts have filled in the form ───────────────────────────
from sklearn.metrics import cohen_kappa_score
FILLED = ATAB / 'C1_field_blinded_form_FILLED.csv'      # ← the completed form
if FILLED.exists():
    f = pd.read_csv(FILLED).merge(pd.read_csv(ATAB / 'C1_field_KEY_do_not_share.csv'), on='img_id')
    f = f.merge(pd.read_csv(ATAB / 'C1_field_predictions.csv'), on='path')
    both = f.dropna(subset=['expert1_label', 'expert2_label'])
    if len(both):
        print(f"Cohen's κ (n={len(both)}): {cohen_kappa_score(both.expert1_label, both.expert2_label):.3f}")
    f['ref'] = np.where(f.expert2_label.notna() & (f.expert1_label != f.expert2_label), np.nan, f.expert1_label)
    ev = f[f.ref.isin(CLASS_NAMES)]
    print(f'Images used for evaluation (experts agree and label is one of the 3 classes): {len(ev)} / {len(f)}')
    rows = []
    for c in [c for c in f.columns if c.endswith('_pred')]:
        corr = (ev[c] == ev.ref).values
        lo, hi = boot_ci(corr) if len(ev) else (np.nan, np.nan)
        conf = ev[c.replace('_pred', '_conf')].values
        rows.append({'model': c.replace('_pred', ''), 'acc': corr.mean() * 100, 'ci_lo': lo, 'ci_hi': hi,
                     'mean_conf': conf.mean() * 100,
                     f'coverage@{TAU}': (conf >= TAU).mean() * 100,
                     f'sel_acc@{TAU}': corr[conf >= TAU].mean() * 100 if (conf >= TAU).any() else np.nan})
    print(save_table(pd.DataFrame(rows).round(2), 'C1_field_blinded_eval').to_string(index=False))
    print('\nLabels outside the 3 classes (OOD found in the field):')
    print(f.expert1_label.value_counts())

In [ ]:
# ── C1.3 Field images: S5c vs S6c (B4 — same training code, only TropAug differs) ──
# FIELD_DIR: set in the Config cell
field = [(str(p), -1) for p in sorted(Path(FIELD_DIR).rglob('*')) if p.suffix.lower() in IMG_EXT]
f = pd.read_csv(ATAB / 'C1_field_blinded_form_FILLED.csv').merge(
    pd.read_csv(ATAB / 'C1_field_KEY_do_not_share.csv'), on='img_id')
f = f[(f.expert1_label == f.expert2_label) & f.expert1_label.isin(CLASS_NAMES)]
pos = {p: i for i, (p, _) in enumerate(field)}
assert f.path.isin(pos).all(), 'paths in KEY do not match the files in FIELD_DIR'
idx = f.path.map(pos).values
y = f.expert1_label.map(CLASS_NAMES.index).values
print(f'field images {len(field)} | reference images {len(y)}')       # expected 109 | 103

corr, rows = {}, []
for arch in ['convnext_base', 'resnet50']:
    for nm, tag in [('S5c', 'ABL-PV+PD+SLD'), ('S6c', 'TROP-PV+PD+SLD')]:
        ck = AMOD / f'{tag}_{arch}_s42.pt'
        if not ck.exists():
            print(f'not found: {ck.name}'); continue
        m = build_model(arch, num_classes=N_UNION, pretrained=False)
        m.load_state_dict(torch.load(ck, map_location=DEVICE)); m = m.to(DEVICE).eval()
        z_clean = to_thai(infer_logits(m, make_loader(field)))
        z_tta = to_thai(np.stack([infer_logits(m, make_loader(field, transform=t, seed=TTA_SEED + 1000 * v))
                                  for v, t in enumerate(TTA_TRANSFORMS)]))
        for kind, p in [('noTTA', softmax_np(z_clean)), ('TTA', probs_from_views(z_tta))]:
            p = p[idx]; c = p.argmax(1) == y; conf = p.max(1)
            corr[(arch, nm, kind)] = c
            lo, hi = boot_ci(c)
            rows.append({'arch': arch, 'model': nm, 'tta': kind, 'acc': c.mean() * 100, 'ci_lo': lo, 'ci_hi': hi,
                         f'coverage@{TAU}': (conf >= TAU).mean() * 100,
                         f'sel_acc@{TAU}': c[conf >= TAU].mean() * 100 if (conf >= TAU).any() else np.nan})
        del m; torch.cuda.empty_cache()
print(save_table(pd.DataFrame(rows).round(2), 'C1_field_controlled_S5c_S6c').to_string(index=False))

rows = []
for arch in ['convnext_base', 'resnet50']:
    for kind in ['noTTA', 'TTA']:
        a, b = corr.get((arch, 'S5c', kind)), corr.get((arch, 'S6c', kind))
        if a is None or b is None:
            continue
        lo, hi = paired_boot_ci(a, b)
        pv, n10, n01 = mcnemar_p(a, b)
        rows.append({'arch': arch, 'tta': kind, 'S5c': a.mean() * 100, 'S6c': b.mean() * 100,
                     'diff(S6c−S5c)': (b.mean() - a.mean()) * 100, 'diff_lo': lo, 'diff_hi': hi,
                     'S5c_only_correct': n10, 'S6c_only_correct': n01, 'mcnemar_p': pv})
print(save_table(pd.DataFrame(rows).round(3), 'C1_field_tropaug_effect').to_string(index=False))
print('diff > 0 = TropAug helps on field images | CI spanning 0 = no conclusive difference')

In [ ]:
# ── C1.4 ConvNeXt S5c vs the deployed model (paired on the same 103 images) ──
pr = pd.read_csv(ATAB / 'C1_field_predictions.csv').set_index('path').loc[f.path]
ref_lab = f.expert1_label.values
cands = {'ConvNeXt S5c (noTTA)': corr[('convnext_base', 'S5c', 'noTTA')],
         'ConvNeXt S5 original (noTTA)': pr['S5_convnext_base_pred'].values == ref_lab}
dep = pr['S6+TTA_resnet50_pred'].values == ref_lab          # the deployed model
rows = []
for name, c in cands.items():
    lo, hi = paired_boot_ci(dep, c)
    pv, n_dep, n_c = mcnemar_p(dep, c)
    rows.append({'vs ResNet S6+TTA (deployed)': name, 'deploy_acc': dep.mean() * 100, 'cand_acc': c.mean() * 100,
                 'diff': (c.mean() - dep.mean()) * 100, 'diff_lo': lo, 'diff_hi': hi,
                 'deploy_only_correct': n_dep, 'cand_only_correct': n_c, 'mcnemar_p': pv})
print(save_table(pd.DataFrame(rows).round(3), 'C1_field_vs_deployed').to_string(index=False))
print('diff > 0 = candidate better than the deployed model | CI spanning 0 = no conclusive difference')

## S2-fixed — PlantVillage → PlantDoc, matching classes only (replaces the original S2, which compared mismatched class indices)

In [ ]:
# ── S2-fixed: PlantVillage → PlantDoc, matching classes only ─────────────────
from torchvision.datasets import ImageFolder
from sklearn.metrics import f1_score
PV_CLS = ImageFolder(DATASET_PATHS['PlantVillage']).classes
PD_ROOT = DATASET_PATHS['PlantDoc']
PD_CLS = ImageFolder(PD_ROOT).classes

PD2PV = {
 'Apple Scab Leaf': 'Apple___Apple_scab', 'Apple leaf': 'Apple___healthy',
 'Apple rust leaf': 'Apple___Cedar_apple_rust', 'Bell_pepper leaf': 'Pepper,_bell___healthy',
 'Bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot', 'Blueberry leaf': 'Blueberry___healthy',
 'Cherry leaf': 'Cherry_(including_sour)___healthy',
 'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
 'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight', 'Corn rust leaf': 'Corn_(maize)___Common_rust_',
 'Peach leaf': 'Peach___healthy', 'Potato leaf': 'Potato___healthy',
 'Potato leaf early blight': 'Potato___Early_blight', 'Potato leaf late blight': 'Potato___Late_blight',
 'Raspberry leaf': 'Raspberry___healthy', 'Soyabean leaf': 'Soybean___healthy',
 'Squash Powdery mildew leaf': 'Squash___Powdery_mildew', 'Strawberry leaf': 'Strawberry___healthy',
 'Tomato Early blight leaf': 'Tomato___Early_blight', 'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot',
 'Tomato leaf': 'Tomato___healthy', 'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot',
 'Tomato leaf late blight': 'Tomato___Late_blight', 'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
 'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato mold leaf': 'Tomato___Leaf_Mold',
 'Tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
 'grape leaf': 'Grape___healthy', 'grape leaf black rot': 'Grape___Black_rot'}

pairs = [(a, PV_CLS.index(b)) for a, b in PD2PV.items() if a in PD_CLS and b in PV_CLS]
print(f'matched {len(pairs)} classes out of {len(PD_CLS)} PlantDoc classes')
print('Unmatched PlantDoc classes:', [c for c in PD_CLS if c not in dict(pairs)])
print('PV names not found:', [b for a, b in PD2PV.items() if a in PD_CLS and b not in PV_CLS])
assert len(pairs) >= 2, 'could not match classes — check the class lists printed above'

pv_idx = [j for _, j in pairs]
samples = [(p, k) for k, (pd_c, _) in enumerate(pairs) for p, _ in list_images(PD_ROOT, [pd_c])]
y = np.array([l for _, l in samples])
rows = []
for arch in EVAL_ARCHS:
    m = load_ckpt('S1', arch).to(DEVICE)
    pred = infer_logits(m, make_loader(samples))[:, pv_idx].argmax(1)
    lo, hi = boot_ci(pred == y)
    rows.append({'arch': ARCH_LABEL[arch], 'n_classes': len(pairs), 'n_images': len(y),
                 'acc': (pred == y).mean() * 100, 'ci_lo': lo, 'ci_hi': hi,
                 'macro_f1': f1_score(y, pred, average='macro') * 100})
    del m; torch.cuda.empty_cache()
print(save_table(pd.DataFrame(rows).round(2), 'S2_fixed_pv_to_plantdoc').to_string(index=False))
print(f'chance level = {100 / len(pairs):.1f}%')

## A7c — SDA v2 on S5c/S6c (same training code, only TropAug differs)
Exactly the same degraded images as A7, pixel for pixel (same seeds) → directly comparable with A7.
Run cell A7 first (if it has been run before, the results are loaded within seconds).

In [ ]:
# ── A7c. SDA v2 — controlled S5c vs S6c ─────────────────────────────────────
CTRL_TAGS = [('S5c', 'ABL-PV+PD+SLD'), ('S6c', 'TROP-PV+PD+SLD')]
CTRL_MODELS = [(nm, arch, AMOD / f'{tag}_{arch}_s42.pt')
               for arch in ['convnext_base', 'resnet50'] for nm, tag in CTRL_TAGS
               if (AMOD / f'{tag}_{arch}_s42.pt').exists()]
print('models:', [(nm, arch) for nm, arch, _ in CTRL_MODELS])

sdac_npz = ATAB / 'A7c_sda_controlled_correct.npz'
if sdac_npz.exists():
    _d = np.load(sdac_npz, allow_pickle=True)
    CORR_C, CONDS_C, MODELS_C = _d['corr'], [tuple(c) for c in _d['conds']], [tuple(m) for m in _d['models']]
    print('loaded A7c results')
else:
    nets = {}
    for nm, arch, p in CTRL_MODELS:
        m = build_model(arch, num_classes=N_UNION, pretrained=False)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        nets[(nm, arch)] = SelectiveHead(m, THAI_IDX).to(DEVICE).eval()
    MODELS_C = list(nets)
    CONDS_C = [('Clean', 0, 0)] + [(f, s, r) for f in FACTORS for s in SEV for r in range(SDA_REPS)]
    y = labels_of('thai_test')
    CORR_C = np.zeros((len(MODELS_C), len(CONDS_C), len(y)), bool)
    t0 = time.time()
    for ci, (f, s, r) in enumerate(CONDS_C):
        tfm = VAL_TRANSFORM if f == 'Clean' else degraded_tfm(make_factor(f, s))
        seed = 100_000 * (r + 1) + 1000 * s + 10 * (FACTORS.index(f) if f in FACTORS else 0)   # = A7
        with torch.no_grad():
            for x, lab, idx in make_loader(thai_test, transform=tfm, seed=seed, return_index=True):
                x = x.to(DEVICE, non_blocking=True)
                for mi, k in enumerate(MODELS_C):
                    CORR_C[mi, ci, idx.numpy()] = (nets[k](x).argmax(1).cpu() == lab).numpy()
        if ci % 15 == 0:
            print(f'  {ci + 1}/{len(CONDS_C)} {f} s{s} r{r} | {time.time() - t0:.0f}s')
    np.savez_compressed(sdac_npz, corr=CORR_C, conds=np.array(CONDS_C, dtype=object),
                        models=np.array(MODELS_C, dtype=object))
    del nets; torch.cuda.empty_cache()

rows = []
for mi, (nm, arch) in enumerate(MODELS_C):
    clean = CORR_C[mi, 0].mean() * 100
    for f in FACTORS:
        for s in SEV:
            msk = [i for i, c in enumerate(CONDS_C) if c[0] == f and c[1] == s]
            acc = np.mean([CORR_C[mi, i].mean() * 100 for i in msk])
            rows.append({'model': nm, 'arch': ARCH_LABEL[arch], 'factor': f, 'severity': s,
                         'acc': acc, 'clean': clean, 'delta': clean - acc})
df_c = save_table(pd.DataFrame(rows).round(3), 'A7c_sda_controlled_long')
print(df_c.groupby(['arch', 'model', 'factor']).delta.mean().unstack('factor').round(1))

rows = []
yN = CORR_C.shape[2]; B = np.random.default_rng(SEED).integers(0, yN, size=(2000, yN))
for arch in ['convnext_base', 'resnet50']:
    if ('S5c', arch) not in MODELS_C or ('S6c', arch) not in MODELS_C:
        continue
    i5, i6 = MODELS_C.index(('S5c', arch)), MODELS_C.index(('S6c', arch))
    for f in FACTORS:
        msk = [i for i, c in enumerate(CONDS_C) if c[0] == f]
        d5 = CORR_C[i5, 0].astype(float) - CORR_C[i5, msk].mean(0)
        d6 = CORR_C[i6, 0].astype(float) - CORR_C[i6, msk].mean(0)
        a5 = CORR_C[i5, msk].mean(0); a6 = CORR_C[i6, msk].mean(0)
        g = d5 - d6; bs = g[B].mean(1) * 100; ba = (a6 - a5)[B].mean(1) * 100
        rows.append({'arch': ARCH_LABEL[arch], 'factor': f,
                     'clean_S5c': CORR_C[i5, 0].mean() * 100, 'clean_S6c': CORR_C[i6, 0].mean() * 100,
                     'drop_S5c': d5.mean() * 100, 'drop_S6c': d6.mean() * 100,
                     'robustness_gain': g.mean() * 100, 'gain_lo': np.percentile(bs, 2.5), 'gain_hi': np.percentile(bs, 97.5),
                     'degraded_acc_S5c': a5.mean() * 100, 'degraded_acc_S6c': a6.mean() * 100,
                     'degraded_diff(S6c−S5c)': (a6 - a5).mean() * 100,
                     'diff_lo': np.percentile(ba, 2.5), 'diff_hi': np.percentile(ba, 97.5)})
df_cg = save_table(pd.DataFrame(rows).round(2), 'A7c_tropaug_gain_controlled')
print(df_cg[['arch', 'factor', 'drop_S5c', 'drop_S6c', 'robustness_gain', 'gain_lo', 'gain_hi',
             'degraded_diff(S6c−S5c)', 'diff_lo', 'diff_hi']].to_string(index=False))
print('robustness_gain > 0 = TropAug "drops less" | degraded_diff > 0 = S6c more accurate than S5c on degraded images')

## C1.0 — Check that the offline evaluation matches the deployed system
Compares `pred_class` / `confidence` in the LINE system's log with ResNet-50 S6+TTA run offline on the same images.
They should agree on almost every image (TTA in the deployed system is not seeded, so confidences can differ slightly).
Run C1.1 first.

In [ ]:
# ── C1.0 Offline vs deployed consistency ────────────────────────────────────
LOG_CSV = FIELD_LOG_CSV
log = pd.read_csv(LOG_CSV)
log['file'] = log.image_url.astype(str).str.split('/').str[-1].str.split('?').str[0]
_norm = lambda s: ''.join(ch for ch in str(s).lower() if ch.isalnum())
NAME_MAP = {_norm(c): c for c in CLASS_NAMES}
NAME_MAP.update({'redrot': 'RedRot', 'rust': 'Rust', 'healthy': 'Healthy',
                 'โรคเน่าแดง': 'RedRot', 'โรคราสนิม': 'Rust', 'ปกติ': 'Healthy', 'ใบปกติ': 'Healthy'})  # Thai class names from the deployed system's log
log['deployed_pred'] = log.pred_class.map(lambda s: NAME_MAP.get(_norm(s), NAME_MAP.get(str(s).strip(), None)))
print('pred_class values found:', sorted(log.pred_class.astype(str).unique()))
if log.deployed_pred.isna().any():
    print('[WARNING] could not map:', sorted(log.loc[log.deployed_pred.isna(), 'pred_class'].astype(str).unique()),
          '→ add them to NAME_MAP and re-run')
conf = pd.to_numeric(log.confidence, errors='coerce')
log['deployed_conf'] = np.where(conf > 1, conf / 100, conf)

p_off = get_probs('S6+TTA', 'resnet50', 'field')
off = pd.DataFrame({'file': [Path(p).name for p, _ in field],
                    'offline_pred': [CLASS_NAMES[i] for i in p_off.argmax(1)],
                    'offline_conf': p_off.max(1)})
cmp = log.merge(off, on='file', how='left')
print(f'matched images: {cmp.offline_pred.notna().sum()} / {len(cmp)} rows')
m = cmp.dropna(subset=['offline_pred', 'deployed_pred'])
agree = (m.offline_pred == m.deployed_pred).mean() * 100
dconf = (m.offline_conf - m.deployed_conf).abs()
print(f'class agreement: {agree:.1f}%  (n={len(m)})')
print(f'confidence difference: median {dconf.median():.3f}, max {dconf.max():.3f}')
print('→ expect ≥ 95% agreement and small confidence differences; lower values mean the deployed system preprocesses images differently from this notebook')
m[m.offline_pred != m.deployed_pred][['file', 'deployed_pred', 'deployed_conf', 'offline_pred', 'offline_conf']] \
    .to_csv(ATAB / 'C1_offline_vs_deployed_mismatch.csv', index=False)
save_table(pd.DataFrame([{'n': len(m), 'class_agreement_%': round(agree, 2),
                          'median_abs_conf_diff': round(dconf.median(), 4), 'max_abs_conf_diff': round(dconf.max(), 4)}]),
           'C1_offline_vs_deployed')

## C1.1b — Excel form with images (one file per expert)

In [ ]:
# ── C1.1b Excel form with an image in each row (one file per expert) ──────────
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Font, Alignment, PatternFill
from PIL import Image as PILImage
import io

LABELS = ['Healthy', 'RedRot', 'Rust', 'RingSpot', 'Other-disease', 'Not-sugarcane', 'Cannot-tell']
THUMB = 360                       # maximum image width/height in the file (pixels)
key = pd.read_csv(ATAB / 'C1_field_KEY_do_not_share.csv')   # img_id, path (already shuffled)

def make_form(expert_no):
    wb = Workbook(); ws = wb.active; ws.title = 'form'
    head = ['img_id', 'image', 'label', 'certainty (1-5)', 'notes']
    ws.append(head)
    for c in ws[1]:
        c.font = Font(bold=True, color='FFFFFF'); c.fill = PatternFill('solid', fgColor='2F5233')
        c.alignment = Alignment(horizontal='center', vertical='center')
    ws.column_dimensions['A'].width = 10; ws.column_dimensions['B'].width = 52
    ws.column_dimensions['C'].width = 18; ws.column_dimensions['D'].width = 15; ws.column_dimensions['E'].width = 40
    ws.freeze_panes = 'A2'
    dv_lab = DataValidation(type='list', formula1='"' + ','.join(LABELS) + '"', allow_blank=True)
    dv_cer = DataValidation(type='whole', operator='between', formula1='1', formula2='5', allow_blank=True)
    ws.add_data_validation(dv_lab); ws.add_data_validation(dv_cer)
    for i, r in enumerate(key.itertuples(index=False), start=2):
        ws.cell(row=i, column=1, value=r.img_id).alignment = Alignment(vertical='center')
        im = PILImage.open(r.path).convert('RGB'); im.thumbnail((THUMB, THUMB))
        buf = io.BytesIO(); im.save(buf, format='JPEG', quality=85); buf.seek(0)
        xi = XLImage(buf); ws.add_image(xi, f'B{i}')
        ws.row_dimensions[i].height = im.height * 0.75 + 6          # pixels → points
        dv_lab.add(f'C{i}'); dv_cer.add(f'D{i}')
        for col in 'CDE':
            ws[f'{col}{i}'].alignment = Alignment(vertical='center', wrap_text=True)
    # Instruction sheet shown to the (Thai) experts; kept in Thai as used in the study.
    # It says: expert form no. N; view each image and pick a label from the dropdown; labels: ...;
    # Cannot-tell = cannot judge from the image (e.g. blurred, too far); certainty 1 = not sure at all ... 5 = very sure;
    # evaluate independently, do not consult others, and do not look at the LINE system's results.
    info = wb.create_sheet('คำอธิบาย')
    for line in [f'แบบฟอร์มผู้เชี่ยวชาญท่านที่ {expert_no}',
                 'กรุณาดูรูปแต่ละแถว แล้วเลือกป้ายในคอลัมน์ label จาก dropdown',
                 'ป้าย: ' + ' / '.join(LABELS),
                 'Cannot-tell = ตัดสินจากรูปไม่ได้ (เช่น เบลอ ไกลเกินไป)',
                 'certainty: 1 = ไม่แน่ใจเลย ... 5 = แน่ใจมาก',
                 'กรุณาประเมินด้วยตนเอง ไม่ปรึกษาผู้อื่น และไม่ดูผลจากระบบ LINE']:
        info.append([line])
    info.column_dimensions['A'].width = 90
    out = Path(f'{BASE_DRIVE}/C1_expert{expert_no}_form.xlsx'); wb.save(out)
    return out

for e in (1, 2):
    print('[OK]', make_form(e))

## Z — Combine all tables into one Excel file

In [ ]:
# ── Z. Compile ──────────────────────────────────────────────────────────────
out_xlsx = ADDON_DIR / 'addon_results_all.xlsx'
with pd.ExcelWriter(out_xlsx) as w:
    for f in sorted(ATAB.glob('*.csv')):
        if f.name.startswith(('N1_', 'C1_field_KEY', 'C1_field_blinded_form', 'A8_proxies', 'A9_gradcam_per_image', 'A1_dup_pairs')):
            continue
        pd.read_csv(f).to_excel(w, sheet_name=f.stem[:31], index=False)
print(f'[OK] {out_xlsx}')

## Environment — record library versions (run after all other cells)

In [ ]:
import sys, platform, importlib
for _m in ['torch', 'torchvision', 'timm', 'albumentations', 'cv2', 'numpy', 'pandas', 'sklearn',
           'statsmodels', 'scipy', 'PIL', 'imagehash', 'open_clip', 'pytorch_grad_cam', 'openpyxl']:
    try: print(f'{_m:18s}', importlib.import_module(_m).__version__)
    except Exception as e: print(f'{_m:18s}', 'n/a')
print('python', sys.version.split()[0], '|', platform.platform())
